In [3]:
import os
import re
import json
import shutil
import hashlib
import subprocess
import time
import requests
import urllib.parse
from datetime import datetime, timedelta
from typing import TypedDict, List, Dict, Any
import pandas as pd
from bs4 import BeautifulSoup

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from exa_py import Exa
from composio import Composio
from dotenv import load_dotenv
load_dotenv()  # This loads your variables from the local .env file

# ==========================================
# 1. API KEYS & CREDENTIALS SETUP
# ==========================================
os.environ ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
os.environ ["COMPOSIO_API_KEY"] = os.getenv("COMPOSIO_API_KEY", "")
os.environ ["GETEMAIL_API_KEY"] = os.getenv("GETEMAIL_API_KEY", "")

# Initialize Search Clients
composio = Composio(api_key=os.environ["COMPOSIO_API_KEY"])
exa = Exa(api_key=os.environ["EXA_API_KEY"])

# Local LLM for Private CV Ingestion & Fallback
# Local Qwen with automatic fallback when Ollama can't run it (out of memory, not running, timeout)
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b")
_QWEN_DOWN = [False, ""]   # once the model fails, the rest of the run skips it (no repeated slow load attempts)


class _Text:
    def __init__(self, content: str):
        self.content = content


class ResilientLLM:
    """OLLAMA_MODEL first. If it fails, tries (in order) Claude CLI when claude_first, Gemini (JOB_SCORER=gemini + key)
    when allow_gemini, then Claude CLI when allow_claude. Prints what it switched to; raises only if all fail."""
    def __init__(self, num_ctx: int = None, temperature=None, allow_gemini: bool = True, claude_first: bool = False,
                 allow_claude: bool = True):
        kw = {k: v for k, v in {"num_ctx": num_ctx, "temperature": temperature}.items() if v is not None}
        self.llm = ChatOllama(model=OLLAMA_MODEL, base_url="http://localhost:11434",
                              client_kwargs={"timeout": 120}, **kw)  # fail after 2 min instead of hanging
        self.allow_gemini, self.claude_first, self.allow_claude = allow_gemini, claude_first, allow_claude

    def invoke(self, prompt: str):
        if not _QWEN_DOWN[0]:
            try:
                return self.llm.invoke(prompt)
            except Exception as e:
                _QWEN_DOWN[:] = [True, re.sub(r"\s+", " ", str(e))[:140]]
                print(f"  🔁 {OLLAMA_MODEL} unavailable ({_QWEN_DOWN[1]}). Using a fallback for the rest of this run.")
        errors_seen = [f"{OLLAMA_MODEL}: {_QWEN_DOWN[1]}"]
        steps = ([("Claude CLI", claude_cli_text)] if self.claude_first else []) \
            + ([("Gemini", gemini_text)] if self.allow_gemini and os.getenv("JOB_SCORER", "gemini").strip().lower() == "gemini"
               and os.getenv("GEMINI_API_KEY") else []) \
            + ([("Claude CLI", claude_cli_text)] if self.allow_claude and not self.claude_first else [])
        for name, fn in steps:
            try:
                print(f"  ↪ {name} handles this {OLLAMA_MODEL} step.")
                return _Text(fn(prompt))
            except Exception as e:
                errors_seen.append(f"{name}: {str(e)[:100]}")
                print(f"  ⚠️ {name} fallback failed ({str(e)[:100]}).")
        raise RuntimeError("No model available: " + " | ".join(errors_seen))

def claude_cli_text(prompt: str, timeout_s: int = 180) -> str:
    """One plain-text Claude CLI call with the lean flags (no Claude Code system prompt, tools or MCP)."""
    if not os.path.exists(CLAUDE_EXE):
        raise RuntimeError("Claude CLI not found at CLAUDE_EXE")
    out = subprocess.run([CLAUDE_EXE, "-p", "--tools", "", "--strict-mcp-config", "--system-prompt",
                          "You write concise, factual summaries.", "--output-format", "text"],
                         input=prompt, capture_output=True, text=True, encoding="utf-8", timeout=timeout_s)
    if out.returncode != 0 or not out.stdout.strip():
        raise RuntimeError(f"Claude CLI exit {out.returncode}: {(out.stderr or '').strip()[:120]}")
    return out.stdout.strip()


ollama_llm = ResilientLLM()
cv_summary_llm = ResilientLLM(claude_first=True)  # CV summary: Qwen -> Claude CLI -> Gemini (cached, so rare)
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

# Directories & Storage
RESUMES_DIR = r"C:\Users\simar\Desktop\CV_2026"
CACHE_FILE = r"C:\Users\simar\Downloads\multi_resume_cache.json"
CHROMA_PERSIST_DIR = r"C:\Users\simar\Downloads\chroma_db_resumes"
EXCEL_PATH = r"C:\Users\simar\Downloads\Matched_Jobs.xlsx"

# Path to authenticated Claude executable
CLAUDE_EXE = r"C:\Users\simar\.local\bin\claude.exe"

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_file_hash(filepath: str) -> str:
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read(65536)
        while len(buf) > 0:
            hasher.update(buf)
            buf = f.read(65536)
    return hasher.hexdigest()

def load_cache() -> Dict[str, Any]:
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_cache(cache_data: Dict[str, Any]):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache_data, f, indent=2)

def parse_duration_to_days(user_input: str) -> int:
    txt = user_input.lower().strip()
    if "24" in txt or "hour" in txt or "today" in txt or "1 day" in txt:
        return 1
    day_match = re.search(r"(\d+)\s*day", txt)
    if day_match:
        return int(day_match.group(1))
    if "week" in txt:
        week_match = re.search(r"(\d+)\s*week", txt)
        return int(week_match.group(1)) * 7 if week_match else 7
    if "month" in txt:
        month_match = re.search(r"(\d+)\s*month", txt)
        return int(month_match.group(1)) * 30 if month_match else 30
    return 7

# ==========================================
# 3. CHROMA SETUP
# ==========================================
vectorstore = Chroma(
    persist_directory=CHROMA_PERSIST_DIR,
    embedding_function=ollama_embeddings
)

def sync_resumes_to_chroma(resumes_dir: str):
    if not os.path.exists(resumes_dir):
        return
    cache = load_cache()
    pdf_files = [f for f in os.listdir(resumes_dir) if f.endswith(".pdf")]

    for file_name in pdf_files:
        file_path = os.path.join(resumes_dir, file_name)
        file_hash = get_file_hash(file_path)
        persona_key = os.path.splitext(file_name)[0]

        if persona_key in cache and cache[persona_key].get("pdf_hash") == file_hash:
            continue

        print(f"📦 Indexing updated resume persona: {file_name} into ChromaDB...")
        loader = PyPDFLoader(file_path)
        docs = loader.load_and_split()
        for d in docs:
            d.metadata["persona_key"] = persona_key
            d.metadata["file_name"] = file_name

        vectorstore.add_documents(docs)
        print(f"  ✓ {file_name} indexed with persona metadata.")

sync_resumes_to_chroma(RESUMES_DIR)

# ==========================================
# 4. PIPELINE STATE
# ==========================================
class PipelineState(TypedDict):
    candidate_name: str
    roles: List[str]
    target_geos: List[str]
    job_duration_days: int
    duration_label: str
    target_approved_count: int
    resume_summary: str
    raw_jobs: List[Dict[str, Any]]
    evaluated_jobs: List[Dict[str, Any]]
    approved_jobs: List[Dict[str, Any]]
    low_match_summary: str
    search_iteration: int
    user_decision: str
    search_mode: str                 # "both" (default, parallel) | "linkedin" | "career" (= web agent)
    target_domain: str               # blank = derive from CV
    career_company_count: int
    role_variants: List[str]
    shortlisted_companies: List[Dict[str, Any]]
    company_batch: List[Dict[str, Any]]
    career_jobs: List[Dict[str, Any]]
    board_jobs: List[Dict[str, Any]]
    vc_funds: List[str]              # blank = Accel, Matrix Partners + local funds
    fund_boards: List[Dict[str, Any]]
    new_fund_boards: List[Dict[str, Any]]
    outreach_drafts: List[Dict[str, Any]]
    scoring_mode: str                # "title" | "full" (asked once, before the first scoring)
    candidate_years: Any             # your years of experience (None = experience filter off)
    shown_job_keys: List[str]        # job IDs + company/title of every job already shown at sign-off this run
    discovery_company_count: int     # companies the discovery agent reads per location (50-100)
    discovery_batch: List[Dict[str, Any]]
    discovery_jobs: List[Dict[str, Any]]

# ==========================================
# 5. GRAPH NODES
# ==========================================

# Node 1: Multi-Resume Cached Reader
def parse_resume_node(state: PipelineState) -> Dict[str, Any]:
    persona_key = state.get("candidate_name", "Default")
    print(f"\n📄 Node 1: Checking Resume Profile Cache for '{persona_key}'...")

    cache = load_cache()
    expected_file = os.path.join(RESUMES_DIR, f"{persona_key}.pdf")
    current_hash = get_file_hash(expected_file) if os.path.exists(expected_file) else None

    if persona_key in cache:
        entry = cache[persona_key]
        if current_hash is None or entry.get("pdf_hash") == current_hash:
            print(f"  ⚡ [Instant] Loaded cached profile for '{persona_key}' (0ms)!")
            return {"resume_summary": entry["summary"]}

    print(f"  ⚙️ Generating new summary for '{persona_key}' via local Ollama...")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"persona_key": persona_key}})
    docs = retriever.invoke(f"Key technical skills, target roles, impact for {persona_key}")
    context = "\n\n".join([d.page_content for d in docs])

    if not context.strip() and os.path.exists(expected_file):
        loader = PyPDFLoader(expected_file)
        context = "\n\n".join([d.page_content for d in loader.load()])

    prompt = f"""Summarize this candidate profile into concise bullet points for job matching:
1. Primary Roles (e.g. Chief of Staff, Product Manager, AI Product Builder)
2. Core Technical, AI & Product Competencies
3. Industry Domains & Impact

Resume Context:
{context}"""

    try:
        resp = cv_summary_llm.invoke(prompt)
    except Exception as e:
        print(f"  🔁 No model could summarise the CV ({str(e)[:120]}). Using the raw CV text this run (not cached).")
        return {"resume_summary": context[:3000]}
    summary_text = resp.content

    cache[persona_key] = {
        "file_name": f"{persona_key}.pdf",
        "pdf_hash": current_hash,
        "last_updated": datetime.now().isoformat(),
        "summary": summary_text
    }
    save_cache(cache)
    print(f"  ✓ Summary cached successfully for '{persona_key}'!")
    return {"resume_summary": summary_text}


# Fast Live LinkedIn Direct Guest Scraper
def fetch_live_linkedin_jobs(role: str, geo: str, days: int = 3, limit: int = 20) -> List[Dict[str, Any]]:
    seconds = days * 86400
    region, remote = parse_geo(geo)  # 'European Union (Remote)' -> region + LinkedIn remote filter
    encoded_role = urllib.parse.quote(role)
    encoded_geo = urllib.parse.quote(region or "Worldwide")
    url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={encoded_role}&location={encoded_geo}&f_TPR=r{seconds}&start=0"
    if remote:
        url += "&f_WT=2"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"}
    
    jobs = []
    try:
        resp = requests.get(url, headers=headers, timeout=8)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            for post in soup.find_all("li")[:limit]:
                title_tag = post.find("h3", class_="base-search-card__title")
                company_tag = post.find("h4", class_="base-search-card__subtitle")
                location_tag = post.find("span", class_="job-search-card__location")
                link_tag = post.find("a", class_="base-card__full-link")
                date_tag = post.find("time")
                
                if title_tag and link_tag:
                    jobs.append({
                        "Job Profile": title_tag.get_text(strip=True),
                        "Target Company": company_tag.get_text(strip=True) if company_tag else "Unknown Company",
                        "Location": location_tag.get_text(strip=True) if location_tag else geo,
                        "Job Link": link_tag.get("href", "").split("?")[0],
                        "Platform Name": "LinkedIn",
                        "Date Posted": (date_tag.get("datetime") or date_tag.get_text(strip=True)) if date_tag else f"Past {days} days"
                    })
    except Exception as e:
        print(f"  ⚠️ Live LinkedIn query issue for {role}: {e}")
    return jobs


# Node 2: Fast Dual-Engine Job Search
def search_jobs_node(state: PipelineState) -> Dict[str, Any]:
    days = state.get("job_duration_days", 3)
    iteration = state.get("search_iteration", 1)
    if not agent_on(state, "linkedin"):
        return {"raw_jobs": [], "search_iteration": iteration + 1}
    print(f"\n🔍 Node 2: Searching live postings [Iteration {iteration}] (past {days} days)...")

    all_jobs = []

    for role in state["roles"]:
        for geo in state["target_geos"]:
            print(f"  ⚡ Querying live LinkedIn index for '{role}' in '{geo}'...")
            li_jobs = fetch_live_linkedin_jobs(role, geo, days=days, limit=15)
            all_jobs.extend(li_jobs)

    # ATS boards (Greenhouse/Lever/Ashby...) are searched by the web agent (Node 2c)

    unique_map = {j["Job Link"]: j for j in all_jobs if j.get("Job Link")}
    final_raw = list(unique_map.values())

    print(f"  ✓ Found {len(final_raw)} fresh live postings in ~3 seconds!")
    return {"raw_jobs": final_raw, "search_iteration": iteration + 1}


# ==========================================
# CAREER-PAGE SEARCH (runs in parallel with the LinkedIn search)
# 1. Role-first: find postings for the role in the target area and collect the hiring companies
# 2. Domain: add companies in the user's domain headquartered in / with offices in the area
# 3. Read every shortlisted company's careers page / ATS board for matching roles
# ==========================================
from concurrent.futures import ThreadPoolExecutor

CAREER_HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"}
CAREER_PATHS = ["/careers", "/jobs", "/careers/jobs", "/join-us", "/company/careers", "/about/careers"]
ATS_DOMAINS = ["greenhouse.io", "lever.co", "ashbyhq.com", "smartrecruiters.com", "workable.com", "myworkdayjobs.com"]
JOB_BOARD_DOMAINS = ["linkedin.com", "indeed.com", "glassdoor.com", "naukri.com", "wellfound.com", "builtin.com",
                     "monster.com", "ziprecruiter.com", "remoteok.com", "weworkremotely.com", "remote.co", "instahyre.com",
                     "iimjobs.com", "foundit.in", "otta.com", "welcometothejungle.com", "stepstone.de", "jooble.org"]
SKIP_COMPANY_DOMAINS = JOB_BOARD_DOMAINS + ["wikipedia.", "crunchbase.", "tracxn.", "ambitionbox.", "medium.",
                                            "youtube.", "forbes.", "reddit.", "quora.", "github."]

EU_PLACES = ["eu", "europe", "european union", "emea", "austria", "belgium", "bulgaria", "croatia", "cyprus", "czech",
             "czechia", "denmark", "estonia", "finland", "france", "germany", "greece", "hungary", "ireland", "italy",
             "latvia", "lithuania", "luxembourg", "malta", "netherlands", "poland", "portugal", "romania", "slovakia",
             "slovenia", "spain", "sweden", "berlin", "munich", "hamburg", "paris", "amsterdam", "rotterdam", "madrid",
             "barcelona", "dublin", "lisbon", "porto", "stockholm", "copenhagen", "helsinki", "warsaw", "krakow",
             "prague", "vienna", "brussels", "milan", "rome", "athens", "budapest", "bucharest", "tallinn", "vilnius",
             "riga", "sofia", "zagreb", "valencia"]
LOCATION_ALIASES = {
    "eu": EU_PLACES, "european union": EU_PLACES,
    "europe": EU_PLACES + ["uk", "united kingdom", "london", "switzerland", "zurich", "norway", "oslo"],
    "bangalore": ["bangalore", "bengaluru"], "bengaluru": ["bangalore", "bengaluru"],
    "gurgaon": ["gurgaon", "gurugram"], "gurugram": ["gurgaon", "gurugram"],
    "delhi": ["delhi", "ncr", "noida", "gurgaon", "gurugram"],
    "mumbai": ["mumbai", "bombay"],
    "london": ["london", "united kingdom", "uk"], "uk": ["uk", "united kingdom", "london", "england"],
    "netherlands": ["netherlands", "amsterdam", "rotterdam", "utrecht", "eindhoven"],
    "spain": ["spain", "madrid", "barcelona", "valencia"],
    "india": ["india", "bangalore", "bengaluru", "delhi", "mumbai", "pune", "hyderabad", "gurgaon", "gurugram", "noida", "chennai"],
    "singapore": ["singapore", "apac"],
}
ROLE_ABBREVIATIONS = {"pm": "product manager", "apm": "associate product manager", "gpm": "group product manager",
                      "tpm": "technical program manager", "cos": "chief of staff", "em": "engineering manager",
                      "genai": "generative ai", "ml": "machine learning"}
ROLE_STOPWORDS = {"of", "and", "the", "a", "an", "&", "-", "to", "senior", "sr", "junior", "jr", "lead", "head", "principal", "staff"}

ATS_PATTERNS = [
    ("Greenhouse", re.compile(r"(?:boards|job-boards)(?:\.eu)?\.greenhouse\.io/(?:embed/job_board(?:/js)?\?for=)?([A-Za-z0-9_-]+)")),
    ("Lever", re.compile(r"jobs\.lever\.co/([A-Za-z0-9_.-]+)")),
    ("Ashby", re.compile(r"jobs\.ashbyhq\.com/([A-Za-z0-9_.%-]+)")),
    ("SmartRecruiters", re.compile(r"(?:jobs|careers)\.smartrecruiters\.com/([A-Za-z0-9_-]+)")),
    ("Workable", re.compile(r"apply\.workable\.com/([A-Za-z0-9_-]+)")),
    ("Workday", re.compile(r"([a-z0-9-]+)\.(wd\d+)\.myworkdayjobs\.com/(?:[a-z]{2}-[A-Z]{2}/)?([A-Za-z0-9_-]+)")),
]
BAD_SLUGS = {"embed", "js", "v1", "api", "jobs", "careers", "en-us", "wday", "search"}


def parse_geo(geo: str):
    """'European Union (Remote)' -> ('European Union', True); 'Remote' -> ('', True)."""
    remote = bool(re.search(r"\bremote\b", geo, re.I))
    region = re.sub(r"(?i)\bremote\b", " ", geo)
    region = re.sub(r"[()\[\]]", " ", region)
    region = re.sub(r"\s+", " ", region).strip(" -,/|")
    return region, remote


def _place_keys(region: str) -> List[str]:
    r = region.lower().strip()
    return LOCATION_ALIASES.get(r, [r] + [w for w in re.findall(r"[a-z]+", r) if len(w) > 3])


def location_matches(job_location: str, geo: str) -> bool:
    loc = (job_location or "").lower()
    if not loc:
        return True  # careers page didn't state a location; let the scorer decide
    region, remote = parse_geo(geo)
    if remote and not re.search(r"remote|anywhere|distributed|work from home|wfh", loc):
        return False
    if not region:
        return True
    if any(re.search(rf"\b{re.escape(k)}\b", loc) for k in _place_keys(region)):
        return True
    # A plain "Remote" posting with no country attached is kept for remote searches
    return remote and bool(re.fullmatch(r"[\s\W]*(fully\s+)?(remote|anywhere)[\s\W]*", loc))


def _normalize_title(text: str) -> str:
    t = (text or "").lower().replace("/", " ")
    for abbr, full in ROLE_ABBREVIATIONS.items():
        t = re.sub(rf"\b{abbr}\b", full, t)
    return t


def expand_role_variants(roles: List[str]) -> List[str]:
    """Adds similar titles for each target role (via local Qwen) so 'AI PM' also catches 'Product Manager, AI'."""
    variants = [_normalize_title(r) for r in roles]
    prompt = f"""For each of these target job titles, list up to 4 closely similar job titles that a company might use
for the same job (same seniority band and function). Target titles: {', '.join(roles)}
Return STRICTLY a JSON array of strings."""
    try:
        match = re.search(r"\[.*\]", ollama_llm.invoke(prompt).content, re.DOTALL)
        variants += [_normalize_title(v) for v in (json.loads(match.group(0)) if match else []) if isinstance(v, str)]
    except Exception as e:
        print(f"  ⚠️ Role variant notice: {e}")
    return list(dict.fromkeys(v.strip() for v in variants if v.strip()))


_ROLE_VARIANT_CACHE: Dict[tuple, List[str]] = {}
_ROLE_VARIANT_LOCK = __import__("threading").Lock()

def expand_role_variants_cached(roles: List[str]) -> List[str]:
    """Parallel branches share one Qwen call for similar titles."""
    with _ROLE_VARIANT_LOCK:
        key = tuple(roles)
        if key not in _ROLE_VARIANT_CACHE:
            _ROLE_VARIANT_CACHE[key] = expand_role_variants(roles)
        return _ROLE_VARIANT_CACHE[key]


def title_matches_roles(title: str, roles: List[str]) -> bool:
    t = _normalize_title(title)
    for role in roles:
        role_l = _normalize_title(role).strip()
        if role_l and role_l in t:
            return True
        tokens = [w for w in re.findall(r"[a-z0-9]+", role_l) if w not in ROLE_STOPWORDS]
        if tokens and all(re.search(rf"\b{re.escape(w)}", t) for w in tokens):
            return True
    return False


def _parse_iso(value) -> datetime:
    if not value:
        return None
    try:
        if isinstance(value, (int, float)):  # Lever uses epoch milliseconds
            return datetime.fromtimestamp(value / 1000)
        return datetime.fromisoformat(str(value).replace("Z", "+00:00")).replace(tzinfo=None)
    except Exception:
        return None


def _parse_workday_posted(text: str) -> datetime:
    txt = (text or "").lower()
    if "today" in txt:
        return datetime.now()
    if "yesterday" in txt:
        return datetime.now() - timedelta(days=1)
    m = re.search(r"(\d+)\+?\s*day", txt)
    return datetime.now() - timedelta(days=int(m.group(1))) if m else None


def _root_url(url: str) -> str:
    p = urllib.parse.urlparse(url if "://" in url else f"https://{url}")
    return f"{p.scheme or 'https'}://{p.netloc}"


def _clean_company_name(title: str, url: str) -> str:
    name = re.split(r"\s+[\-|–—:·]\s+", (title or "").strip())[0].strip()
    if not name or len(name) > 50:
        name = urllib.parse.urlparse(url).netloc.replace("www.", "").split(".")[0].title()
    return name


def _match_ats(text: str):
    for name, pat in ATS_PATTERNS:
        m = pat.search(text or "")
        if m and m.group(1).lower() not in BAD_SLUGS:
            return name, list(m.groups())
    return None, None


def derive_domain_from_resume(persona_key: str, resume_summary: str) -> str:
    """Reads the candidate's current industry domain from the cached CV summary (cached per persona)."""
    cache = load_cache()
    if cache.get(persona_key, {}).get("domain"):
        return cache[persona_key]["domain"]

    prompt = f"""From this candidate profile, name the ONE industry domain they currently work in,
in 2-4 words (e.g. 'Fintech', 'B2B SaaS', 'EdTech', 'Healthcare AI'). Return only the domain, nothing else.

Candidate Profile:
{resume_summary}"""
    try:
        domain = ollama_llm.invoke(prompt).content.strip().splitlines()[0].strip(" .'\"*")[:60]
    except Exception:
        domain = ""
    domain = domain or "Technology"

    if persona_key in cache:
        cache[persona_key]["domain"] = domain
        save_cache(cache)
    return domain


def discover_hiring_companies(roles: List[str], geo: str, domain: str, limit: int, exclude: set) -> List[Dict[str, Any]]:
    """Role-first shortlist: search postings for the role in the area, keep the companies behind them."""
    region, remote = parse_geo(geo)
    where = f"in {region}" if region else ""
    mode = "remote" if remote else ""
    found: Dict[str, Dict[str, Any]] = {}

    queries = [re.sub(r"\s+", " ", q).strip() for role in roles
               for q in (f"{mode} {role} {domain} job {where}", f"{mode} {role} job opening {where}")]

    def _ats(q):
        try:
            return exa.search(query=q, type="auto", include_domains=ATS_DOMAINS, num_results=25).results
        except Exception as e:
            print(f"  ⚠️ Exa ATS posting search issue for '{q}': {e}")
            return []

    def _web(q):
        try:
            return exa.search(query=f"{q} careers", type="auto", exclude_domains=ATS_DOMAINS + JOB_BOARD_DOMAINS, num_results=15).results
        except Exception as e:
            print(f"  ⚠️ Exa web posting search issue for '{q}': {e}")
            return []

    with ThreadPoolExecutor(max_workers=8) as pool:  # all role searches at once
        ats_results = list(pool.map(_ats, queries))
        web_results = list(pool.map(_web, queries))
    for ats_res, web_res in zip(ats_results, web_results):
        # a) Postings on public ATS boards -> company + its board, no website lookup needed
        for r in ats_res:
            ats, slug = _match_ats(r.url)
            key = f"{ats}:{'/'.join(slug or [])}".lower()
            if ats and key not in exclude and key not in found:
                found[key] = {"name": slug[0].replace("-", " ").title(), "website": "", "ats": ats,
                              "slug": slug, "key": key, "source": "role posting"}
        # b) Postings on companies' own websites
        for r in web_res:
            ats, slug = _match_ats(r.url)
            site = _root_url(r.url)
            if ats or site in exclude or site in found or any(s in site for s in SKIP_COMPANY_DOMAINS):
                continue
            found[site] = {"name": urllib.parse.urlparse(site).netloc.replace("www.", "").split(".")[0].title(),
                           "website": site, "key": site, "source": "role posting"}
    return list(found.values())[:limit]


def shortlist_domain_companies(domain: str, geo: str, roles: List[str], count: int, exclude: set) -> List[Dict[str, Any]]:
    """Domain shortlist: companies in `domain` headquartered in / with offices in the area (Exa, then local Qwen)."""
    region, remote = parse_geo(geo)
    where = f"headquartered in or with offices in {region}" if region else "remote-first"
    hiring = " hiring remote" if remote else " hiring"
    found: Dict[str, Dict[str, Any]] = {}
    queries = [f"{domain} company {where}"] + [f"{domain} company {where}{hiring} {role}" for role in roles[:2]]

    for q in queries:
        if len(found) >= count:
            break
        try:
            res = exa.search(query=q, type="auto", category="company", num_results=count * 2)
            for r in res.results:
                site = _root_url(r.url)
                if site in exclude or site in found or any(s in site for s in SKIP_COMPANY_DOMAINS):
                    continue
                found[site] = {"name": _clean_company_name(r.title, r.url), "website": site, "key": site, "source": "domain"}
                if len(found) >= count:
                    break
        except Exception as e:
            print(f"  ⚠️ Exa company search issue for '{q}': {e}")

    if len(found) < count:
        prompt = f"""List {count - len(found)} real {domain} companies {where} that hire for: {', '.join(roles)}{' (remote)' if remote else ''}.
Return STRICTLY a JSON array: [{{"name": "Company", "website": "https://company.com"}}]"""
        try:
            match = re.search(r"\[.*\]", ollama_llm.invoke(prompt).content, re.DOTALL)
            for c in (json.loads(match.group(0)) if match else []):
                site = c.get("website") or ""
                if not site:
                    res = exa.search(query=f"{c.get('name')} official website", type="auto", category="company", num_results=1)
                    site = res.results[0].url if res.results else ""
                if not site:
                    continue
                site = _root_url(site)
                if site not in exclude and site not in found:
                    found[site] = {"name": c.get("name") or _clean_company_name("", site), "website": site, "key": site, "source": "domain"}
        except Exception as e:
            print(f"  ⚠️ Qwen company shortlist notice: {e}")

    return list(found.values())[:count]


def detect_career_source(website: str):
    """Visits the homepage and common careers paths; returns (ats_name, slug_parts, careers_page_url)."""
    candidates, careers_url = [], None
    try:
        home = requests.get(website, headers=CAREER_HEADERS, timeout=6)
        if home.status_code == 200:
            ats, slug = _match_ats(home.text)
            if ats:
                return ats, slug, home.url
            soup = BeautifulSoup(home.text, "html.parser")
            for a in soup.find_all("a", href=True):
                if re.search(r"career|jobs|join", a["href"] + " " + a.get_text(" ", strip=True), re.I):
                    candidates.append(urllib.parse.urljoin(home.url, a["href"]))
                if len(candidates) >= 2:
                    break
    except Exception:
        pass
    candidates += [website.rstrip("/") + p for p in CAREER_PATHS]

    for url in dict.fromkeys(candidates):
        ats, slug = _match_ats(url)  # careers link may itself point to the ATS
        if ats:
            return ats, slug, url
        try:
            r = requests.get(url, headers=CAREER_HEADERS, timeout=6)
        except Exception:
            continue
        if r.status_code != 200:
            continue
        ats, slug = _match_ats(r.url + " " + r.text)
        if ats:
            return ats, slug, r.url
        careers_url = careers_url or r.url
    return None, None, careers_url


def guess_ats_from_name(company: str, website: str):
    """Many boards use the company name as slug; probe the common public APIs directly."""
    root = urllib.parse.urlparse(website).netloc.replace("www.", "").split(".")[0]
    for slug in dict.fromkeys([re.sub(r"[^a-z0-9]", "", company.lower()), root]):
        if not slug:
            continue
        for ats, url in [("Greenhouse", f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"),
                         ("Lever", f"https://api.lever.co/v0/postings/{slug}?mode=json&limit=1"),
                         ("Ashby", f"https://api.ashbyhq.com/posting-api/job-board/{slug}")]:
            try:
                r = requests.get(url, timeout=5)
                if r.status_code == 200 and r.text.strip() not in ("", "[]"):
                    return ats, [slug]
            except Exception:
                pass
    return None, None


def fetch_ats_jobs(ats: str, slug_parts, roles: List[str]) -> List[Dict[str, Any]]:
    """Pulls open postings from a public ATS job-board API."""
    slug = slug_parts[0]
    out = []
    if ats == "Greenhouse":
        r = requests.get(f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs", timeout=8)
        for j in r.json().get("jobs", []) if r.status_code == 200 else []:
            out.append({"title": j.get("title"), "location": (j.get("location") or {}).get("name", ""),
                        "url": j.get("absolute_url"), "posted": _parse_iso(j.get("first_published") or j.get("updated_at"))})
    elif ats == "Lever":
        r = requests.get(f"https://api.lever.co/v0/postings/{slug}?mode=json", timeout=8)
        for j in r.json() if r.status_code == 200 else []:
            cat = j.get("categories") or {}
            loc = cat.get("location", "") + (" / Remote" if j.get("workplaceType") == "remote" else "")
            out.append({"title": j.get("text"), "location": loc, "url": j.get("hostedUrl"), "posted": _parse_iso(j.get("createdAt"))})
    elif ats == "Ashby":
        r = requests.get(f"https://api.ashbyhq.com/posting-api/job-board/{slug}", timeout=8)
        for j in r.json().get("jobs", []) if r.status_code == 200 else []:
            loc = (j.get("location") or "") + (" / Remote" if j.get("isRemote") else "")
            out.append({"title": j.get("title"), "location": loc, "url": j.get("jobUrl"), "posted": _parse_iso(j.get("publishedAt"))})
    elif ats == "SmartRecruiters":
        r = requests.get(f"https://api.smartrecruiters.com/v1/companies/{slug}/postings?limit=100", timeout=8)
        for j in r.json().get("content", []) if r.status_code == 200 else []:
            l = j.get("location") or {}
            loc = ", ".join(x for x in [l.get("city"), l.get("country")] if x) + (" / Remote" if l.get("remote") else "")
            out.append({"title": j.get("name"), "location": loc, "url": f"https://jobs.smartrecruiters.com/{slug}/{j.get('id')}",
                        "posted": _parse_iso(j.get("releasedDate"))})
    elif ats == "Workable":
        r = requests.get(f"https://apply.workable.com/api/v1/widget/accounts/{slug}", timeout=8)
        for j in r.json().get("jobs", []) if r.status_code == 200 else []:
            loc = ", ".join(x for x in [j.get("city"), j.get("country")] if x) + (" / Remote" if j.get("telecommuting") else "")
            out.append({"title": j.get("title"), "location": loc, "url": j.get("url") or j.get("shortlink"),
                        "posted": _parse_iso(j.get("published_on") or j.get("created_at"))})
    elif ats == "Workday":
        tenant, wd, site = slug_parts
        base = f"https://{tenant}.{wd}.myworkdayjobs.com"
        for role in roles:
            r = requests.post(f"{base}/wday/cxs/{tenant}/{site}/jobs", timeout=8, headers=CAREER_HEADERS,
                              json={"appliedFacets": {}, "limit": 20, "offset": 0, "searchText": role})
            for j in r.json().get("jobPostings", []) if r.status_code == 200 else []:
                out.append({"title": j.get("title"), "location": j.get("locationsText", ""),
                            "url": f"{base}/en-US/{site}{j.get('externalPath', '')}", "posted": _parse_workday_posted(j.get("postedOn"))})
    return out


def scrape_generic_career_page(careers_url: str, roles: List[str]) -> List[Dict[str, Any]]:
    """Plain-HTML careers page: keep links whose text looks like one of the target roles."""
    out = []
    try:
        r = requests.get(careers_url, headers=CAREER_HEADERS, timeout=8)
        soup = BeautifulSoup(r.text, "html.parser")
        for a in soup.find_all("a", href=True):
            text = a.get_text(" ", strip=True)
            if 3 < len(text) < 120 and title_matches_roles(text, roles):
                out.append({"title": text, "location": "", "url": urllib.parse.urljoin(r.url, a["href"]), "posted": None})
    except Exception:
        pass
    return out


def fetch_career_page_jobs(company: Dict[str, Any], roles: List[str], search_roles: List[str], geo: str, days: int) -> List[Dict[str, Any]]:
    name, website = company["name"], company.get("website", "")
    ats, slug_parts, careers_url = company.get("ats"), company.get("slug"), None
    if not ats and website:
        ats, slug_parts, careers_url = detect_career_source(website)
        if not ats:
            ats, slug_parts = guess_ats_from_name(name, website)

    if ats == "Greenhouse":  # boards found via postings only know the slug; fetch the real company name
        try:
            name = requests.get(f"https://boards-api.greenhouse.io/v1/boards/{slug_parts[0]}", timeout=5).json().get("name") or name
        except Exception:
            pass

    postings, source = [], ats or "Website"
    try:
        if ats:
            postings = fetch_ats_jobs(ats, slug_parts, search_roles)
        elif careers_url:
            postings = scrape_generic_career_page(careers_url, roles)
    except Exception as e:
        print(f"  ⚠️ Career page read issue for {name}: {e}")

    # Last resort for JS-rendered careers pages: let Exa find role pages on the company's own domain
    if not postings and website:
        try:
            netloc = urllib.parse.urlparse(website).netloc
            for role in search_roles:
                res = exa.search(query=f"{name} {role} job opening", type="auto", include_domains=[netloc], num_results=3)
                for r in res.results:
                    postings.append({"title": (r.title or "").split("|")[0].strip(), "location": "", "url": r.url, "posted": None})
            source = "Website (Exa)"
        except Exception:
            pass

    jobs = finalize_postings([{**p, "company": name} for p in postings], roles, geo, days, f"Career Page ({source})")
    print(f"  {'✓' if jobs else '·'} {name} [{source}]: {len(jobs)} matching role(s)")
    return jobs


# ==========================================
# VC / PE PORTFOLIO + JOB BOARD / HEADHUNTER SEARCH
# ==========================================
DEFAULT_VC_FUNDS = ["Accel", "Matrix Partners","Peak XV", "Lightspeed", "Antler", "Entrepreneur First", "Y Combinator"]  # always included; local funds for each area are added automatically

# Local job boards & headhunters per area ("global" is always searched). Add any site here, e.g. "numi.com".
JOB_BOARDS = {
    "global": ["wellfound.com", "jobgether.com", "michaelpage.com", "hays.com", "robertwalters.com", "iimjobs.com",
               "cutshort.io", "indeed.com", "glassdoor.com", "workatastartup.com", "himalayas.app", "remoteok.com",
               "weworkremotely.com", "welcometothejungle.com"],
    "eu": ["jobgether.com", "welcometothejungle.com", "eu-startups.com", "berlinstartupjobs.com", "stepstone.de",
           "michaelpage.de", "michaelpage.fr", "michaelpage.nl", "michaelpage.es", "michaelpage.it",
           "michaelpage.ie", "michaelpage.be", "michaelpage.pt"],
    "uk": ["michaelpage.co.uk", "welcometothejungle.com", "hays.co.uk", "robertwalters.co.uk", "cwjobs.co.uk"],
    "netherlands": ["michaelpage.nl", "welcometothejungle.com"],
    "spain": ["michaelpage.es", "infojobs.net", "welcometothejungle.com"],
    "india": ["naukri.com", "iimjobs.com", "instahyre.com", "cutshort.io", "hirist.tech", "foundit.in", "michaelpage.co.in"],
    "singapore": ["michaelpage.com.sg", "mycareersfuture.gov.sg", "jobstreet.com.sg", "hays.com.sg"],
}
JOB_BOARDS["european union"] = JOB_BOARDS["eu"]
JOB_BOARDS["europe"] = JOB_BOARDS["eu"] + JOB_BOARDS["uk"]
JOB_BOARDS["london"] = JOB_BOARDS["uk"]
HEADHUNTERS = {"michaelpage": "Michael Page", "hays": "Hays", "robertwalters": "Robert Walters"}
# Hiring posts on social sites ("we're hiring an AI PM ...")
SOCIAL_SOURCES = {"X/Twitter": ["x.com", "twitter.com"], "Reddit": ["reddit.com"]}
SOCIAL_DOMAINS = ["linkedin.", "twitter.", "x.com", "facebook.", "instagram.", "youtube.", "medium.", "github.", "apple.com", "google."]


def boards_for_geo(geo: str) -> List[str]:
    region, _ = parse_geo(geo)
    r = region.lower()
    boards = list(JOB_BOARDS["global"])
    for key, sites in JOB_BOARDS.items():
        if key != "global" and r and (r == key or r in LOCATION_ALIASES.get(key, [])):
            boards += sites
    return list(dict.fromkeys(boards))


def _board_label(url: str) -> str:
    netloc = urllib.parse.urlparse(url).netloc.replace("www.", "")
    if any(d in netloc for d in ATS_DOMAINS):
        return f"ATS ({netloc})"
    for key, name in HEADHUNTERS.items():
        if key in netloc:
            return f"Recruiter ({name})"
    return f"Job Board ({netloc})"


def _split_board_title(title: str):
    """'AI Product Manager at Acme - Berlin | Jobgether' -> ('AI Product Manager', 'Acme')."""
    t = (title or "").strip()
    m = re.match(r"(.+?)\s+(?:at|@|bei|chez)\s+(.+?)(?:\s+[|\-–—]\s+.*)?$", t)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    parts = re.split(r"\s+[|\-–—]\s+", t)
    company = parts[1].strip() if len(parts) > 2 else "Unknown Company"  # 2-part titles are usually 'Role | Site'
    return parts[0].strip(), company


def _exa_postings(query: str, domains: List[str], days: int, label_fn, num: int = 25) -> List[Dict[str, Any]]:
    """Exa search restricted to `domains`; every result is treated as one posting."""
    out = []
    try:
        res = exa.search(query=query, type="auto", include_domains=domains, num_results=num)
        for r in res.results:
            title, company = _split_board_title(r.title)
            ats, slug = _match_ats(r.url)
            if ats:  # ATS posting URLs carry the company in the slug
                company = slug[0].replace("-", " ").title()
            out.append({"title": title, "company": company, "location": "", "url": r.url,
                        "posted": _parse_iso(getattr(r, "published_date", None)), "platform": label_fn(r.url)})
    except Exception as e:
        print(f"  ⚠️ Exa board search issue for '{query}': {e}")
    return out


def _jobs_from_next_data(html: str, base_url: str) -> List[Dict[str, Any]]:
    """Portfolio job boards (Getro / Consider) ship their job list as JSON inside the page."""
    m = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', html, re.S)
    if not m:
        return []
    out, stack = [], [json.loads(m.group(1))]
    while stack:
        node = stack.pop()
        if isinstance(node, list):
            stack.extend(node)
        elif isinstance(node, dict):
            url = node.get("url") or node.get("applicationUrl") or node.get("applyUrl")
            if isinstance(node.get("title"), str) and isinstance(url, str):
                org = node.get("organization") or node.get("company") or {}
                locs = node.get("locations") or node.get("location") or []
                locs = locs if isinstance(locs, list) else [locs]
                created = node.get("createdAt") or node.get("created_at") or node.get("postedAt")
                if isinstance(created, (int, float)) and created < 1e11:  # epoch seconds -> ms
                    created *= 1000
                out.append({"title": node["title"],
                            "company": (org.get("name") if isinstance(org, dict) else str(org)) or "Unknown Company",
                            "location": ", ".join(l.get("name", "") if isinstance(l, dict) else str(l) for l in locs),
                            "url": urllib.parse.urljoin(base_url, url), "posted": _parse_iso(created)})
            stack.extend(v for v in node.values() if isinstance(v, (dict, list)))
    return out


FUND_CACHE_FILE = os.path.join(RESUMES_DIR, "vc_fund_cache.json")  # fund boards / portfolios change slowly
FUND_CACHE_DAYS = 7


def load_fund_cache(geo: str, named_funds: List[str]):
    try:
        entry = json.load(open(FUND_CACHE_FILE, encoding="utf-8")).get(f"{geo.lower()}|{'/'.join(sorted(named_funds)).lower()}")
    except Exception:
        return None
    if entry and entry.get("saved_on", "") >= (datetime.now() - timedelta(days=FUND_CACHE_DAYS)).strftime("%Y-%m-%d"):
        return entry["explored"]
    return None


def save_fund_cache(geo: str, named_funds: List[str], explored: List[Dict[str, Any]]):
    try:
        cache = json.load(open(FUND_CACHE_FILE, encoding="utf-8"))
    except Exception:
        cache = {}
    cache[f"{geo.lower()}|{'/'.join(sorted(named_funds)).lower()}"] = {"saved_on": datetime.now().strftime("%Y-%m-%d"), "explored": explored}
    with open(FUND_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f)


def find_vc_funds(geo: str, named_funds: List[str]) -> List[Dict[str, str]]:
    """Named funds (Accel, Matrix Partners, user's own) + VC/PE funds headquartered in the area."""
    region, _ = parse_geo(geo)
    funds: Dict[str, Dict[str, str]] = {}
    queries = [(f"{f} venture capital fund {region}".strip(), 1) for f in named_funds]
    if region:
        queries += [(f"venture capital fund headquartered in {region} investing in startups", 5),
                    (f"private equity growth fund headquartered in {region}", 3)]
    def _search(qn):
        try:
            return exa.search(query=qn[0], type="auto", category="company", num_results=qn[1]).results
        except Exception as e:
            print(f"  ⚠️ Fund search issue for '{qn[0]}': {e}")
            return []
    with ThreadPoolExecutor(max_workers=8) as pool:  # all fund searches at once instead of one by one
        all_results = list(pool.map(_search, queries))
    for results in all_results:
        for r in results:
            site = _root_url(r.url)
            if site not in funds and not any(s in site for s in SKIP_COMPANY_DOMAINS + SOCIAL_DOMAINS):
                funds[site] = {"name": _clean_company_name(r.title, r.url), "website": site}
    return list(funds.values())


def explore_vc_fund(fund: Dict[str, str], max_companies: int) -> Dict[str, Any]:
    """Finds the fund's portfolio job board and, if it has none, its portfolio companies' websites."""
    board, portfolio_url, companies = None, None, []
    fund_netloc = urllib.parse.urlparse(fund["website"]).netloc.replace("www.", "")
    try:
        home = requests.get(fund["website"], headers=CAREER_HEADERS, timeout=6)
        soup = BeautifulSoup(home.text, "html.parser")
        for a in soup.find_all("a", href=True):
            href = urllib.parse.urljoin(home.url, a["href"])
            text = (a.get_text(" ", strip=True) + " " + href).lower()
            if not board and re.search(r"jobs|careers|talent|getro|consider", text) and re.search(r"portfolio|jobs\.|talent", text):
                board = _root_url(href) if "jobs." in href or "getro" in href else href
            if not portfolio_url and re.search(r"portfolio|companies", text) and fund_netloc in href:
                portfolio_url = href
    except Exception:
        pass

    if not board and portfolio_url:
        try:
            page = requests.get(portfolio_url, headers=CAREER_HEADERS, timeout=8)
            for a in BeautifulSoup(page.text, "html.parser").find_all("a", href=True):
                href = urllib.parse.urljoin(page.url, a["href"])
                site = _root_url(href)
                if href.startswith("http") and fund_netloc not in site and not any(s in site for s in SOCIAL_DOMAINS + SKIP_COMPANY_DOMAINS):
                    if site not in [c["website"] for c in companies]:
                        companies.append({"name": a.get_text(" ", strip=True)[:50] or _clean_company_name("", site),
                                          "website": site, "key": site, "source": f"VC portfolio: {fund['name']}"})
                if len(companies) >= max_companies:
                    break
        except Exception:
            pass
    return {"fund": fund["name"], "board": board, "companies": companies}


def fetch_fund_board_jobs(board: str, fund_name: str, roles: List[str], search_roles: List[str], geo: str, days: int) -> List[Dict[str, Any]]:
    """Reads a VC portfolio job board (Getro / Consider style), falling back to Exa on the board's domain."""
    postings = []
    for role in search_roles:
        try:
            r = requests.get(f"{board.rstrip('/')}/jobs?q={urllib.parse.quote(role)}", headers=CAREER_HEADERS, timeout=8)
            postings += _jobs_from_next_data(r.text, r.url) if r.status_code == 200 else []
        except Exception:
            pass
    if not postings:
        netloc = urllib.parse.urlparse(board).netloc
        region, remote = parse_geo(geo)
        for role in search_roles:
            postings += _exa_postings(f"{'remote ' if remote else ''}{role} job {region}", [netloc], days, lambda u: "")
    jobs = finalize_postings(postings, roles, geo, days, f"VC Portfolio Board ({fund_name})")
    print(f"  {'✓' if jobs else '·'} {fund_name} portfolio board: {len(jobs)} matching role(s)")
    return jobs


def finalize_postings(postings: List[Dict[str, Any]], roles: List[str], geo: str, days: int, platform: str) -> List[Dict[str, Any]]:
    """Shared role / location / recency filter that maps postings into the pipeline's job format."""
    cutoff = datetime.now() - timedelta(days=days)
    jobs = []
    for p in postings:
        if not p.get("url") or not title_matches_roles(p.get("title"), roles):
            continue
        if not location_matches(p.get("location"), geo):
            continue
        if p.get("posted") and p["posted"] < cutoff:
            continue
        jobs.append({
            "Job Profile": p["title"],
            "Target Company": p.get("company") or "Unknown Company",
            "Location": p.get("location") or geo,
            "Job Link": p["url"],
            "Platform Name": p.get("platform") or platform,
            "Date Posted": p["posted"].strftime("%Y-%m-%d") if p.get("posted") else "Not listed"
        })
    return jobs


# Node 2c: ATS boards, job boards, headhunters & social hiring posts (rest of the web agent)
def search_job_boards_node(state: PipelineState) -> Dict[str, Any]:
    if not agent_on(state, "web"):
        return {}
    days = state.get("job_duration_days", 3)
    search_roles = [_normalize_title(r) for r in state["roles"]]
    variants = expand_role_variants_cached(state["roles"])
    print(f"\n📋 Node 2c: Searching ATS boards, job boards, headhunters, Reddit & X (past {days} days)...")

    jobs = []
    for geo in state["target_geos"]:
        region, remote = parse_geo(geo)
        boards = boards_for_geo(geo)
        found, social = [], []
        for role in search_roles:
            q = f"{'remote ' if remote else ''}{role} job {region}".strip()
            found += _exa_postings(q, ATS_DOMAINS, days, _board_label)
            found += _exa_postings(q, boards, days, _board_label)
            for name, domains in SOCIAL_SOURCES.items():
                social += _exa_postings(f"hiring {q}", domains, days, lambda u, n=name: f"Social ({n})", num=10)
        geo_jobs = finalize_postings(found, variants, geo, days, "Job Board")
        # Social posts have no posting date in the title, so keep only ones Exa dates inside the window
        geo_jobs += finalize_postings([p for p in social if p.get("posted")], variants, geo, days, "Social")
        print(f"  ✓ {geo}: {len(geo_jobs)} posting(s) from ATS boards, {len(boards)} job sites/recruiters, Reddit & X")
        jobs += geo_jobs
    return {"board_jobs": jobs}

# ==========================================
# COMPANY DISCOVERY AGENT (3rd parallel search branch: Node 2d + 2e)
# 2d. Builds a pool of companies for your CV's domain in each location, across stages
#     (seed, Series A-B, growth, large/public), from Exa company search + one Gemini list per location.
#     The pool is saved in CV_2026\company_pool.json and topped up when it runs low (refreshed weekly).
#     Each batch takes DISCOVERY_NEW_SHARE (default 75%) companies that were NOT in the previous batch.
#     The other slots go only to last batch's companies that posted recently ("hot"), so they get rechecked.
# 2e. Reads each company's careers page / ATS board (the board found once is cached in the pool, so later
#     runs need one request per company), keeps the 2-3 best-fitting roles per company, and flags fresh postings.
# HN. Hacker News "Ask HN: Who is hiring?" (free Algolia API, last 2 monthly threads): companies whose post names a
#     fitting role in your location join the batch (on top of N) and the pool. Their careers page is read like any
#     other; if it shows no fitting role, the HN post itself becomes the job (if it's inside your recency window).
# ==========================================
import random

COMPANY_POOL_FILE = os.path.join(RESUMES_DIR, "company_pool.json")
COMPANY_POOL_DAYS = int(os.getenv("COMPANY_POOL_DAYS", "7"))                       # pool top-up interval
DISCOVERY_NEW_SHARE = float(os.getenv("DISCOVERY_NEW_SHARE", "0.75"))              # 0.70-0.80 = 70-80% new companies per batch
DISCOVERY_MAX_ROLES_PER_COMPANY = int(os.getenv("DISCOVERY_MAX_ROLES_PER_COMPANY", "3"))
DISCOVERY_DOMAINS = os.getenv("DISCOVERY_DOMAINS", "")                             # e.g. "Fintech / Insurtech"; blank = CV domain
DISCOVERY_EXA_FALLBACK = os.getenv("DISCOVERY_EXA_FALLBACK", "0").strip() == "1"   # 1 = Exa search inside JS-only careers sites (credits)
DISCOVERY_WORKERS = int(os.getenv("DISCOVERY_WORKERS", "12"))
HOT_DAYS, FRESH_DAYS, MOMENTUM_DAYS = 3, 7, 14
HN_ENABLED = os.getenv("HN_WHO_IS_HIRING", "1").strip() != "0"                     # 0 = skip Hacker News
HN_MAX_COMPANIES = int(os.getenv("HN_MAX_COMPANIES", "20"))                        # HN companies added per location per batch
HN_THREADS = int(os.getenv("HN_THREADS", "2"))                                     # monthly threads to read (this + last month)
HN_API = "https://hn.algolia.com/api/v1"
COMPANY_STAGES = {   # stage: (share of each batch, wording for the Exa company search)
    "Seed / Pre-seed": (0.20, "seed stage startup"),
    "Series A-B": (0.25, "Series A or Series B funded startup"),
    "Growth (Series C+)": (0.25, "late-stage growth company Series C or later"),
    "Large / Public": (0.30, "large enterprise or publicly listed company"),
}
_POOL_LOCK = __import__("threading").Lock()


def agent_on(state: PipelineState, agent: str) -> bool:
    """agent = 'linkedin' | 'web' | 'discover'. search_mode 'both'/'all' (default) runs all three in parallel."""
    mode = state.get("search_mode", "both")
    return mode in ("both", "all") or mode == {"linkedin": "linkedin", "web": "career", "discover": "discover"}[agent]


def _site_key(url: str) -> str:
    return urllib.parse.urlparse(url if "://" in url else f"https://{url}").netloc.lower().replace("www.", "")


def load_company_pool() -> Dict[str, Any]:
    try:
        with open(COMPANY_POOL_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
            return {"pools": data.get("pools", {}), "history": data.get("history", {})}
    except Exception:
        return {"pools": {}, "history": {}}


def save_company_pool(data: Dict[str, Any]):
    try:
        with open(COMPANY_POOL_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=1, default=str)
    except Exception as e:
        print(f"  ⚠️ Could not save the company pool: {e}")


def pool_key(domain: str, geo: str) -> str:
    return f"{_norm_text(domain) or 'all'}|{_norm_text(geo)}"


def company_heat(c: Dict[str, Any]) -> int:
    """How actively a company is hiring, from its last scan: fitting roles count 3x, any posting in 14 days 1x."""
    newest = parse_posted_text(c.get("newest_post")) if c.get("newest_post") else None
    if not newest or (datetime.now() - newest).days > MOMENTUM_DAYS:
        return 0
    return 3 * int(c.get("matches") or 0) + int(c.get("recent_posts") or 0)


def posting_freshness(date_posted) -> str:
    d = parse_posted_text(date_posted)
    if not d:
        return ""
    age = max(0, (datetime.now() - d).days)
    return f"🔥 New ({age}d)" if age <= HOT_DAYS else f"🆕 This week ({age}d)" if age <= FRESH_DAYS else ""


def _add_company(found: Dict[str, Dict[str, Any]], name: str, website: str, stage: str, source: str):
    website = (website or "").strip()
    if not website or not name:
        return
    site = _root_url(website)
    key = _site_key(site)
    if not key or "." not in key or any(s in site for s in SKIP_COMPANY_DOMAINS + ATS_DOMAINS):
        return
    if key not in found:
        found[key] = {"name": name.strip()[:60], "website": site, "key": key, "stage": stage if stage in COMPANY_STAGES else "",
                      "source": source, "added": datetime.now().strftime("%Y-%m-%d")}


def exa_stage_companies(domain: str, geo: str, num: int) -> Dict[str, Dict[str, Any]]:
    """One Exa company search per stage (run in parallel)."""
    region, remote = parse_geo(geo)
    where = f"headquartered in or with an office in {region}" if region else ""
    where += " remote-first" if remote else ""
    found: Dict[str, Dict[str, Any]] = {}

    def _one(item):
        stage, (_, words) = item
        q = re.sub(r"\s+", " ", f"{domain if domain else 'technology'} {words} {where}").strip()
        try:
            return stage, exa.search(query=q, type="auto", category="company", num_results=num).results
        except Exception as e:
            print(f"  ⚠️ Exa company search issue for '{q}': {e}")
            return stage, []

    with ThreadPoolExecutor(max_workers=4) as pool:
        for stage, results in pool.map(_one, COMPANY_STAGES.items()):
            for r in results:
                _add_company(found, _clean_company_name(r.title, r.url), r.url, stage, "Exa")
    return found


def llm_company_list(domain: str, geo: str, n: int, known_names: List[str]) -> Dict[str, Dict[str, Any]]:
    """One Gemini call per location (Qwen/Claude only if Gemini is unavailable): real companies by stage, with websites."""
    region, remote = parse_geo(geo)
    where = (f"headquartered in, or with an office in, {region}" if region else "anywhere") + (" and hiring remotely" if remote else "")
    mix = ", ".join(f"{round(share * n)} {stage}" for stage, (share, _) in COMPANY_STAGES.items())
    prompt = f"""List {n} real companies in {domain or 'technology'} (and closely adjacent areas) that are {where}.
Mix of stages: {mix}. Include well-known large organisations AND lesser-known startups.
Do not list any of these: {', '.join(known_names[:120]) or 'none'}.
Return STRICTLY a JSON array: [{{"name": "Company", "website": "https://company.com", "stage": "one of {list(COMPANY_STAGES)}"}}]"""
    text = ""
    try:
        text = gemini_text(prompt) if GEMINI_API_KEY else ""
    except Exception as e:
        print(f"  ⚠️ Gemini company list notice: {str(e)[:120]}")
    if not text:
        try:
            text = ollama_llm.invoke(prompt).content
        except Exception as e:
            print(f"  ⚠️ No model could list companies ({str(e)[:120]}); using Exa results only.")
    found: Dict[str, Dict[str, Any]] = {}
    match = re.search(r"\[.*\]", text or "", re.DOTALL)
    try:
        for c in json.loads(match.group(0)) if match else []:
            if isinstance(c, dict):
                _add_company(found, c.get("name") or "", c.get("website") or "", c.get("stage") or "", "LLM")
    except Exception:
        pass
    return found


def ensure_company_pool(data: Dict[str, Any], domain: str, geo: str, need: int) -> List[Dict[str, Any]]:
    """Returns the pool for (domain, geo), topping it up when it's older than COMPANY_POOL_DAYS or too small
    to give `need` companies that weren't in the last batch."""
    pk = pool_key(domain, geo)
    pool = data["pools"].setdefault(pk, {"built": "", "companies": {}})
    last = set(data["history"].get(pk, {}).get("last_batch", []))
    usable = [c for c in pool["companies"].values() if c["key"] not in last and int(c.get("no_source") or 0) < 2]
    built = parse_posted_text(pool.get("built")) if pool.get("built") else None
    stale = not built or (datetime.now() - built).days >= COMPANY_POOL_DAYS
    if not stale and len(usable) >= need:
        return list(pool["companies"].values())
    print(f"  🔎 {geo}: topping up the company pool ({len(pool['companies'])} known, {len(usable)} usable, need {need})...")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=2) as ex:
        f_exa = ex.submit(exa_stage_companies, domain, geo, 25)
        f_llm = ex.submit(llm_company_list, domain, geo, max(40, need), [c["name"] for c in pool["companies"].values()])
        new = {**f_exa.result(), **f_llm.result()}   # the LLM's stage label beats the Exa query's guess
    added = 0
    for k, c in new.items():
        if k not in pool["companies"]:
            pool["companies"][k] = c
            added += 1
        elif not pool["companies"][k].get("stage") and c.get("stage"):
            pool["companies"][k]["stage"] = c["stage"]
    pool["built"] = datetime.now().strftime("%Y-%m-%d")
    print(f"  ⏱ {geo}: +{added} companies in {time.time() - t0:.0f}s (pool now {len(pool['companies'])}).")
    return list(pool["companies"].values())


def select_company_batch(companies: List[Dict[str, Any]], last_batch: List[str], n: int,
                         new_share: float = None, picked: Dict[str, str] = None, seed=None) -> List[Dict[str, Any]]:
    """Batch of n companies: at least new_share of them were not in last_batch. Carry-overs are only last batch's
    hot companies (recent postings). New ones: never-picked first, then least recently picked, spread over stages."""
    new_share = DISCOVERY_NEW_SHARE if new_share is None else new_share
    picked, last, rng = picked or {}, set(last_batch or []), random.Random(seed)
    order = {c["key"]: rng.random() for c in companies}
    carry_cap = int(n * (1 - new_share))   # rounded down, so the new share is never below new_share
    carry = sorted([c for c in companies if c["key"] in last and company_heat(c) > 0], key=lambda c: -company_heat(c))[:carry_cap]
    fresh = [c for c in companies if c["key"] not in last and int(c.get("no_source") or 0) < 2]
    rank = lambda c: (c["key"] in picked, picked.get(c["key"], ""), -company_heat(c), order[c["key"]])
    need_new = n - len(carry)
    new: List[Dict[str, Any]] = []

    def take(cands, k):
        chosen = {c["key"] for c in new}
        new.extend(sorted([c for c in cands if c["key"] not in chosen], key=rank)[:max(0, k)])

    never = [c for c in fresh if c["key"] not in picked]
    for tier in (never, fresh):   # never-picked companies first, then the least recently picked
        for stage, (share, _) in COMPANY_STAGES.items():   # stage quotas
            have = sum(1 for c in new if c.get("stage") == stage)
            take([c for c in tier if c.get("stage") == stage], min(round(need_new * share) - have, need_new - len(new)))
        take(tier, need_new - len(new))   # top up from any stage
    return carry + new[:need_new]


def pick_best_roles(jobs: List[Dict[str, Any]], variants: List[str], k: int = None) -> List[Dict[str, Any]]:
    """Up to k roles per company: closest title to your target titles first, then the newest."""
    k = DISCOVERY_MAX_ROLES_PER_COMPANY if k is None else k
    targets = [_title_words(v) for v in variants] or [set()]

    def closeness(j):
        t = _title_words(j.get("Job Profile"))
        return max(len(t & v) / max(1, len(t | v)) for v in targets)

    def newest(j):
        d = parse_posted_text(j.get("Date Posted"))
        return d.timestamp() if d else 0

    return sorted(jobs, key=lambda j: (-round(closeness(j), 2), -newest(j)))[:k]


def scan_discovered_company(company: Dict[str, Any], variants: List[str], search_roles: List[str], geo: str, days: int):
    """Returns (jobs, pool update). The careers source found the first time is cached in the pool."""
    name, website = company["name"], company.get("website", "")
    ats, slug, careers = company.get("ats"), company.get("slug"), company.get("careers_url")
    if not ats and not careers and website:
        ats, slug, careers = detect_career_source(website)
        if not ats:
            ats, slug = guess_ats_from_name(name, website)
    postings, source = [], ats or ("Website" if careers else "none")
    try:
        if ats:
            postings = fetch_ats_jobs(ats, slug, search_roles)
        elif careers:
            postings = scrape_generic_career_page(careers, variants)
    except Exception as e:
        print(f"  ⚠️ Career page read issue for {name}: {e}")
    if not postings and website and DISCOVERY_EXA_FALLBACK:
        try:
            for role in search_roles:
                res = exa.search(query=f"{name} {role} job opening", type="auto", include_domains=[_site_key(website)], num_results=3)
                postings += [{"title": (r.title or "").split("|")[0].strip(), "location": "", "url": r.url, "posted": None} for r in res.results]
            source = "Website (Exa)"
        except Exception:
            pass

    now = datetime.now()
    dated = [p["posted"] for p in postings if p.get("posted")]
    jobs = finalize_postings([{**p, "company": name} for p in postings], variants, geo, days, f"Company Discovery ({source})")
    jobs = pick_best_roles(jobs, variants)
    hn = company.get("hn_post")
    if hn and not jobs:   # careers page unreadable or shows no fitting role: the HN post itself is the job
        jobs = finalize_postings([{"title": hn["title"], "location": hn["header"][:150], "url": hn["hn_link"], "posted": hn["posted"],
                                   "company": name}], variants, geo, days, "Hacker News (Who is hiring)")
        for j in jobs:
            j["Also Posted At"] = " | ".join(hn.get("links", [])[:3])
            j["Location"] = geo
    recent = sum(1 for d in dated if (now - d).days <= MOMENTUM_DAYS)
    for j in jobs:
        j["Freshness"] = posting_freshness(j.get("Date Posted"))
        j["Company Stage"] = company.get("stage") or ""
        j["Hiring Momentum"] = f"{recent} posting(s) in {MOMENTUM_DAYS}d" if dated else ""
    update = {"ats": ats, "slug": slug, "careers_url": careers, "checked": now.strftime("%Y-%m-%d"), "matches": len(jobs),
              "recent_posts": recent, "newest_post": max(dated).strftime("%Y-%m-%d") if dated else company.get("newest_post", ""),
              "no_source": 0 if (ats or careers) else int(company.get("no_source") or 0) + 1}
    return jobs, update


# ---------- Hacker News "Who is hiring?" ----------
_HN_CACHE: Dict[str, Any] = {}


def hn_hiring_threads(n: int = None) -> List[Dict[str, Any]]:
    """The latest monthly 'Ask HN: Who is hiring?' stories (posted by the whoishiring account)."""
    n = HN_THREADS if n is None else n
    if "threads" not in _HN_CACHE:
        try:
            r = requests.get(f"{HN_API}/search_by_date", params={"tags": "story,author_whoishiring", "hitsPerPage": 12}, timeout=10)
            hits = r.json().get("hits", []) if r.status_code == 200 else []
        except Exception as e:
            print(f"  ⚠️ Hacker News search issue: {e}")
            hits = []
        _HN_CACHE["threads"] = sorted([h for h in hits if re.match(r"(?i)ask hn: who is hiring", h.get("title") or "")],
                                      key=lambda h: -int(h.get("created_at_i") or 0))
    return _HN_CACHE["threads"][:n]


def parse_hn_post(item: Dict[str, Any]) -> Dict[str, Any]:
    """'Acme (YC W21) | Senior PM | Bangalore or REMOTE (India) | https://acme.com/careers' + body -> fields."""
    html = item.get("text") or ""
    header = BeautifulSoup(re.split(r"(?i)<p>", html)[0], "html.parser").get_text(" ", strip=True)
    soup = BeautifulSoup(html, "html.parser")
    body = soup.get_text("\n", strip=True)
    company = re.split(r"\s*[|•]\s*|\s+[-–—]\s+", header)[0]
    company = re.sub(r"\s*\((?:yc|ycombinator)[^)]*\)|\s*\(https?://[^)]*\)|https?://\S+", "", company, flags=re.I).strip(" :,.")
    links = [a["href"] for a in soup.find_all("a", href=True) if "ycombinator.com" not in a["href"]]
    links += [u for u in re.findall(r"https?://[^\s<>()\"']+", body) if u not in links and "ycombinator.com" not in u]
    ats, slug = next(((a, sl) for a, sl in (_match_ats(u) for u in links) if a), (None, None))
    site = next((_root_url(u) for u in links if not _match_ats(u)[0]
                 and not any(s in u for s in SKIP_COMPANY_DOMAINS + SOCIAL_DOMAINS + ["github.com"])), "")
    posted = datetime.fromtimestamp(item["created_at_i"]) if item.get("created_at_i") else _parse_iso(item.get("created_at"))
    return {"company": company if 1 < len(company) <= 60 else "", "header": header, "body": body, "links": links,
            "ats": ats, "slug": slug, "website": site, "posted": posted,
            "hn_link": f"https://news.ycombinator.com/item?id={item.get('id')}"}


def hn_posts() -> List[Dict[str, Any]]:
    """Top-level job posts of the latest Who-is-hiring threads (one request per thread, cached for the run)."""
    posts = []
    for t in hn_hiring_threads():
        sid = str(t.get("objectID") or t.get("story_id") or "")
        if sid and sid not in _HN_CACHE:
            try:
                r = requests.get(f"{HN_API}/items/{sid}", timeout=30)
                kids = r.json().get("children", []) if r.status_code == 200 else []
            except Exception as e:
                print(f"  ⚠️ Hacker News thread read issue: {e}")
                kids = []
            _HN_CACHE[sid] = [parse_hn_post(k) for k in kids if k.get("text") and k.get("author")]
        posts += _HN_CACHE.get(sid, [])
    return posts


def hn_role_title(post: Dict[str, Any], variants: List[str]) -> str:
    """The fitting role named in the post: a header segment first, then a short body line."""
    for seg in re.split(r"\s*[|•]\s*", post["header"])[1:]:
        if 3 < len(seg) < 120 and title_matches_roles(seg, variants):
            return seg.strip()
    for line in re.split(r"\n|;|\s[-*•]\s", post["body"]):
        line = line.strip(" -*•:")
        if 3 < len(line) < 90 and title_matches_roles(line, variants):
            return line
    return ""


def hn_matching_posts(variants: List[str], geo: str) -> List[Dict[str, Any]]:
    """HN posts that name a fitting role and whose header fits the location (newest first)."""
    out = []
    for p in hn_posts():
        title = hn_role_title(p, variants) if p["company"] else ""
        if title and (p["ats"] or p["website"]) and location_matches(p["header"], geo):
            out.append({**p, "title": title})
    return sorted(out, key=lambda p: p["posted"] or datetime.min, reverse=True)


def hn_company_key(p: Dict[str, Any]) -> str:
    return f"{p['ats']}:{p['slug'][0]}".lower() if p.get("ats") else _site_key(p.get("website") or "")


def add_hn_companies(data: Dict[str, Any], pk: str, variants: List[str], geo: str, skip: set) -> List[Dict[str, Any]]:
    """Adds up to HN_MAX_COMPANIES hiring HN companies to the pool and returns them for this batch."""
    if not HN_ENABLED or HN_MAX_COMPANIES <= 0:
        return []
    pool = data["pools"][pk]["companies"]
    added, seen = [], set(skip)
    for p in hn_matching_posts(variants, geo):
        key = hn_company_key(p)
        if not key or key in seen:
            continue
        seen.add(key)
        rec = pool.setdefault(key, {"name": p["company"], "website": p["website"], "key": key, "stage": "",
                                    "source": "HN Who is hiring", "added": datetime.now().strftime("%Y-%m-%d")})
        if p["ats"] and not rec.get("ats"):
            rec.update(ats=p["ats"], slug=p["slug"])
        rec["newest_post"] = max(rec.get("newest_post") or "", p["posted"].strftime("%Y-%m-%d") if p["posted"] else "")
        added.append({**rec, "hn_post": {k: p[k] for k in ("title", "header", "hn_link", "posted", "company", "links")}})
        if len(added) >= HN_MAX_COMPANIES:
            break
    return added


# Node 2d: Company discovery (pool by CV domain + location, rotated batch)
def discover_companies_node(state: PipelineState) -> Dict[str, Any]:
    if not agent_on(state, "discover"):
        return {"discovery_batch": []}
    typed = (state.get("target_domain") or "").strip()
    if DISCOVERY_DOMAINS:
        domains = [d.strip() for d in re.split(r"[/,]", DISCOVERY_DOMAINS) if d.strip()]
    elif typed and typed.lower() not in ("resume", "cv", "resume specific"):
        domains = [typed]
    else:   # default: the domain your CV is in
        domains = [derive_domain_from_resume(state.get("candidate_name", "Default"), state.get("resume_summary", ""))]
    n = max(10, min(100, int(state.get("discovery_company_count") or 60)))
    per_domain = max(10, n // len(domains))
    print(f"\n🧭 Node 2d: Discovering companies in {', '.join(domains)} ({n} per location, "
          f"≥{DISCOVERY_NEW_SHARE:.0%} new each batch)...")

    variants = expand_role_variants_cached(state["roles"]) if HN_ENABLED else []
    batch = []
    with _POOL_LOCK:
        data = load_company_pool()
        for geo in state["target_geos"]:
            for domain in domains:
                pk = pool_key(domain, geo)
                hist = data["history"].setdefault(pk, {"last_batch": [], "picked": {}})
                companies = ensure_company_pool(data, domain, geo, per_domain)
                chosen = select_company_batch(companies, hist["last_batch"], per_domain, picked=hist["picked"])
                overlap = len({c["key"] for c in chosen} & set(hist["last_batch"]))
                stages = {s: sum(1 for c in chosen if c.get("stage") == s) for s in COMPANY_STAGES}
                print(f"  ✓ {geo} / {domain}: {len(chosen)} companies ({len(chosen) - overlap} new, {overlap} hot carry-overs) | "
                      + ", ".join(f"{s}: {k}" for s, k in stages.items()))
                if len(chosen) < per_domain:
                    print(f"  ℹ️ Only {len(chosen)} fresh companies left in this pool; it grows again at the next top-up.")
                hn = add_hn_companies(data, pk, variants, geo, {c["key"] for c in chosen} | set(hist["last_batch"]))
                if HN_ENABLED:
                    print(f"  🟧 {geo} / {domain}: +{len(hn)} companies from Hacker News 'Who is hiring' naming a fitting role"
                          + (": " + ", ".join(c["name"] for c in hn[:8]) if hn else ""))
                chosen = chosen + hn
                today = datetime.now().strftime("%Y-%m-%d")
                hist["last_batch"] = [c["key"] for c in chosen]
                hist["picked"].update({c["key"]: today for c in chosen})
                batch += [{**c, "geo": geo, "pool": pk} for c in chosen]
        save_company_pool(data)
    return {"discovery_batch": batch}


# Node 2e: Read the discovered companies' careers pages
def search_discovered_careers_node(state: PipelineState) -> Dict[str, Any]:
    batch = state.get("discovery_batch") or []
    if not batch:
        return {"discovery_jobs": []}
    days = state.get("job_duration_days", 3)
    variants = expand_role_variants_cached(state["roles"])
    search_roles = [_normalize_title(r) for r in state["roles"]]
    print(f"\n🏭 Node 2e: Reading careers pages of {len(batch)} discovered companies (past {days} days, "
          f"max {DISCOVERY_MAX_ROLES_PER_COMPANY} roles each)...")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=DISCOVERY_WORKERS) as pool:
        results = list(pool.map(lambda c: scan_discovered_company(c, variants, search_roles, c["geo"], days), batch))

    with _POOL_LOCK:   # remember each company's careers source + hiring activity for the next batch
        data = load_company_pool()
        for c, (_, update) in zip(batch, results):
            rec = data["pools"].get(c["pool"], {}).get("companies", {}).get(c["key"])
            if rec is not None:
                rec.update(update)
        save_company_pool(data)

    jobs = [j for js, _ in results for j in js]
    hot = sorted([(c, u) for c, (_, u) in zip(batch, results) if u["matches"]], key=lambda x: -x[1]["recent_posts"])
    no_source = sum(1 for _, u in results if u["no_source"])
    print(f"  ✓ {len(jobs)} fitting role(s) at {len(hot)} companies in {time.time() - t0:.0f}s "
          f"({no_source} with no readable careers page).")
    for c, u in hot[:10]:
        badge = "🔥" if any(j.get("Freshness", "").startswith("🔥") for j in results[batch.index(c)][0]) else "  "
        print(f"    {badge} {c['name']} [{c.get('stage') or '?'}]: {u['matches']} fit, {u['recent_posts']} posting(s) in {MOMENTUM_DAYS}d")
    return {"discovery_jobs": jobs}


# Fan-out point: LinkedIn, web and company-discovery branches start from here in parallel
def start_search_node(state: PipelineState) -> Dict[str, Any]:
    mode = state.get("search_mode", "both")
    print(f"\n🔀 Starting search branches: {'LinkedIn + web + company discovery (parallel)' if mode in ('both', 'all') else mode + ' agent only'}")
    return {}


# Node 2a: Company Shortlist (role postings in the area + domain companies)
def shortlist_companies_node(state: PipelineState) -> Dict[str, Any]:
    if not agent_on(state, "web"):
        return {}

        # "" = all domains (default); "resume" = your CV's domain; anything else = that domain
    domain = (state.get("target_domain") or "").strip()
    if domain.lower() in ("resume", "cv", "resume specific"):
        domain = derive_domain_from_resume(state.get("candidate_name", "Default"), state.get("resume_summary", ""))
    variants = expand_role_variants_cached(state["roles"])
    count = state.get("career_company_count", 15)
    already = state.get("shortlisted_companies", [])
    seen = {c["key"] for c in already}
    print(f"\n🏢 Node 2a: Shortlisting companies for {state['roles']} in '{domain or 'all domains'}' ({count} per location)...")
    print(f"  ↳ Matching similar titles: {', '.join(variants[:8])}")

    named_funds = state.get("vc_funds") or DEFAULT_VC_FUNDS
    known_boards = {(b["board"], b["geo"]) for b in state.get("fund_boards", [])}
    fund_boards, batch = [], []
    for geo in state["target_geos"]:
        # VC / PE funds: portfolio job boards, or portfolio companies when a fund has no board
        t0 = time.time()
        explored = load_fund_cache(geo, named_funds)
        if explored is None:
            print(f"  ⏱ {geo}: finding VC/PE funds ({len(named_funds)} named + local)...")
            funds = find_vc_funds(geo, named_funds)
            print(f"  ⏱ {geo}: {len(funds)} fund sites found in {time.time() - t0:.0f}s, reading their portfolio pages...")
            with ThreadPoolExecutor(max_workers=8) as pool:
                explored = list(pool.map(lambda f: explore_vc_fund(f, max(3, count // 3)), funds))
            save_fund_cache(geo, named_funds, explored)
        else:
            print(f"  ⚡ {geo}: VC/PE funds loaded from cache (refreshes every {FUND_CACHE_DAYS} days).")
        print(f"  ⏱ {geo}: VC/PE step took {time.time() - t0:.0f}s")
        for e in explored:
            if e["board"] and (e["board"], geo) not in known_boards:
                fund_boards.append({"board": e["board"], "fund": e["fund"], "geo": geo})
                known_boards.add((e["board"], geo))
        portfolio = [c for e in explored for c in e["companies"] if c["key"] not in seen][:count]
        seen.update(c["key"] for c in portfolio)
        for c in portfolio:
            c["geo"] = geo
        batch.extend(portfolio)
        fund_notes = [f"{e['fund']} (" + ("job board" if e["board"] else f"{len(e['companies'])} portfolio cos") + ")" for e in explored]
        print(f"  ✓ {geo}: VC/PE funds → " + (", ".join(fund_notes) or "none found"))

        # Role-first gets most of the slots: these companies are proven to be hiring for the role in the area
        t0 = time.time()
        hiring = discover_hiring_companies(state["roles"], geo, domain, max(1, count * 2 // 3), seen)
        print(f"  ⏱ {geo}: role-posting search took {time.time() - t0:.0f}s ({len(hiring)} companies)")
        seen.update(c["key"] for c in hiring)
        t0 = time.time()
        domain_cos = shortlist_domain_companies(domain, geo, state["roles"], count - len(hiring), seen)
        print(f"  ⏱ {geo}: domain company search took {time.time() - t0:.0f}s ({len(domain_cos)} companies)")
        seen.update(c["key"] for c in domain_cos)
        for c in hiring + domain_cos:
            c["geo"] = geo
        batch.extend(hiring + domain_cos)
        print(f"  ✓ {geo}: {len(hiring)} hiring for the role, {len(domain_cos)} from domain → "
              + ", ".join(c["name"] for c in hiring + domain_cos))

    return {"target_domain": domain, "role_variants": variants, "shortlisted_companies": already + batch,
            "company_batch": batch, "fund_boards": state.get("fund_boards", []) + fund_boards, "new_fund_boards": fund_boards}


# Node 2b: Company Career Page Search
def search_career_pages_node(state: PipelineState) -> Dict[str, Any]:
    if not agent_on(state, "web"):
        return {}

    batch = state.get("company_batch", [])
    days = state.get("job_duration_days", 3)
    variants = state.get("role_variants") or state["roles"]
    search_roles = [_normalize_title(r) for r in state["roles"]]
    print(f"\n🌐 Node 2b: Reading career pages of {len(batch)} companies (past {days} days)...")

    with ThreadPoolExecutor(max_workers=8) as pool:
        results = list(pool.map(lambda c: fetch_career_page_jobs(c, variants, search_roles, c["geo"], days), batch))

        board_results = list(pool.map(lambda b: fetch_fund_board_jobs(b["board"], b["fund"], variants, search_roles, b["geo"], days),
                                      state.get("new_fund_boards", [])))

    jobs = [j for res in results + board_results for j in res]
    print(f"  ✓ {len(jobs)} career-page + VC portfolio postings found.")
    return {"career_jobs": jobs}


# Join point: merge both branches before scoring
def merge_search_results_node(state: PipelineState) -> Dict[str, Any]:
    all_jobs = (state.get("raw_jobs", []) + state.get("career_jobs", []) + state.get("board_jobs", [])
                + state.get("discovery_jobs", []))
    unique_map = {}
    for j in all_jobs:   # one entry per posting ID (tracking params, /apply, trailing '/', LinkedIn country domains ignored)
        if j.get("Job Link"):
            tidy_job_fields(j)
            cid = canonical_job_id(j["Job Link"])
            if cid not in unique_map or is_unknown_company(unique_map[cid].get("Target Company")):
                unique_map[cid] = {**unique_map.get(cid, {}), **j}
            else:   # same posting found by two agents: keep the first, add the other's extra fields (stage, freshness)
                for k, v in j.items():
                    unique_map[cid].setdefault(k, v)
    print(f"\n🧩 Merged {len(unique_map)} unique postings (LinkedIn + web agent + company discovery) for scoring.")
    return {"raw_jobs": list(unique_map.values()), "career_jobs": [], "board_jobs": [], "discovery_jobs": []}


# ==========================================
# SCORING GATE: rejected-jobs list, dedupe, title prefilter, JD experience filter, score cache
# Only jobs that survive this gate reach Node 3's scoring.
# ==========================================
REJECTED_JOBS_FILE = os.path.join(RESUMES_DIR, "rejected_jobs.csv")   # every job you reject at sign-off
SCORE_CACHE_FILE = os.path.join(RESUMES_DIR, "job_score_cache.json")  # scores reused across runs
SCORE_CACHE_DAYS = 14


def _norm_text(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()


def _job_key(job: Dict[str, Any]):
    company = _norm_text(job.get("Target Company"))
    return None if company in ("", "unknown company") else (company, _norm_text(job.get("Job Profile")))


def load_rejected_jobs():
    links, keys = set(), set()
    if os.path.exists(REJECTED_JOBS_FILE):
        import csv
        with open(REJECTED_JOBS_FILE, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                links.add(row.get("link", ""))
                key = _job_key({"Target Company": row.get("company"), "Job Profile": row.get("title")})
                if key:
                    keys.add(key)
    return links, keys


def save_rejected_jobs(jobs: List[Dict[str, Any]]):
    if not jobs:
        return
    import csv
    new_file = not os.path.exists(REJECTED_JOBS_FILE)
    with open(REJECTED_JOBS_FILE, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["rejected_on", "title", "company", "location", "link", "score"])
        for j in jobs:
            w.writerow([datetime.now().strftime("%Y-%m-%d"), j.get("Job Profile"), j.get("Target Company"),
                        j.get("Location"), j.get("Job Link"), j.get("Match Score")])
    print(f"  🗑️ {len(jobs)} rejected job(s) saved; they won't be shown again.")


def ask_scoring_mode(n_jobs: int) -> str:
    ans = input(f"\n{n_jobs} jobs passed the filters. Scoring: 'title' = instant keyword match on job titles (no AI), "
                f"or Enter for 'full' = AI reads each JD against your full CV [Default: full]: ").strip().lower()
    return "title" if ans.startswith("t") else "full"


def title_keyword_score(job: Dict[str, Any], state: PipelineState) -> Dict[str, Any]:
    """'title' mode: no AI. Exact role phrase 80, all role words 72, similar title 65."""
    t = _normalize_title(job.get("Job Profile", ""))
    for role in state.get("roles", []):
        if _normalize_title(role).strip() in t:
            return {"match_score": 80, "rating": "Good", "rationale": f"Title match only: '{role}'.", "scored_by": "Title keywords"}
    if title_matches_roles(job.get("Job Profile", ""), state.get("roles", [])):
        return {"match_score": 72, "rating": "Good", "rationale": "Title match only: all role keywords present.", "scored_by": "Title keywords"}
    return {"match_score": 65, "rating": "Average", "rationale": "Title match only: similar title.", "scored_by": "Title keywords"}


def score_cache_key(job: Dict[str, Any], state: PipelineState) -> str:
    return f"{state.get('candidate_name', 'Default')}|{'/'.join(state.get('roles', [])).lower()}|{job.get('Job Link')}"


def load_score_cache() -> Dict[str, Any]:
    try:
        cache = json.load(open(SCORE_CACHE_FILE, encoding="utf-8"))
    except Exception:
        return {}
    cutoff = (datetime.now() - timedelta(days=SCORE_CACHE_DAYS)).strftime("%Y-%m-%d")
    return {k: v for k, v in cache.items() if v.get("cached_on", "") >= cutoff}


def save_score_cache(cache: Dict[str, Any]):
    today = datetime.now().strftime("%Y-%m-%d")
    for v in cache.values():
        v.setdefault("cached_on", today)
    with open(SCORE_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f)


CLAUDE_MAX_JOBS_PER_RUN = 40   # applies only when JOB_SCORER=claude (Gemini scores every job); the rest go to Qwen
CLAUDE_JOBS_SENT = [0]         # running count for this run (resets when the cell is re-run)
CLAUDE_FALLBACK_SENT = [0]     # jobs Claude scored as the last fallback (after Gemini + Qwen failed), same 40 cap
_IN_CLAUDE_FALLBACK = [False]

# Claude token usage per scoring batch (read from the CLI's final "result" event)
CLAUDE_USAGE_RUN: List[Dict[str, Any]] = []


def log_claude_usage(event: Dict[str, Any], prompt: str, streamed_text: str):
    """Prints this batch's Claude tokens. If the CLI was stopped before reporting, prints a labelled estimate (~4 chars/token)."""
    u = (event or {}).get("usage") or {}
    if u:
        row = {"input": u.get("input_tokens", 0), "cache_write": u.get("cache_creation_input_tokens", 0),
               "cache_read": u.get("cache_read_input_tokens", 0), "output": u.get("output_tokens", 0),
               "cost": event.get("total_cost_usd") or 0.0, "estimated": False}
    else:
        row = {"input": len(prompt) // 4, "cache_write": 0, "cache_read": 0, "output": len(streamed_text) // 4,
               "cost": 0.0, "estimated": True}
    CLAUDE_USAGE_RUN.append(row)
    total_in = row["input"] + row["cache_write"] + row["cache_read"]
    label = "ESTIMATE (CLI stopped before reporting usage)" if row["estimated"] else f"list-price equiv. ${row['cost']:.4f}"
    print(f"  🔢 Claude tokens this batch: {total_in:,} in (new {row['input']:,}, cache write {row['cache_write']:,}, "
          f"cache read {row['cache_read']:,}) + {row['output']:,} out | {label}")


def print_claude_usage_total():
    if not CLAUDE_USAGE_RUN:
        return
    t = {k: sum(r[k] for r in CLAUDE_USAGE_RUN) for k in ("input", "cache_write", "cache_read", "output", "cost")}
    est = sum(r["estimated"] for r in CLAUDE_USAGE_RUN)
    note = "  (on Claude Pro this counts toward your usage limits, it isn't billed)" if SCORER == "claude" else "  (Gemini free tier, not billed)"
    print(f"  🔢 {SCORER_LABEL} tokens this pass ({len(CLAUDE_USAGE_RUN)} batch(es)): "
          f"{t['input'] + t['cache_write'] + t['cache_read']:,} in + {t['output']:,} out"
          + (f" | list-price equiv. ${t['cost']:.4f}" if SCORER == "claude" else "")
          + (f" | {est} batch(es) estimated" if est else "") + note)
    CLAUDE_USAGE_RUN.clear()


# ==========================================
# RELEVANCE SCORING HELPERS: real JD text + full CV for Node 3
# ==========================================
SCORING_JD_CHARS = 1800   # JD text Claude sees per job (the top of a JD holds the role + requirements)
QWEN_JD_CHARS = 400       # Qwen 7B has a small context, so it gets a shorter excerpt
CV_TEXT_CHARS = 8000
qwen_scorer = ResilientLLM(num_ctx=8192, temperature=0, allow_gemini=False, allow_claude=False)  # scoring handles Claude itself
_JD_CACHE: Dict[str, str] = {}
_CV_TEXT_CACHE: Dict[str, str] = {}


# JD trimming: keep only role / responsibilities / requirements; drop company, culture, perks, EEO (no LLM)
_JD_KEEP_HEADINGS = re.compile(
    r"^(about (the|this) (role|job|position|opportunity)|the role|your role|role (overview|summary|description)|job (description|summary)|"
    r"(key |main |core )?responsibilit|duties|what you('ll| will)? (do|own|work on|be doing)|in this role|the (job|position|opportunity)|your (mission|impact|tasks)|"
    r"(minimum |basic |preferred |key )?(requirements|qualifications)|what you('ll| will)? (bring|need)|what we('re| are)? looking for|"
    r"who you are|you (have|bring|are|will)|must[- ]haves?|nice[- ]to[- ]haves?|bonus points|(skills|experience)( (&|and) (experience|skills))?( required)?|your (profile|background|skills|experience))\b", re.I)
_JD_DROP_HEADINGS = re.compile(
    r"^(about\b|who we are|our (mission|vision|values|culture|story|team|company|commitment|benefits)|why (join|work|you('ll)? love)|"
    r"what we offer|we offer|benefits|perks|compensation|salary|pay (range|transparency)|life at|working (at|here|with us)|culture|"
    r"equal (employment )?opportunit|eeo|diversity|inclusion|accommodation|how to apply|(our |the )?(hiring|interview|recruitment) process|"
    r"privacy|disclaimer|additional information|location (and|&) benefits)", re.I)
_JD_DROP_SENTENCE = re.compile(
    r"equal opportunity|regardless of (race|gender)|reasonable accommodation|health (insurance|care)|dental|paid time off|\bpto\b|"
    r"parental leave|wellness|gym|stock options|\besop\b|401\(?k|pension|snacks|team (offsites|events)|flexible (hours|working)|"
    r"we are (a|an|the) (leading|fast-growing|global)|our mission is|founded in \d{4}|backed by", re.I)


def extract_role_sections(raw: str) -> str:
    """raw = JD text with one line per paragraph/bullet. Falls back to the original when no sections are found."""
    lines = [l.strip(" \t•·-–*:") for l in raw.split("\n") if l.strip()]
    is_heading = lambda l: len(l) <= 70 and len(l.split()) <= 8
    has_keep = any(is_heading(l) and _JD_KEEP_HEADINGS.match(l) for l in lines)
    keep = not has_keep  # text before the first role/requirements heading is usually the company blurb
    out = []
    for l in lines:
        if is_heading(l) and _JD_KEEP_HEADINGS.match(l):
            keep = True
        elif is_heading(l) and _JD_DROP_HEADINGS.match(l):
            keep = False
            continue
        if keep and not _JD_DROP_SENTENCE.search(l):
            out.append(l)
    trimmed = "\n".join(out)
    return trimmed if len(trimmed) >= 300 else raw

# Seniority = the years of experience the JD asks for (never the job title)
EXPERIENCE_STRETCH_YEARS = 1      # keep jobs asking up to this many years MORE than you have
EXPERIENCE_OVERQUALIFIED_YEARS = 3  # drop "2-4 years" style ranges whose top is this far BELOW your years
_JD_YEARS: Dict[str, Any] = {}    # job link -> (min_years, max_years or None), read from the JD
_YEARS_NUM = r"(\d{1,2})(?:\.\d)?"
_YEARS_PATTERNS = [  # (regex, kind); only matches near the word "experience" count
    (re.compile(rf"{_YEARS_NUM}\s*(?:-|–|—|to)\s*{_YEARS_NUM}\s*\+?\s*(?:years?|yrs?)", re.I), "range"),
    (re.compile(rf"(?:minimum|min\.?|at least)\s*(?:of\s*)?{_YEARS_NUM}\s*\+?\s*(?:years?|yrs?)", re.I), "min"),
    (re.compile(rf"{_YEARS_NUM}\s*(?:\+|plus|or more)\s*(?:years?|yrs?)", re.I), "min"),
    (re.compile(rf"{_YEARS_NUM}\s*(?:years?|yrs?)\s*(?:of\s*)?(?:\w+\s*){{0,4}}experience", re.I), "min"),
]


def required_experience(text: str):
    """(min_years, max_years) from the JD, using the main (largest) requirement; None if the JD states none."""
    best, taken = None, []  # taken = spans already used by a range, so "3-5 years" isn't also read as "5 years"
    for rx, kind in _YEARS_PATTERNS:
        for m in rx.finditer(text or ""):
            if any(a <= m.start() < b or a < m.end() <= b for a, b in taken):
                continue
            context = text[max(0, m.start() - 80): m.end() + 80].lower()
            if "experience" not in context and "exp" not in context:
                continue
            lo = int(m.group(1))
            hi = int(m.group(2)) if kind == "range" else None
            if lo > 25 or (hi is not None and (hi > 30 or hi < lo)):
                continue
            taken.append(m.span())
            if best is None or lo > best[0]:
                best = (lo, hi)
    return best


def experience_fits(job_link: str, my_years) -> bool:
    req = _JD_YEARS.get(job_link)
    if my_years is None or not req:
        return True  # no years given / JD states no requirement -> keep
    lo, hi = req
    if lo > my_years + EXPERIENCE_STRETCH_YEARS:
        return False  # asks for much more experience
    return not (hi is not None and hi < my_years - EXPERIENCE_OVERQUALIFIED_YEARS)  # range clearly below you


_MONTHS = {m: i for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}
_MON = r"(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?"
_DATE_RANGE = re.compile(
    rf"(?:{_MON}\s*'?|(\d{{1,2}})\s*[/.-]\s*)?((?:19|20)\d{{2}})\s*(?:-|–|—|to|until|till)\s*"
    rf"(?:(?:{_MON}\s*'?|(\d{{1,2}})\s*[/.-]\s*)?((?:19|20)\d{{2}})|(present|current|now|today|date|ongoing))", re.I)
_EDU_WORDS = re.compile(r"universit|college|school|bachelor|master|b\.?\s?tech|b\.?\s?e\b|m\.?\s?tech|mba|b\.?sc|m\.?sc|ph\.?d|degree|education|diploma|cgpa|gpa", re.I)


def years_from_date_ranges(cv_text: str):
    """Total work experience from ranges like 'Jan 2018 – Present' or '03/2016 - 12/2019', overlaps merged and
    education ranges skipped. Returns (years, number_of_ranges) or (None, 0)."""
    now = datetime.now()
    spans = []
    for m in _DATE_RANGE.finditer(cv_text or ""):
        line_start = cv_text.rfind("\n", 0, cv_text.rfind("\n", 0, m.start())) + 1  # this line + the one above
        line_end = cv_text.find("\n", m.end())
        if _EDU_WORDS.search(cv_text[line_start: line_end if line_end != -1 else len(cv_text)]):
            continue
        m1 = _MONTHS.get((m.group(1) or "")[:3].lower()) or (int(m.group(2)) if m.group(2) and 1 <= int(m.group(2)) <= 12 else 1)
        start = int(m.group(3)) * 12 + m1 - 1
        if m.group(7):
            end = now.year * 12 + now.month - 1
        else:
            m2 = _MONTHS.get((m.group(4) or "")[:3].lower()) or (int(m.group(5)) if m.group(5) and 1 <= int(m.group(5)) <= 12 else 12)
            end = int(m.group(6)) * 12 + m2 - 1
        if 0 <= end - start <= 40 * 12:
            spans.append((start, end + 1))
    if not spans:
        return None, 0
    spans.sort()
    merged = [list(spans[0])]
    for s, e in spans[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    months = sum(e - s for s, e in merged)
    return round(months / 12 * 2) / 2, len(spans)  # nearest half year


def parse_candidate_years(text: str, persona_key: str):
    """Years typed at start-up > CANDIDATE_YEARS in .env > '8+ years of experience' in the CV > employment date
    ranges in the CV. None = no experience filter."""
    m = re.search(r"\d+(\.\d+)?", text or "") or re.search(r"\d+(\.\d+)?", os.getenv("CANDIDATE_YEARS", ""))
    if m:
        return float(m.group(0))
    path = os.path.join(RESUMES_DIR, f"{persona_key}.pdf")
    try:
        cv_text = "\n".join(d.page_content for d in PyPDFLoader(path).load()) if os.path.exists(path) else ""
    except Exception:
        cv_text = load_full_cv_text(persona_key)
    found = [int(x) for x in re.findall(r"(\d{1,2})\s*\+?\s*years?\s*(?:of\s*)?(?:\w+\s*){0,3}experience", cv_text, re.I)]
    found = [y for y in found if y <= 40]
    if found:
        print(f"  📏 Read {max(found)} years of experience from your CV.")
        return float(max(found))
    years, n = years_from_date_ranges(cv_text)
    if years:
        print(f"  📏 Worked out ~{years:g} years of experience from {n} job date range(s) in your CV "
              f"(overlaps merged, education skipped). Override with CANDIDATE_YEARS=... in .env.")
        return years
    print("  ⚠️ Couldn't read your years of experience from the CV, so the experience filter is off. "
          "Type it at the prompt or add CANDIDATE_YEARS=8 to .env.")
    return None


def fetch_jd_for_scoring(url: str) -> str:
    """Posting text for scoring. LinkedIn guest pages keep the JD in one div; other sites use the page text."""
    if url in _JD_CACHE:
        return _JD_CACHE[url]
    text = ""
    try:
        resp = requests.get(url, headers=CAREER_HEADERS, timeout=8)
        soup = BeautifulSoup(resp.text, "html.parser")
        box = soup.select_one("div.show-more-less-html__markup, div.description__text, [class*='job-description'], [class*='jobDescription'], main, article")
        for tag in soup(["script", "style", "nav", "header", "footer", "noscript"]):
            tag.decompose()
        text = re.sub(r"\s+", " ", extract_role_sections((box or soup).get_text("\n", strip=True)))  # role + requirements only
    except Exception:
        pass
    _JD_YEARS[url] = required_experience(text)  # read before the 1,800-char cut so late requirements still count
    _JD_CACHE[url] = text[:SCORING_JD_CHARS] if len(text) > 200 else ""  # tiny text = JS-only page / login wall
    return _JD_CACHE[url]


def attach_job_descriptions(jobs: List[Dict[str, Any]]) -> Dict[str, str]:
    """Fetches every JD in parallel. Returns {job link: JD text} (kept out of the job dicts so Excel is unchanged)."""
    from concurrent.futures import ThreadPoolExecutor
    links = [j.get("Job Link") for j in jobs if j.get("Job Link")]
    with ThreadPoolExecutor(max_workers=8) as pool:
        texts = list(pool.map(fetch_jd_for_scoring, links))
    found = sum(1 for t in texts if t)
    print(f"  📄 Read {found}/{len(links)} job descriptions (the rest are scored on title + company only).")
    return dict(zip(links, texts))


def load_full_cv_text(persona_key: str) -> str:
    """Full CV text, so scoring isn't limited to the 3-chunk summary."""
    if persona_key not in _CV_TEXT_CACHE:
        path = os.path.join(RESUMES_DIR, f"{persona_key}.pdf")
        try:
            text = "\n".join(d.page_content for d in PyPDFLoader(path).load()) if os.path.exists(path) else ""
        except Exception:
            text = ""
        _CV_TEXT_CACHE[persona_key] = re.sub(r"[ \t]+", " ", text)[:CV_TEXT_CHARS]
    return _CV_TEXT_CACHE[persona_key]


def build_scoring_profile(state: PipelineState) -> str:
    cv = load_full_cv_text(state.get("candidate_name", "Default"))
    return (f"Target roles: {', '.join(state.get('roles', []))}\n"
            f"Target locations: {', '.join(state.get('target_geos', []))}\n\n"
            f"Summary:\n{state.get('resume_summary', '')}\n\n"
            f"Full CV:\n{cv or '(CV text not available, use the summary)'}")


def build_jobs_payload(batch: List[Dict[str, Any]], jd_map: Dict[str, str]) -> str:
    """One line per job (the partial-result recovery reads lines starting with 'ID n:')."""
    lines = []
    for i, j in enumerate(batch):
        jd = jd_map.get(j.get("Job Link"), "")
        lines.append(f"ID {i+1}: {j.get('Job Profile')} at {j.get('Target Company')} ({j.get('Location')}) "
                     f"| JD: {jd or 'NOT AVAILABLE'}")
    return "\n".join(lines)


# ==========================================
# SCORER SETTINGS: Gemini (AI Studio free tier) first, Qwen fallback; Claude kept behind a setting
# ==========================================
SCORER = os.getenv("JOB_SCORER", "gemini").strip().lower()      # "gemini" (default) | "claude" | "qwen"
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")    # e.g. gemini-3.5-flash-lite for higher free limits
GEMINI_FALLBACK_MODEL = os.getenv("GEMINI_FALLBACK_MODEL", "gemini-3.5-flash-lite")  # busy/rate-limited: 1 try; daily quota or 404: rest of run
GEMINI_RPM = int(os.getenv("GEMINI_RPM", "5"))                  # requests/minute to stay under (see AI Studio > Rate limits)
SCORER_LABEL = {"gemini": f"Gemini ({GEMINI_MODEL})", "claude": "Claude Pro", "qwen": "Qwen"}.get(SCORER, SCORER)
CALIBRATION_FILE = os.path.join(RESUMES_DIR, "score_calibration.json")  # Qwen -> primary scorer score mapping (made once)
PRIMARY_DISABLED = [False, ""]   # set when the free-tier quota is used up, so the rest of the run goes to Qwen
_GEMINI = {"client": None, "last_call": 0.0, "dead": set()}   # dead = models out for this run (daily quota / 404)


def gemini_active_model():
    """Main model, or Flash-Lite once the main model's daily quota is gone (or it isn't available to the key)."""
    return next((m for m in dict.fromkeys([GEMINI_MODEL, GEMINI_FALLBACK_MODEL]) if m and m not in _GEMINI["dead"]), None)


def _is_daily_quota(e) -> bool:
    msg = str(e).lower().replace(" ", "")
    return getattr(e, "code", None) == 429 and ("perday" in msg or "daily" in msg)

SCORING_RUBRIC = """Scoring Rules (score each job 0-100 as the SUM of these weighted criteria; judge from the JD text, not the title):
1. Responsibilities fit (0-35): does the day-to-day work in the JD match work the candidate has actually done in the CV? Cite the overlap.
2. Skills & domain fit (0-25): required skills, tools, industry and product type vs evidence in the CV.
3. Seniority fit (0-20): years of experience and scope the JD asks for vs the candidate's. Never judge seniority from the job title. Far above or far below = low.
4. Location & eligibility (0-10): location, remote policy, language and work-authorisation requirements vs the target locations.
5. Hard requirements (0-10): full marks unless the JD has an explicit must-have the CV clearly lacks (specific degree, language, clearance, N+ years in X).
Calibration:
- Score ONLY the role, responsibilities and requirements. Ignore company description, mission, culture, benefits and perks entirely.
- Do NOT give high scores for a matching title alone. 85+ only when criteria 1-3 are all strong with concrete evidence.
- If the JD is NOT AVAILABLE, score from title + company only, cap the score at 70, and start the rationale with "No JD:".
- Rating: Perfect >= 85, Good 70-84, Average 55-69, Low < 55.
- rationale: max 25 words naming the specific JD requirement(s) that drove the score. low_match_reason: the main gap, or null.
- breakdown: the five sub-scores as "resp/35 skills/25 seniority/20 loc/10 reqs/10".

Output format (one line per job):
{"id": 1, "match_score": 86, "rating": "Perfect", "breakdown": "31/35 22/25 16/20 9/10 8/10", "rationale": "Owns LLM product roadmap and B2B discovery, same as CV's AI PM work; asks 5+ yrs PM, candidate fits.", "low_match_reason": null}
{"id": 2, "match_score": 48, "rating": "Low", "breakdown": "12/35 10/25 10/20 8/10 8/10", "rationale": "Title says PM but JD is hands-on ML pipeline engineering in PyTorch.", "low_match_reason": "Role is ML engineering, not product ownership."}"""


def _gemini_client():
    if _GEMINI["client"] is None:
        if not GEMINI_API_KEY:
            raise RuntimeError("GEMINI_API_KEY is empty: add GEMINI_API_KEY=AIza... to the .env next to this notebook, then re-run the cell.")
        try:
            from google import genai
        except ImportError:
            raise RuntimeError("google-genai isn't installed in this notebook's Python: run  %pip install -U google-genai  in a cell, then restart the kernel.")
        _GEMINI["client"] = genai.Client(api_key=GEMINI_API_KEY)
    return _GEMINI["client"]


def _parse_score_items(text: str) -> List[Dict[str, Any]]:
    """Accepts a JSON array, {"jobs": [...]} or JSON lines."""
    text = (text or "").strip()
    try:
        data = json.loads(text)
        data = data.get("jobs", data.get("results", [])) if isinstance(data, dict) else data
        return [d for d in data if isinstance(d, dict)]
    except Exception:
        items = []
        for line in text.split("\n"):
            try:
                d = json.loads(line.strip().strip(","))
                if isinstance(d, dict):
                    items.append(d)
            except Exception:
                pass
        return items


def log_gemini_usage(resp):
    u = getattr(resp, "usage_metadata", None)
    row = {"input": (getattr(u, "prompt_token_count", 0) or 0) - (getattr(u, "cached_content_token_count", 0) or 0),
           "cache_write": 0, "cache_read": getattr(u, "cached_content_token_count", 0) or 0,
           "output": (getattr(u, "candidates_token_count", 0) or 0) + (getattr(u, "thoughts_token_count", 0) or 0),
           "cost": 0.0, "estimated": u is None}
    CLAUDE_USAGE_RUN.append(row)
    print(f"  🔢 Gemini tokens this batch: {row['input'] + row['cache_read']:,} in + {row['output']:,} out "
          f"(incl. {getattr(u, 'thoughts_token_count', 0) or 0:,} thinking) | free tier")


def call_gemini_evaluation(resume_summary: str, jobs_payload: str) -> Dict[int, Any]:
    """Scores one batch with Gemini (JSON mode), paced to GEMINI_RPM. Jobs it leaves out go to Qwen."""
    client = _gemini_client()
    from google.genai import types, errors
    total = len(re.findall(r"^ID \d+:", jobs_payload, re.M))
    prompt = (f"You are a job-scoring function. Return ONLY a JSON array with one object per job, in ID order.\n\n"
              f"Candidate Profile:\n{resume_summary}\n\nJobs to Evaluate:\n{jobs_payload}\n\n{SCORING_RUBRIC}\n"
              f"Return the objects above as ONE JSON array (not one per line).")
    def _config(m):
        return types.GenerateContentConfig(
            response_mime_type="application/json", temperature=0,
            thinking_config=types.ThinkingConfig(thinking_level="low") if m.startswith("gemini-3") else None)

    model, backoff, attempt = gemini_active_model(), [10, 20, 30], 0  # ~60s of retries for busy errors per model
    if not model:
        raise RuntimeError("No Gemini model left this run (daily quota used up / not available).")
    while True:
        wait = 60.0 / max(GEMINI_RPM, 1) - (time.time() - _GEMINI["last_call"])
        if wait > 0:
            _GEMINI["waiting"] = True   # lets Qwen take a batch meanwhile (shared-queue mode)
            time.sleep(wait)  # stay under the free-tier requests-per-minute limit
            _GEMINI["waiting"] = False
        _GEMINI["last_call"] = time.time()
        try:
            resp = client.models.generate_content(model=model, contents=prompt, config=_config(model))
            break
        except errors.APIError as e:
            daily = _is_daily_quota(e)
            if daily or e.code == 404:   # this model is out for the run -> Flash-Lite (with its own free quota) takes over
                _GEMINI["dead"].add(model)
                nxt = gemini_active_model()
                if nxt:
                    print(f"  🔀 {model}: {'daily free quota used up' if daily else 'not available to your key (404)'}. "
                          f"Switching to {nxt} for the rest of this run.")
                    model, attempt = nxt, 0
                    continue
                if daily:
                    PRIMARY_DISABLED[:] = [True, "free-tier daily quota used up on every Gemini model"]  # rest of run -> Qwen
                    raise RuntimeError("Gemini daily free-tier quota used up on every model (resets at midnight Pacific).")
            if e.code in (429, 500, 503, 504) and attempt < len(backoff):
                print(f"  ⏳ {model} {e.code} ({'busy' if e.code != 429 else 'rate limit'}), retrying in {backoff[attempt]}s "
                      f"({attempt + 1}/{len(backoff)})...")
                _GEMINI["waiting"] = True
                time.sleep(backoff[attempt])
                _GEMINI["waiting"] = False
                attempt += 1
                continue
            fb = GEMINI_FALLBACK_MODEL
            if e.code in (429, 500, 503, 504) and fb and model != fb and fb not in _GEMINI["dead"]:
                print(f"  ⏳ {model} still {'rate-limited' if e.code == 429 else 'busy'}, trying {fb} for this batch...")
                model, attempt = fb, len(backoff)   # one try; the next batch starts on the main model again
                continue
            hint = {400: "check GEMINI_API_KEY is a valid AI Studio key", 401: "API key rejected",
                    403: "API key not allowed for this model/project", 404: f"model '{model}' not available to your key: "
                    "set GEMINI_MODEL in .env (e.g. gemini-3.5-flash-lite)"}.get(e.code, "temporary, next batch tries Gemini again")
            raise RuntimeError(f"Gemini error {e.code} ({hint}): {str(e)[:140]}")

    log_gemini_usage(resp)
    results = {}
    for item in _parse_score_items(resp.text):
        try:
            item["scored_by"] = f"Gemini ({model})"
            results.setdefault(int(item["id"]), item)
        except (KeyError, TypeError, ValueError):
            pass
    print(f"  🧮 Gemini scored {len(results)}/{total} jobs")
    if not results:
        raise RuntimeError("Gemini returned no scores.")

    missing = [l for l in jobs_payload.split("\n")
               if re.match(r"ID (\d+):", l) and int(re.match(r"ID (\d+):", l).group(1)) not in results]
    if missing:
        print(f"  🔁 Switching to Qwen: Gemini returned {len(results)}/{total}. Qwen scores the remaining {len(missing)} job(s) of this batch.")
        results.update(evaluate_with_local_qwen(resume_summary, "\n".join(missing)))
    return results


SHARED_QUEUE = os.getenv("SCORING_SHARED_QUEUE", "0").strip().lower() in ("1", "true", "yes", "on")  # off by default


def score_with_shared_queue(batches: List[List[int]], raw_jobs: List[Dict[str, Any]], jd_map: Dict[str, str], profile: str):
    """Gemini and Qwen pull 10-job batches from one queue. Qwen only takes a batch while Gemini is waiting on its
    per-minute limit (or once Gemini's quota is gone). Turn on with SCORING_SHARED_QUEUE=1 in .env."""
    import threading
    queue, lock, scored = list(batches), threading.Lock(), []

    def take():
        with lock:
            return queue.pop(0) if queue else None

    def run(positions, use_gemini: bool):
        batch = [raw_jobs[p] for p in positions]
        payload = build_jobs_payload(batch, jd_map)
        if use_gemini:
            CLAUDE_JOBS_SENT[0] += len(batch)
            try:
                bm = call_gemini_evaluation(profile, payload)
                print(f"  ✓ Batch of {len(batch)} scored via {SCORER_LABEL}!")
                maybe_calibrate(profile, payload, bm)
            except Exception as e:
                print(f"  ⚠️ {SCORER_LABEL} notice: {e}")
                print(f"  🔁 Switching to Qwen: {SCORER_LABEL} call failed. Qwen scores this batch of {len(batch)} job(s).")
                bm = evaluate_with_local_qwen(profile, payload)
        elif _GEMINI.get("waiting") and not PRIMARY_DISABLED[0]:
            print(f"  🔀 Qwen takes a batch of {len(batch)} while {SCORER_LABEL} waits on its rate limit.")
            bm = evaluate_with_local_qwen(profile, payload)
        else:
            print(f"  🔁 Qwen scores a batch of {len(batch)} ({SCORER_LABEL} unavailable).")
            bm = evaluate_with_local_qwen(profile, payload)
        with lock:
            scored.append((positions, batch, bm))

    def gemini_worker():
        while not PRIMARY_DISABLED[0]:
            positions = take()
            if positions is None:
                return
            run(positions, True)
        if queue:
            print(f"  🔁 Switching to Qwen: {SCORER_LABEL} unavailable ({PRIMARY_DISABLED[1]}). Qwen scores the remaining {sum(map(len, queue))} job(s).")

    def qwen_worker():
        while True:
            if not queue:
                return
            if _GEMINI.get("waiting") or PRIMARY_DISABLED[0] or not gem.is_alive():
                positions = take()
                if positions is None:
                    return
                run(positions, False)
            else:
                time.sleep(0.5)

    print(f"  🔀 Shared queue: {SCORER_LABEL} + Qwen on {len(batches)} batch(es).")
    gem = threading.Thread(target=gemini_worker)
    qwen = threading.Thread(target=qwen_worker)
    gem.start(); qwen.start()
    gem.join(); qwen.join()
    return scored


def gemini_text(prompt: str) -> str:
    """Plain-text Gemini call (paced), used when local Qwen can't run. Main model first, then Flash-Lite."""
    from google.genai import errors
    last = None
    for model in [m for m in dict.fromkeys([GEMINI_MODEL, GEMINI_FALLBACK_MODEL]) if m and m not in _GEMINI["dead"]]:
        wait = 60.0 / max(GEMINI_RPM, 1) - (time.time() - _GEMINI["last_call"])
        if wait > 0:
            time.sleep(wait)
        _GEMINI["last_call"] = time.time()
        try:
            return _gemini_client().models.generate_content(model=model, contents=prompt).text or ""
        except errors.APIError as e:
            last = e
            if e.code == 404 or _is_daily_quota(e):
                _GEMINI["dead"].add(model)
            if e.code in (404, 429, 500, 503, 504):
                print(f"  ⏳ {model} {e.code} on a text call, trying the next Gemini model...")
                continue
            raise
    raise RuntimeError(f"No Gemini model answered ({last})")


def call_primary_scorer(resume_summary: str, jobs_payload: str) -> Dict[int, Any]:
    if SCORER == "gemini":
        return call_gemini_evaluation(resume_summary, jobs_payload)
    return call_claude_pro_evaluation(resume_summary, jobs_payload)


def _load_calibration() -> Dict[str, Any]:
    try:
        return json.load(open(CALIBRATION_FILE, encoding="utf-8"))
    except Exception:
        return {}


_CALIB_SKIP = [False]  # set when local Qwen can't run, so calibration isn't retried every batch


def maybe_calibrate(resume_summary: str, jobs_payload: str, primary_map: Dict[int, Any]):
    """One time per scorer: Qwen re-scores the first primary batch, and a line maps Qwen scores onto the primary's scale."""
    calib = _load_calibration()
    if SCORER == "qwen" or SCORER in calib or _CALIB_SKIP[0]:
        return
    pairs = []
    _IN_CLAUDE_FALLBACK[0] = True  # calibration compares Qwen only, never Claude
    try:
        qwen_map = evaluate_with_local_qwen(resume_summary, jobs_payload)
    finally:
        _IN_CLAUDE_FALLBACK[0] = False
    for i, ev in primary_map.items():
        q = qwen_map.get(i) or qwen_map.get(str(i))
        if q and not str(ev.get("scored_by", "")).startswith("Qwen"):
            try:
                pairs.append((float(q["match_score"]), float(ev["match_score"])))
            except (KeyError, TypeError, ValueError):
                pass
    if not qwen_map:
        _CALIB_SKIP[0] = True
        print("  📐 Calibration skipped this run (local Qwen unavailable).")
        return
    if len(pairs) < 5:
        print(f"  📐 Calibration skipped (only {len(pairs)} comparable jobs); will try again next batch.")
        return
    mq = sum(q for q, _ in pairs) / len(pairs)
    mp = sum(p for _, p in pairs) / len(pairs)
    var_q = sum((q - mq) ** 2 for q, _ in pairs) / len(pairs)
    a = 1.0 if var_q < 25 else max(0.5, min(1.5, sum((q - mq) * (p - mp) for q, p in pairs) / len(pairs) / var_q))
    b = mp - a * mq
    calib[SCORER] = {"a": round(a, 3), "b": round(b, 2), "n": len(pairs), "made_on": datetime.now().strftime("%Y-%m-%d")}
    with open(CALIBRATION_FILE, "w", encoding="utf-8") as f:
        json.dump(calib, f, indent=1)
    print(f"  📐 Calibration saved: Qwen score × {a:.2f} {'+' if b >= 0 else '-'} {abs(b):.1f} ≈ {SCORER_LABEL} score "
          f"(from {len(pairs)} jobs). Runs once; delete score_calibration.json to redo.")


def apply_calibration(batch_map: Dict[int, Any]) -> Dict[int, Any]:
    c = _load_calibration().get(SCORER)
    for ev in batch_map.values():
        if c and str(ev.get("scored_by", "")).startswith("Qwen") and not ev.get("calibrated"):
            try:
                ev["match_score"] = max(0, min(100, round(c["a"] * float(ev["match_score"]) + c["b"])))
            except (KeyError, TypeError, ValueError):
                continue
            s = ev["match_score"]
            ev["rating"] = "Perfect" if s >= 85 else "Good" if s >= 70 else "Average" if s >= 55 else "Low"
            ev["scored_by"] = f"Qwen (calibrated to {SCORER_LABEL})"
            ev["calibrated"] = True
    return batch_map


# Direct Headless Claude Pro Evaluation (Windows-safe list args)
# Direct Headless Claude Pro Evaluation (streams scores live; keeps partial results)
def call_claude_pro_evaluation(resume_summary: str, jobs_payload: str, timeout_s: int = 270) -> Dict[int, Any]:
    import threading
    if not os.path.exists(CLAUDE_EXE):
        raise RuntimeError("Claude CLI executable not found at specified path.")

    total = len(re.findall(r"^ID \d+:", jobs_payload, re.M))
    prompt = f"""[CRITICAL INSTRUCTION: You are an automated batch evaluation script. Do NOT greet. Do NOT offer services. Output ONLY JSON Lines: exactly one JSON object per line, one line per job, in ID order. No array brackets, no markdown, nothing else.]

Candidate Profile:
{resume_summary}

Jobs to Evaluate:
{jobs_payload}

{SCORING_RUBRIC}
"""
    results = {}

    def record(line: str):
        try:
            item = json.loads(line.strip().strip(","))
            if isinstance(item, dict) and "id" in item and int(item["id"]) not in results:
                item["scored_by"] = "Claude Pro"
                results[int(item["id"])] = item
                print(f"\r  🧮 Claude scored {len(results)}/{total} jobs", end="", flush=True)
        except Exception:
            pass

    # List argument structure prevents cmd.exe from truncating multi-line strings
    # Prompt goes in via stdin: with JDs it is longer than Windows' 32k command-line limit
    proc = subprocess.Popen(
        [CLAUDE_EXE, "-p", "--tools", "", "--strict-mcp-config", "--system-prompt", "You are a strict JSON job-scoring script.",  # skip Claude Code's own prompt + MCP tools = fewer tokens
         "--output-format", "stream-json", "--verbose", "--include-partial-messages"],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, encoding="utf-8"
    )
    proc.stdin.write(prompt)
    proc.stdin.close()
    timer = threading.Timer(timeout_s, proc.kill)  # hard stop after timeout_s seconds
    timer.start()
    buffer, streamed, result_event = "", "", {}
    try:
        for raw in proc.stdout:
            try:
                event = json.loads(raw)
            except Exception:
                continue
            if event.get("type") == "stream_event":  # live text as Claude writes it
                delta = event.get("event", {}).get("delta", {})
                if delta.get("type") == "text_delta":
                    buffer += delta.get("text", "")
                    streamed += delta.get("text", "")
                    *complete, buffer = buffer.split("\n")
                    for line in complete:
                        record(line)
            elif event.get("type") == "result":  # final full answer (also covers CLIs without live text)
                result_event = event
                for line in str(event.get("result", "")).split("\n"):
                    record(line)
        record(buffer)
    finally:
        timer.cancel()
        proc.wait()
    print()
    log_claude_usage(result_event, prompt, streamed)

    if not results:
        raise RuntimeError(f"Claude CLI returned no scores (error, or stopped after {timeout_s}s).")

    # Claude stopped partway: keep its scores, let Qwen score only the rest
    missing = [l for l in jobs_payload.split("\n")
               if re.match(r"ID (\d+):", l) and int(re.match(r"ID (\d+):", l).group(1)) not in results]
    if missing:
        print(f"  🔁 Switching to Qwen: Claude stopped (timeout) at {len(results)}/{total}. Qwen scores the remaining {len(missing)} job(s) of this batch.")
        results.update(evaluate_with_local_qwen(resume_summary, "\n".join(missing)))
    return results


    
# Working Local Qwen Evaluator Fallback
def evaluate_with_local_qwen(resume_summary: str, jobs_payload: str) -> Dict[int, Any]:
    qwen_label = f"Qwen ({OLLAMA_MODEL})"
    print(f"  ⚙️ Invoking local {qwen_label} to score these jobs...")
    full_profile, full_payload = resume_summary, jobs_payload  # untrimmed, for the Claude fallback
    resume_summary = resume_summary[:4000]  # keep CV + 20 JDs inside Qwen's context window
    jobs_payload = "\n".join(l[:QWEN_JD_CHARS + 150] for l in jobs_payload.split("\n"))
    prompt = f"""Compare this candidate resume profile against each job posting.

Candidate Profile:
{resume_summary}

Jobs to Evaluate:
{jobs_payload}

For each job, score 0-100 as the SUM of: responsibilities fit (0-35), skills & domain fit (0-25), seniority fit (0-20), location & eligibility (0-10), hard requirements met (0-10).
Judge from the JD text, not the title. Ignore company description, culture, benefits and perks. If JD is NOT AVAILABLE, cap the score at 70.
Rating: Perfect >= 85, Good 70-84, Average 55-69, Low < 55. rationale: max 20 words naming the JD requirement that drove the score.

Return STRICTLY a JSON array:
[
  {{"id": 1, "match_score": 78, "rating": "Good", "rationale": "Owns AI roadmap like CV; asks fintech domain the CV lacks.", "low_match_reason": null}}
]"""
    try:
        resp = qwen_scorer.invoke(prompt)
    except Exception as e:
        reason = str(e)
        n_jobs = len(re.findall(r'^ID \d+:', jobs_payload, re.M))
        # Last AI fallback: Claude CLI (lean flags), at most CLAUDE_MAX_JOBS_PER_RUN jobs per run
        if SCORER != "claude" and not _IN_CLAUDE_FALLBACK[0] and CLAUDE_FALLBACK_SENT[0] + n_jobs <= CLAUDE_MAX_JOBS_PER_RUN:
            print(f"  🔁 Switching to Claude CLI: local Qwen failed. Claude scores these {n_jobs} job(s) "
                  f"({CLAUDE_FALLBACK_SENT[0] + n_jobs}/{CLAUDE_MAX_JOBS_PER_RUN} Claude fallback jobs this run).")
            CLAUDE_FALLBACK_SENT[0] += n_jobs
            _IN_CLAUDE_FALLBACK[0] = True  # Claude's own partial-result recovery mustn't loop back here
            try:
                return call_claude_pro_evaluation(full_profile, full_payload)
            except Exception as ce:
                reason = f"Claude CLI also failed: {ce}"
            finally:
                _IN_CLAUDE_FALLBACK[0] = False
        elif SCORER != "claude" and CLAUDE_FALLBACK_SENT[0] + n_jobs > CLAUDE_MAX_JOBS_PER_RUN:
            reason += f" | Claude fallback limit of {CLAUDE_MAX_JOBS_PER_RUN} jobs reached"
        print("\n  " + "!" * 66 + f"\n  ⚠️  KEYWORD MATCH ONLY: no AI model could score these {n_jobs} jobs.\n"
              f"      Reason: {reason[:240]}\n      They get 65 (title matches your role) or 45, marked 'Title heuristic'.\n  " + "!" * 66 + "\n")
        return {}  # Node 3 gives unscored jobs 65/45 by title match, marked "Title heuristic"
    try:
        match = re.search(r"\[.*\]", resp.content, re.DOTALL)
        eval_list = json.loads(match.group(0)) if match else []
        return {item.get("id"): dict(item, scored_by=f"Qwen ({OLLAMA_MODEL})") for item in eval_list}
    except Exception as e:
        print(f"  ⚠️ Qwen parse notice: {e}")
        return {}


# ==========================================
# POSTING QUALITY: job IDs, company from title/URL, live-link + posting-age check, better JD sources
# One request per job tells us: is it still open, when was it posted, the JD text, the company.
# ==========================================
MAX_POSTING_AGE_DAYS = int(os.getenv("MAX_POSTING_AGE_DAYS", "0") or 0)  # 0 = the recency you type at start-up
NO_JD_POLICY = (os.getenv("NO_JD_POLICY") or "last").strip().lower()    # last | drop | score (old behaviour)
QUALITY_LOG_FILE = os.path.join(RESUMES_DIR, "run_quality_log.csv")     # one line per scoring pass (light eval)

_CLOSED_TEXT = re.compile(
    r"no longer accepting applications|no longer (?:available|open|active)|job (?:is )?not found|job not available"
    r"|(?:this|the) (?:job|position|role|posting|vacancy) (?:has been|is|was) (?:closed|filled|removed|expired)"
    r"|(?:posting|job|vacancy) (?:has )?expired|position (?:has been )?filled|not accepting (?:new )?applications"
    r"|we couldn.?t find (?:this|that|the) (?:job|page)|job you(?:.re| are) looking for|page (?:was )?not found", re.I)
_UUID = r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}"
_UNKNOWN_COMPANY_WORDS = {"", "unknown company", "unknown", "n a", "na", "direct ats", "confidential", "company"}
_BOARD_NAMES = {d.split(".")[0] for d in JOB_BOARD_DOMAINS + JOB_BOARDS["global"]} | {"himalayas", "lever", "ashby",
                "greenhouse", "workable", "careers", "jobs", "x", "twitter", "reddit"}
_SLUG_CASE = {"ai": "AI", "io": "IO", "hr": "HR", "ml": "ML", "ag": "AG", "gmbh": "GmbH", "uk": "UK", "us": "US"}
_POSTING_INFO: Dict[str, Dict[str, Any]] = {}   # link -> inspect_posting() result (reused by JD fetch + Excel)
_ATS_BOARD_CACHE: Dict[str, Any] = {}           # Ashby board JSON per slug, fetched once per run
STRICT_BOARD_DATES = os.getenv("STRICT_BOARD_DATES", "1").strip() != "0"   # job-board post with no date AND no check = dropped
EXTRA_BOARD_DOMAINS = ["linkedin.com", "dynamitejobs.com", "weworkremotely.com", "wellfound.com", "himalayas.app",
                       "remoteok.com", "jobs.workable.com", "jobgether.com", "remote.co", "workingnomads.com"]
_LISTING_URL = re.compile(  # search / category / company pages that list many jobs (or none)
    r"linkedin\.com/(?:company|school|showcase|in|pulse)/"
    r"|/(?:category|categories|search|search-results|tag|tags|role|roles|location|locations|collections)(?:/|\?|$)"
    r"|himalayas\.app/jobs(?:/|\?|$)|[?&](?:q|query|keywords|search|s)=", re.I)
_LISTING_TITLE = re.compile(r"^\W*[\d,.]+k?\+?\s+(?:[\w&/-]+\s+){0,6}jobs?\b|\bjobs (?:in|for|near)\b|\bjob search\b"
                            r"|\b(?:browse|search) (?:\w+ ){0,3}jobs\b|\bjobs? board\b", re.I)


def is_listing_url(url: str) -> bool:
    """True for a link to a list of jobs (search, category, company page, ATS board root), not one posting."""
    u = (url or "").strip().lower()
    if not u or canonical_job_id(u).startswith("linkedin:"):   # LinkedIn search URL with currentJobId = one job
        return False
    p = urllib.parse.urlparse(u if "://" in u else f"https://{u}")
    path = p.path.rstrip("/")
    ats, _ = _match_ats(u)
    if ats in ("Ashby", "Lever") and not re.search(_UUID, u):
        return True
    if ats == "Greenhouse" and not re.search(r"jobs/\d|gh_jid=", u):
        return True
    return bool(_LISTING_URL.search(u)) or path.strip("/") in ("", "jobs", "careers", "remote-jobs", "job-board") \
        or bool(re.search(r"(?:^|-)jobs$", path.rsplit("/", 1)[-1]))


def is_board_url(url: str) -> bool:
    netloc = urllib.parse.urlparse(url or "").netloc.lower().replace("www.", "")
    boards = set(JOB_BOARD_DOMAINS + EXTRA_BOARD_DOMAINS + [d for v in JOB_BOARDS.values() for d in v])
    return any(netloc == b or netloc.endswith("." + b) for b in boards)


def is_unknown_company(name: str) -> bool:
    n = _norm_text(name)
    return n in _UNKNOWN_COMPANY_WORDS or n in _BOARD_NAMES


def prettify_slug(slug: str) -> str:
    """'momentive-software' -> 'Momentive Software', 'tempus-ai' -> 'Tempus AI', 'upcover-2' -> 'Upcover'."""
    slug = re.sub(r"[-_]\d+$", "", urllib.parse.unquote(slug or "").strip("/ "))
    words = [w for w in re.split(r"[-_\s]+", slug) if w]
    return " ".join(_SLUG_CASE.get(w.lower(), w if any(c.isupper() for c in w) else w.capitalize()) for w in words)


def canonical_job_id(url: str) -> str:
    """Same job, any URL variant -> one ID. Tracking params, /apply, trailing slashes and LinkedIn country domains are ignored."""
    u = (url or "").strip()
    low = u.lower()
    m = re.search(r"linkedin\.com/.*?(?:currentjobid=|jobs/view/(?:[^/?]*?-)?)(\d{7,})", low)
    if m:
        return f"linkedin:{m.group(1)}"
    m = re.search(r"greenhouse\.io/.*?(?:jobs/|gh_jid=)(\d{5,})", low) or re.search(r"[?&]gh_jid=(\d{5,})", low)
    if m:
        return f"greenhouse:{m.group(1)}"
    m = re.search(r"jobs\.workable\.com/view/([A-Za-z0-9]+)", u) or re.search(r"apply\.workable\.com/[^/]+/j/([A-Za-z0-9]+)", u)
    if m:
        return f"workable:{m.group(1)}"
    m = re.search(_UUID, low)
    if m:  # Lever, Ashby and most modern ATS boards use a UUID per posting
        return f"uuid:{m.group(0)}"
    m = re.search(r"smartrecruiters\.com/[^/]+/(\d{6,})", low)
    if m:
        return f"smartrecruiters:{m.group(1)}"
    p = urllib.parse.urlparse(u if "://" in u else f"https://{u}")
    path = re.sub(r"/(apply|application)/?$", "", p.path.rstrip("/"))
    return f"{p.netloc.lower().replace('www.', '')}{path.lower()}"


def company_from_url(url: str) -> str:
    """Company name hidden in the posting URL (LinkedIn '...-at-lenovo-123', ATS slug, board /companies/<slug>/)."""
    u = urllib.parse.unquote(url or "")
    m = re.search(r"linkedin\.com/jobs/view/([^/?]+?)-(\d{7,})", u, re.I) or re.search(r"jobs\.workable\.com/view/[^/]+/([^/?]+)", u, re.I)
    if m and "-at-" in m.group(1):
        return prettify_slug(m.group(1).rsplit("-at-", 1)[1])
    ats, slug = _match_ats(u)
    if ats and ats != "Workday":
        return prettify_slug(slug[0]) if "." not in slug[0] else slug[0].capitalize()   # 'jerry.ai' -> 'Jerry.ai'
    if ats == "Workday":
        return prettify_slug(slug[0])
    m = re.search(r"/compan(?:y|ies)/([^/?]+)/jobs?\b", u, re.I)   # himalayas, antler/getro boards, wellfound, WTTJ
    if m:
        return prettify_slug(m.group(1))
    return ""


def parse_posting_title(title: str):
    """Cleans the titles LinkedIn / Exa give back. Returns (role, company or '', pattern: 'hiring' | 'at' | '').
    'Ladders hiring Senior Product Manager, AI in United States | LinkedIn' -> ('Senior Product Manager, AI', 'Ladders')
    'NAVEX hiring Senior Technical Product Manager, AI • Remote (Work from Home)' -> ('Senior Technical Product Manager, AI', 'NAVEX')"""
    t = re.sub(r"\s+", " ", (title or "")).strip()
    t = re.sub(r"\s*[|•·]\s*(linkedin|indeed|glassdoor|himalayas|wellfound|jobgether)\b.*$", "", t, flags=re.I)
    m = re.match(r"^(.{2,60}?)\s+(?:is\s+)?(?:now\s+)?hiring:?\s+(?:an?\s+|for\s+)?(.+)$", t, re.I)
    if m and _norm_text(m.group(1)) not in {"we", "we re", "we are", "now", "urgently", "actively"}:
        role = re.split(r"\s+[|•·]\s+", m.group(2))[0]
        role = re.sub(r"\s+in\s+[A-Z][^()|]*$", "", role).strip()   # LinkedIn appends ' in <location>'
        return role, m.group(1).strip(), "hiring"
    m = re.match(r"^(.+?)\s+(?:at|@)\s+([A-Z0-9][^|•·]{1,50}?)(?:\s+[|•·\-–—]\s+.*)?$", t)
    if m:
        return m.group(1).strip(), m.group(2).strip(), "at"
    return t, "", ""


def tidy_job_fields(job: Dict[str, Any]) -> Dict[str, Any]:
    """Runs on every job before scoring: clean 'X hiring Y in Z' titles and fill the company from title, then URL."""
    role, company, kind = parse_posting_title(job.get("Job Profile", ""))
    unknown = is_unknown_company(job.get("Target Company"))
    if role and (kind == "hiring" or (kind == "at" and unknown)):   # 'X at Y' is only split when the company is missing
        job["Job Profile"] = role
    if unknown:
        job["Target Company"] = company or company_from_url(job.get("Job Link", "")) or "Unknown Company"
    cid = canonical_job_id(job.get("Job Link", ""))
    if cid.startswith("linkedin:"):   # search-results?currentJobId=..., es.linkedin.com, ?trk= -> one clean public link
        job["Job Link"] = f"https://www.linkedin.com/jobs/view/{cid.split(':', 1)[1]}/"
    return job


def parse_posted_text(text) -> datetime:
    """'2026-10-20', '3 days ago', 'Reposted 2 weeks ago', '4 months ago', '30+ days ago' -> datetime (None if unknown)."""
    if isinstance(text, datetime):
        return text
    raw = str(text or "").strip()
    txt = raw.lower()
    if not txt or "past" in txt or "not listed" in txt:
        return None
    iso = _parse_iso(raw[:25]) if re.match(r"\d{4}-\d{2}-\d{2}", raw) else None
    if iso:
        return iso
    if "just now" in txt or "today" in txt or re.search(r"\d+\s*(?:minute|hour|min|hr)s?\b", txt):
        return datetime.now()
    if "yesterday" in txt:
        return datetime.now() - timedelta(days=1)
    m = re.search(r"(\d+)\+?\s*(day|week|month|year)", txt)
    if m:
        n, unit = int(m.group(1)), m.group(2)
        return datetime.now() - timedelta(days=n * {"day": 1, "week": 7, "month": 30, "year": 365}[unit])
    clean = re.sub(r"(\d)(st|nd|rd|th)\b", r"\1", re.sub(r"[,.]", " ", raw)).split()
    for fmt in ("%b %d %Y", "%B %d %Y", "%d %b %Y", "%d %B %Y", "%d/%m/%Y"):   # 'Oct 21, 2026', '21 October 2026'
        try:
            return datetime.strptime(" ".join(clean[:3]) if "/" not in raw else raw.strip()[:10], fmt)
        except ValueError:
            continue
    return None


_POSTED_NEAR = re.compile(r"(?:posted|published|listed|reposted|date posted)\s*(?:on|:)?\s*"
                          r"(\d+\+?\s*(?:minute|hour|day|week|month|year)s?\s*ago|today|yesterday"
                          r"|[a-z]{3,9}\.?\s+\d{1,2}(?:st|nd|rd|th)?,?\s+\d{4}|\d{1,2}\s+[a-z]{3,9}\.?\s+\d{4}|\d{4}-\d{2}-\d{2})", re.I)


def page_posted_date(soup, text: str):
    """Posting date when there's no JobPosting data: meta tags, then 'Posted 21 days ago' / 'Posted Oct 21, 2026' in the page."""
    for attrs in ({"itemprop": "datePosted"}, {"property": "article:published_time"}, {"name": "date"}):
        tag = soup.find(attrs=attrs)
        d = parse_posted_text(tag.get("content") or tag.get("datetime") or tag.get_text(strip=True)) if tag else None
        if d:
            return d
    m = _POSTED_NEAR.search(text or "") or re.search(r"\b(\d+\+?\s*(?:day|week|month|year)s?\s+ago)\b", text or "", re.I)
    return parse_posted_text(m.group(1)) if m else None


def _json_ld_posting(soup):
    """schema.org JobPosting embedded in the page (most ATS + job boards ship one, even JS-rendered ones)."""
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(s.string or s.get_text() or "{}")
        except Exception:
            continue
        stack = [data]
        while stack:
            node = stack.pop()
            if isinstance(node, list):
                stack.extend(node)
            elif isinstance(node, dict):
                if "JobPosting" in str(node.get("@type", "")):
                    return node
                stack.extend(v for v in node.values() if isinstance(v, (dict, list)))
    return None


def _html_to_text(html: str) -> str:
    return BeautifulSoup(html or "", "html.parser").get_text("\n", strip=True)


def _inspect_ats_api(url: str):
    """Exact open/closed answer from the ATS's public API (no scraping). None = not an API-backed ATS URL."""
    uuid = re.search(_UUID, url.lower())
    ats, slug = _match_ats(url)
    if ats == "Ashby" and uuid:
        if slug[0] not in _ATS_BOARD_CACHE:
            r = requests.get(f"https://api.ashbyhq.com/posting-api/job-board/{slug[0]}", timeout=8)
            _ATS_BOARD_CACHE[slug[0]] = r.json().get("jobs", []) if r.status_code == 200 else None
        jobs = _ATS_BOARD_CACHE[slug[0]]
        if jobs is None:
            return None
        hit = next((j for j in jobs if str(j.get("id", "")).lower() == uuid.group(0)), None)
        if not hit:
            return {"alive": False, "reason": "removed from the Ashby board"}
        return {"alive": True, "posted": _parse_iso(hit.get("publishedAt")), "company": prettify_slug(slug[0]),
                "jd": hit.get("descriptionPlain") or _html_to_text(hit.get("descriptionHtml"))}
    if ats == "Lever" and uuid:
        r = requests.get(f"https://api.lever.co/v0/postings/{slug[0]}/{uuid.group(0)}?mode=json", timeout=8)
        if r.status_code == 404:
            return {"alive": False, "reason": "Lever says the posting is gone"}
        if r.status_code != 200:
            return None
        j = r.json()
        lists = "\n".join(f"{l.get('text', '')}\n{_html_to_text(l.get('content'))}" for l in j.get("lists") or [])
        return {"alive": True, "posted": _parse_iso(j.get("createdAt")), "company": prettify_slug(slug[0]),
                "jd": f"{j.get('descriptionPlain', '')}\n{lists}"}
    gh = re.search(r"greenhouse\.io/.*?(?:jobs/|gh_jid=)(\d{5,})", url)
    if ats == "Greenhouse" and gh:
        r = requests.get(f"https://boards-api.greenhouse.io/v1/boards/{slug[0]}/jobs/{gh.group(1)}", timeout=8)
        if r.status_code == 404:
            return {"alive": False, "reason": "Greenhouse says the posting is gone"}
        if r.status_code != 200:
            return None
        j = r.json()
        import html as _html
        return {"alive": True, "posted": _parse_iso(j.get("first_published") or j.get("updated_at")),
                "company": j.get("company_name") or prettify_slug(slug[0]), "jd": _html_to_text(_html.unescape(j.get("content") or ""))}
    return None


def _parse_linkedin_html(html: str):
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(" ", strip=True)
    ld = _json_ld_posting(soup) or {}
    valid = _parse_iso(ld.get("validThrough"))
    closed = bool(soup.select_one("figure.closed-job, .closed-job")) or bool(re.search(r"no longer accepting applications", text, re.I)) \
        or bool(valid and valid < datetime.now())
    when = soup.select_one("span.posted-time-ago__text, .posted-time-ago__text")
    org = soup.select_one("a.topcard__org-name-link, .topcard__org-name-link, span.topcard__flavor a")
    box = soup.select_one("div.show-more-less-html__markup, div.description__text")
    return {"alive": not closed, "reason": "LinkedIn: no longer accepting applications" if closed else "",
            "posted": parse_posted_text(when.get_text(strip=True) if when else "") or _parse_iso(ld.get("datePosted")),
            "company": org.get_text(strip=True) if org else "",
            "jd": box.get_text("\n", strip=True) if box else _html_to_text(ld.get("description"))}


def _inspect_linkedin(job_id: str):
    """LinkedIn's guest job page (no login): JD, 'x weeks ago', company and the 'No longer accepting applications' banner.
    Rate-limited (429) -> one retry, then the public /jobs/view/ page, which carries the same banner and date."""
    guest = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
    r = requests.get(guest, headers=CAREER_HEADERS, timeout=8)
    if r.status_code == 429:
        time.sleep(3)
        r = requests.get(guest, headers=CAREER_HEADERS, timeout=8)
    if r.status_code in (404, 410):
        return {"alive": False, "reason": "LinkedIn posting removed"}
    if r.status_code == 200:
        return _parse_linkedin_html(r.text)
    pub = requests.get(f"https://www.linkedin.com/jobs/view/{job_id}/", headers=CAREER_HEADERS, timeout=8)
    if pub.status_code in (404, 410):
        return {"alive": False, "reason": "LinkedIn posting removed"}
    if pub.status_code == 200 and "authwall" not in (pub.url or "") and "login" not in (pub.url or ""):
        return _parse_linkedin_html(pub.text)
    return {"alive": None, "reason": f"LinkedIn answered {r.status_code} (rate limit?)"}


def _inspect_page(url: str):
    """Any other site: HTTP status, redirects, closed-job wording, and the page's JobPosting JSON-LD."""
    r = requests.get(url, headers=CAREER_HEADERS, timeout=8, allow_redirects=True)
    if r.status_code in (404, 410):
        return {"alive": False, "reason": f"page returns {r.status_code}"}
    if r.status_code >= 400:
        return {"alive": None, "reason": f"site answered {r.status_code}"}
    soup = BeautifulSoup(r.text, "html.parser")
    ld = _json_ld_posting(soup) or {}
    valid = _parse_iso(ld.get("validThrough"))
    if valid and valid < datetime.now():
        return {"alive": False, "reason": f"posting expired on {valid:%Y-%m-%d}"}
    old_id, new_id = canonical_job_id(url), canonical_job_id(r.url)
    if old_id != new_id and not ld and re.search(r"/(jobs|careers|search)/?$|[?&](error|notfound)", r.url, re.I):
        return {"alive": False, "reason": "redirects to the general jobs page"}
    for tag in soup(["script", "style", "nav", "header", "footer", "noscript"]):
        tag.decompose()
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    head_text = f"{title} {soup.get_text(' ', strip=True)[:3000]}"
    if not ld and _LISTING_TITLE.search(title):
        return {"alive": False, "reason": f"listing page, not one job ('{title[:50]}')"}
    if not ld and _CLOSED_TEXT.search(head_text):
        return {"alive": False, "reason": f"page says '{_CLOSED_TEXT.search(head_text).group(0)}'"}
    org = ld.get("hiringOrganization") or {}
    box = soup.select_one("[class*='job-description'], [class*='jobDescription'], [class*='description'], main, article")
    jd = _html_to_text(ld.get("description")) if ld.get("description") else (box or soup).get_text("\n", strip=True)
    return {"alive": True, "posted": _parse_iso(ld.get("datePosted")) or page_posted_date(soup, head_text), "jd": jd,
            "company": org.get("name", "") if isinstance(org, dict) else str(org)}


def inspect_posting(url: str) -> Dict[str, Any]:
    """{alive: True/False/None (None = couldn't check, kept), reason, posted, jd, company}. Cached per link for the run."""
    if url in _POSTING_INFO:
        return _POSTING_INFO[url]
    info = {"alive": None, "reason": "", "posted": None, "jd": "", "company": ""}
    try:
        cid = canonical_job_id(url)
        got = (_inspect_linkedin(cid.split(":", 1)[1]) if cid.startswith("linkedin:") else None) \
              or _inspect_ats_api(url) or _inspect_page(url)
        info.update({k: v for k, v in got.items() if v not in (None, "") or k == "alive"})
    except Exception as e:
        info["reason"] = f"couldn't open ({type(e).__name__})"
    raw = re.sub(r"[ \t]+", " ", info.get("jd") or "")
    # Feed the existing JD cache so Node 3 doesn't download the page a second time
    _JD_YEARS[url] = required_experience(raw)
    trimmed = re.sub(r"\s+", " ", extract_role_sections(raw))
    _JD_CACHE[url] = trimmed[:SCORING_JD_CHARS] if len(trimmed) > 200 else ""
    _POSTING_INFO[url] = info
    return info


def verify_postings(jobs: List[Dict[str, Any]], days: int):
    """Drops closed / dead links and postings older than the recency window. Unverifiable links are kept and labelled.
    Returns (kept jobs, {"closed or dead": n, "too old": n})."""
    max_age = MAX_POSTING_AGE_DAYS or max(1, days)
    cutoff = datetime.now() - timedelta(days=max_age)
    links = [j.get("Job Link") for j in jobs]
    with ThreadPoolExecutor(max_workers=6) as pool:   # 6, not 8: LinkedIn rate-limits guest pages
        infos = list(pool.map(inspect_posting, links))
    kept, closed, old, blind, unverified = [], [], [], [], 0
    for job, info in zip(jobs, infos):
        posted = info.get("posted") or parse_posted_text(job.get("Date Posted"))
        if info.get("company") and is_unknown_company(job.get("Target Company")):
            job["Target Company"] = info["company"]
        if info.get("alive") is False:
            closed.append(f"{job.get('Job Profile')} @ {job.get('Target Company')}: {info.get('reason')}")
            continue
        if posted and posted < cutoff:
            old.append(f"{job.get('Job Profile')} @ {job.get('Target Company')}: posted {posted:%Y-%m-%d}")
            continue
        if STRICT_BOARD_DATES and not posted and not info.get("alive") and is_board_url(job.get("Job Link")):
            blind.append(f"{job.get('Job Profile')} @ {job.get('Target Company')}: job board, no date and {info.get('reason') or 'no check'}")
            continue
        if posted:
            job["Date Posted"] = posted.strftime("%Y-%m-%d")
        job["Link Status"] = "Live" if info.get("alive") else f"Not verified ({info.get('reason') or 'unknown'})"
        unverified += 0 if info.get("alive") else 1
        job["JD Read"] = "Yes" if _JD_CACHE.get(job.get("Job Link")) else "No"
        kept.append(job)
    print(f"  🔗 Link check: {len(kept)}/{len(jobs)} kept | {len(closed)} closed or dead | "
          f"{len(old)} older than {max_age} days | {len(blind)} board post(s) with no date that couldn't be opened | "
          f"{unverified} couldn't be checked (kept, marked 'Not verified').")
    for line in (closed + old + blind)[:8]:
        print(f"     ✗ {line[:140]}")
    return kept, {"closed or dead": len(closed), "too old": len(old), "undated board post": len(blind)}


def job_dedupe_keys(job: Dict[str, Any]) -> List[str]:
    """Keys that mean 'same job': the posting ID, and company + title with 'Sr.'/'Senior', '(Remote)', punctuation ignored."""
    keys = [f"id:{canonical_job_id(job.get('Job Link', ''))}"]
    if not is_unknown_company(job.get("Target Company")):
        t = re.sub(r"\([^)]*\)", " ", (job.get("Job Profile") or "").lower())
        t = re.sub(r"\bsr\b\.?", "senior", t)
        words = sorted(set(_norm_text(t).split()) - {"remote", "hybrid", "onsite", "usa", "us", "eu", "uk", "emea", "the", "and", "of"})
        keys.append(f"ct:{_norm_text(job.get('Target Company'))}|{' '.join(words)}")
    return keys


# Same job on two sites (e.g. LinkedIn + Jobgether's Lever board): compare the JD text itself, not company + title.
INTERMEDIARY_COMPANIES = ["jobgether", "ladders", "jobot", "crossover", "turing", "lensa", "jobs via dice", "dice",
                          "talentify", "remotehunter", "hirify", "toptal", "jooble", "himalayas", "wellfound"]
_INTERMEDIARY_SENTENCE = re.compile(r"[^.?!]*\b(?:" + "|".join(map(re.escape, INTERMEDIARY_COMPANIES)) + r")\b[^.?!]*[.?!]?", re.I)
_JD_FP: Dict[str, set] = {}   # posting ID -> JD fingerprint, kept for the whole run (batch 2 vs batch 1)


def _jd_fingerprint(text: str) -> set:
    """5-word shingles of the JD, minus reposter boilerplate ('posted by Jobgether on behalf of...', 'Why Jobgether?')."""
    words = _norm_text(_INTERMEDIARY_SENTENCE.sub(" ", (text or "")[:5000])).split()
    return {" ".join(words[i:i + 5]) for i in range(max(0, len(words) - 4))}


def _title_words(title: str) -> set:
    t = re.sub(r"\([^)]*\)", " ", (title or "").lower())
    t = re.sub(r"\bsr\b\.?", "senior", t)
    return set(_norm_text(t).split()) - {"remote", "hybrid", "us", "usa", "uk", "eu", "emea", "the", "and", "of", "for"}


def same_job(fp_a: set, fp_b: set, title_a: str, title_b: str) -> bool:
    if min(len(fp_a), len(fp_b)) < 15:   # too little JD text to compare safely
        return False
    overlap = len(fp_a & fp_b) / min(len(fp_a), len(fp_b))
    ta, tb = _title_words(title_a), _title_words(title_b)
    title_sim = len(ta & tb) / max(1, len(ta | tb))
    return overlap >= 0.85 or (overlap >= 0.6 and title_sim >= 0.5)


def _board_of(url: str) -> str:
    ats, slug = _match_ats(url or "")
    return f"{ats}:{slug[0]}".lower() if ats else urllib.parse.urlparse(url or "").netloc.lower().replace("www.", "")


def distinct_postings(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    """Two posting IDs on the SAME board with different titles are two openings (a company's 2-3 similar roles),
    never copies of one job, even when they share the company's boilerplate JD text."""
    la, lb = a.get("Job Link") or "", b.get("Job Link") or ""
    if _board_of(la) != _board_of(lb) or canonical_job_id(la) == canonical_job_id(lb):
        return False
    return _title_words(a.get("Job Profile")) != _title_words(b.get("Job Profile"))


def _keep_rank(job: Dict[str, Any]):
    company = _norm_text(job.get("Target Company"))
    real_company = not is_unknown_company(company) and company not in INTERMEDIARY_COMPANIES
    return (real_company, job.get("JD Read") == "Yes", "linkedin.com" not in (job.get("Job Link") or ""))


def dedupe_across_sites(jobs: List[Dict[str, Any]], shown_keys: List[str] = None):
    """Merges copies of one job found on different sites; the copy naming the real company (then with a JD, then the
    direct ATS link) is kept, and the other links go in 'Also Posted At'. Also drops copies of jobs shown in earlier batches.
    Returns (kept jobs, number merged away)."""
    shown_fps = [_JD_FP[k[3:]] for k in (shown_keys or []) if k.startswith("id:") and k[3:] in _JD_FP]
    kept, merged = [], 0
    for job in jobs:
        cid = canonical_job_id(job.get("Job Link", ""))
        fp = _JD_FP.setdefault(cid, _jd_fingerprint((_POSTING_INFO.get(job.get("Job Link")) or {}).get("jd", "")))
        if any(same_job(fp, s, "", "") for s in shown_fps):   # identical JD to a job already shown
            merged += 1
            continue
        twin = next((k for k in kept if not distinct_postings(job, k)
                     and same_job(fp, k["_fp"], job.get("Job Profile"), k.get("Job Profile"))), None)
        if twin is None:
            kept.append({**job, "_fp": fp})
            continue
        merged += 1
        best, other = (job, twin) if _keep_rank(job) > _keep_rank(twin) else (twin, job)
        also = [l for l in [other.get("Job Link"), *(str(other.get("Also Posted At") or "").split(" | ")),
                            *(str(best.get("Also Posted At") or "").split(" | "))] if l and l != best.get("Job Link")]
        best = {**best, "_fp": twin["_fp"], "Also Posted At": " | ".join(dict.fromkeys(also))}
        if not _keep_rank(best)[0] and _keep_rank(other)[0]:
            best["Target Company"] = other.get("Target Company")   # take the real company name from the other copy
        kept[kept.index(twin)] = best
        print(f"  🔗 Same job on two sites: {best.get('Job Profile')} @ {best.get('Target Company')} → keeping "
              f"{best.get('Job Link')[:70]}")
    for k in kept:
        k.pop("_fp", None)
    return kept, merged


def no_jd_score(job: Dict[str, Any], state: PipelineState) -> Dict[str, Any]:
    """NO_JD_POLICY=last: no AI call for a job whose JD can't be read. Title score, clearly labelled, shown after JD-scored jobs."""
    ev = title_keyword_score(job, state)
    ev.update({"match_score": min(ev["match_score"], 65), "rating": "No JD",
               "rationale": "No JD could be read (login wall or JS-only page). Title match only, open the link before approving.",
               "scored_by": "No JD (title only)"})
    return ev


GATE_DROP_REASONS = ["not a single posting", "rejected before", "shown earlier this run", "duplicate", "title mismatch",
                     "closed or dead", "too old", "undated board post", "same job, other site", "no JD"]


def log_run_quality(jobs: List[Dict[str, Any]], dropped: Dict[str, int]):
    """Light eval: the 5 numbers that went wrong before, printed each run and appended to run_quality_log.csv."""
    n = max(1, len(jobs))
    row = {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "jobs_scored": len(jobs),
           "unknown_company_pct": round(100 * sum(is_unknown_company(j.get("Target Company")) for j in jobs) / n),
           "no_jd_pct": round(100 * sum(j.get("JD Read") == "No" for j in jobs) / n),
           "not_verified_pct": round(100 * sum(str(j.get("Link Status", "")).startswith("Not") for j in jobs) / n),
           **{f"dropped_{k.replace(' ', '_')}": dropped.get(k, 0) for k in GATE_DROP_REASONS}}
    print(f"  📊 Quality: {row['unknown_company_pct']}% unknown company | {row['no_jd_pct']}% no JD | "
          f"{row['not_verified_pct']}% link not verified | dropped: "
          + (", ".join(f"{v} {k}" for k, v in dropped.items() if v) or "none"))
    try:
        import csv
        new = not os.path.exists(QUALITY_LOG_FILE)
        if not new and open(QUALITY_LOG_FILE, encoding="utf-8").readline().strip() != ",".join(row):
            os.replace(QUALITY_LOG_FILE, QUALITY_LOG_FILE.replace(".csv", "_old.csv"))   # columns changed: start a new file
            new = True
        with open(QUALITY_LOG_FILE, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            if new:
                w.writeheader()
            w.writerow(row)
    except Exception:
        pass


def gate_jobs_for_scoring(jobs: List[Dict[str, Any]], state: PipelineState):
    """Drops rejected, already-shown, duplicate and wrong-title jobs before any scoring.
    Returns (kept jobs, {drop reason: count})."""
    rejected_links, rejected_keys = load_rejected_jobs()
    rejected_ids = {f"id:{canonical_job_id(l)}" for l in rejected_links if l}
    shown = set(state.get("shown_job_keys") or [])   # every job already shown in an earlier batch of this run
    variants = expand_role_variants_cached(state.get("roles", []))
    kept, seen = [], set()
    dropped = {k: 0 for k in GATE_DROP_REASONS}
    for j in jobs:
        tidy_job_fields(j)   # company from title / URL, 'X hiring Y in Z' -> 'Y'
        keys, old_key = set(job_dedupe_keys(j)), _job_key(j)
        if is_listing_url(j.get("Job Link")):
            dropped["not a single posting"] += 1
        elif j.get("Job Link") in rejected_links or keys & rejected_ids or (old_key and old_key in rejected_keys):
            dropped["rejected before"] += 1
        elif keys & shown:
            dropped["shown earlier this run"] += 1
        elif keys & seen:
            dropped["duplicate"] += 1
        elif not title_matches_roles(j.get("Job Profile", ""), variants):
            dropped["title mismatch"] += 1
        else:
            kept.append(j)
            seen.update(keys)
    reasons = ", ".join(f"{v} {k}" for k, v in dropped.items() if v) or "none"
    print(f"  🚦 Gate: {len(kept)}/{len(jobs)} jobs go on to the link check (dropped: {reasons}).")
    return kept, dropped

# Node 3: Deep Match Scoring
def evaluate_matches_node(state: PipelineState) -> Dict[str, Any]:
    print(f"\n⚖️ Node 3: Deep Match Scoring via {SCORER_LABEL} (Qwen fallback)...")
    raw_jobs, dropped = gate_jobs_for_scoring(state.get("raw_jobs", []), state)  # rejected / shown / duplicate / title out
    raw_jobs, link_drops = verify_postings(raw_jobs, state.get("job_duration_days", 7))  # closed + too-old links out
    dropped.update(link_drops)
    raw_jobs, dropped["same job, other site"] = dedupe_across_sites(raw_jobs, state.get("shown_job_keys"))  # JD-text match
    if not raw_jobs:
        return {"evaluated_jobs": [], "low_match_summary": "No jobs left to score after filtering."}

    mode = state.get("scoring_mode") or ask_scoring_mode(len(raw_jobs))
    eval_map, to_score, jd_map, profile, cache = {}, [], {}, "", {}
    BATCH_SIZE = 10

    if mode == "title":
        eval_map = {i + 1: title_keyword_score(j, state) for i, j in enumerate(raw_jobs)}
    else:
        cache = load_score_cache()
        new_jobs = [j for j in raw_jobs if score_cache_key(j, state) not in cache]
        jd_map = attach_job_descriptions(new_jobs) if new_jobs else {}  # real JD text per job
        # Seniority: drop jobs whose JD asks for years far from yours (no requirement stated = keep)
        before = len(raw_jobs)
        raw_jobs = [j for j in raw_jobs if experience_fits(j.get("Job Link"), state.get("candidate_years"))]
        if before != len(raw_jobs):
            print(f"  📏 Experience filter: dropped {before - len(raw_jobs)} job(s) whose JD asks for years far from your {state.get('candidate_years'):g}.")
        if NO_JD_POLICY == "drop":   # no JD = not scored and not shown
            before = len(raw_jobs)
            raw_jobs = [j for j in raw_jobs if j.get("JD Read") != "No" or score_cache_key(j, state) in cache]
            dropped["no JD"] = before - len(raw_jobs)
            if dropped["no JD"]:
                print(f"  🚫 NO_JD_POLICY=drop: skipped {dropped['no JD']} job(s) whose JD couldn't be read.")
        for i, j in enumerate(raw_jobs):
            if score_cache_key(j, state) in cache:
                eval_map[i + 1] = cache[score_cache_key(j, state)]
        if NO_JD_POLICY == "last":   # no AI call without a JD: labelled title score, shown after JD-scored jobs
            no_jd = [i for i, j in enumerate(raw_jobs) if i + 1 not in eval_map and j.get("JD Read") == "No"]
            for i in no_jd:
                eval_map[i + 1] = no_jd_score(raw_jobs[i], state)
            if no_jd:
                print(f"  📭 {len(no_jd)} job(s) have no readable JD: not sent to the AI, shown last and marked 'No JD'.")
        to_score = [i for i in range(len(raw_jobs)) if i + 1 not in eval_map]
        to_score.sort(key=lambda i: -title_keyword_score(raw_jobs[i], state)["match_score"])  # best title matches get Claude first
        print(f"  ♻️ {sum(1 for j in raw_jobs if score_cache_key(j, state) in cache)} job(s) reused from earlier runs; {len(to_score)} to score now.")
        if to_score:
            profile = build_scoring_profile(state)  # target roles + locations + summary + full CV

    scored = []   # (positions, batch, batch_map) per scored batch
    use_queue = SHARED_QUEUE and SCORER == "gemini" and bool(to_score)
    if use_queue:
        scored = score_with_shared_queue([to_score[i:i + BATCH_SIZE] for i in range(0, len(to_score), BATCH_SIZE)],
                                         raw_jobs, jd_map, profile)
    start, cap_noted = 0, False
    while start < len(to_score) and not use_queue:
        primary_cap = CLAUDE_MAX_JOBS_PER_RUN if SCORER == "claude" else 10 ** 9  # Gemini has no job limit
        claude_left = primary_cap - CLAUDE_JOBS_SENT[0]
        primary_on = SCORER != "qwen" and claude_left > 0 and not PRIMARY_DISABLED[0]
        positions = to_score[start:start + (min(BATCH_SIZE, claude_left) if primary_on else BATCH_SIZE)]
        start += len(positions)
        batch = [raw_jobs[p] for p in positions]
        jobs_payload = build_jobs_payload(batch, jd_map)
        print(f"  📦 Scoring jobs {start - len(batch) + 1}-{start} of {len(to_score)}...")

        if not primary_on:
            if not cap_noted:
                reason = ("JOB_SCORER is set to qwen" if SCORER == "qwen" else
                          f"{SCORER_LABEL} unavailable ({PRIMARY_DISABLED[1]})" if PRIMARY_DISABLED[0] else
                          f"{SCORER_LABEL} limit of {CLAUDE_MAX_JOBS_PER_RUN} jobs per run reached")
                print(f"  🔁 Switching to Qwen: {reason}. Qwen scores the remaining {len(to_score) - start + len(batch)} job(s).")
                cap_noted = True
            batch_map = evaluate_with_local_qwen(profile, jobs_payload)
        else:
            CLAUDE_JOBS_SENT[0] += len(batch)
            # 1. Primary: Gemini (or Claude, if JOB_SCORER=claude)
            try:
                batch_map = call_primary_scorer(profile, jobs_payload)
                print(f"  ✓ Batch scored via {SCORER_LABEL}!")
                maybe_calibrate(profile, jobs_payload, batch_map)  # one time: line Qwen's scale up with the primary's
            except Exception as e:
                print(f"  ⚠️ {SCORER_LABEL} notice: {e}")
                print(f"  🔁 Switching to Qwen: {SCORER_LABEL} call failed. Qwen scores this batch of {len(batch)} job(s).")
                # 2. Working Fallback: Local Qwen 2.5 (this batch only)
                batch_map = evaluate_with_local_qwen(profile, jobs_payload)
        scored.append((positions, batch, batch_map))

    for positions, batch, batch_map in scored:
        batch_map = apply_calibration(batch_map)  # Qwen scores -> primary scorer's scale
        # Batch IDs run 1-10; map them back to positions in the full job list and cache them
        for local_id, ev in batch_map.items():
            try:
                eval_map[positions[int(local_id) - 1] + 1] = ev
                cache[score_cache_key(batch[int(local_id) - 1], state)] = ev
            except (TypeError, ValueError, IndexError):
                pass
    if to_score:
        save_score_cache(cache)
        print(f"  🤖 {SCORER_LABEL} used for {CLAUDE_JOBS_SENT[0]}" + (f"/{CLAUDE_MAX_JOBS_PER_RUN}" if SCORER == "claude" else "")
              + " jobs so far this run.")
    print_claude_usage_total()
    evaluated = []
    low_matches = []

    for i, job in enumerate(raw_jobs):
        ev = eval_map.get(i + 1, {})
        score = ev.get("match_score")
        
        # Safety heuristic if an ID was missed
        if score is None:
            score = 65 if title_matches_roles(job.get("Job Profile", ""), state.get("roles", [])) else 45
            ev = {"rationale": "Not scored by a model (title match only), check manually.", "scored_by": "Title heuristic"}

        job["Match Score"] = score
        job["Rating"] = ev.get("rating", "Good" if score >= 70 else "Average")
        job["Rationale"] = ev.get("rationale", "Aligned with core product leadership capabilities.")
        job["low_match_reason"] = ev.get("low_match_reason")
        job["Score Breakdown"] = ev.get("breakdown", "")
        job["Scored By"] = ev.get("scored_by", "Earlier run (cache)")

        if job["Match Score"] >= 60:
            evaluated.append(job)
        else:
            low_matches.append(job)

    keyword_only = [j for j in raw_jobs if j.get("Scored By") == "Title heuristic"]
    if keyword_only:
        print("\n  " + "=" * 66 + f"\n  ⚠️  {len(keyword_only)} of {len(raw_jobs)} jobs were scored by KEYWORD MATCH ONLY (no AI).\n"
              "      Check them by hand. Excel 'Scored By' column = 'Title heuristic'.\n  " + "=" * 66)
    evaluated.sort(key=lambda x: (x.get("JD Read") != "No", x.get("Match Score", 0)), reverse=True)  # JD-scored first

    low_match_items = []
    for lm in low_matches[:4]:
        reason = lm.get("low_match_reason") or lm.get("Rationale")
        low_match_items.append(f"• {lm.get('Job Profile')} @ {lm.get('Target Company')} ({lm.get('Match Score')}%): {reason[:120]}")
    low_match_summary = "\n".join(low_match_items) if low_match_items else "No low matches."

    log_run_quality(raw_jobs, dropped)
    print(f"  ✓ {len(evaluated)} qualified positions approved for HITL sign-off.")
    return {"evaluated_jobs": evaluated, "low_match_summary": low_match_summary, "scoring_mode": mode}


# Node 4: Batch Human-in-the-Loop (HITL) Sign-Off
def human_review_node(state: PipelineState) -> Dict[str, Any]:
    evaluated = state.get("evaluated_jobs", [])
    already_approved = state.get("approved_jobs", [])
    target = state.get("target_approved_count", 5)

    if not evaluated:
        print("\n⚠️ No new candidate jobs to review in this cycle.")
        return {"approved_jobs": already_approved, "user_decision": "proceed"}

    print("\n" + "="*68)
    print(f"🧑‍💼 HUMAN-IN-THE-LOOP SIGN-OFF (Approved so far: {len(already_approved)}/{target})")
    print("="*68)

    batch = evaluated[:5]
    remaining = evaluated[5:]

    for idx, job in enumerate(batch, 1):
        fresh = job.get("Freshness") or posting_freshness(job.get("Date Posted"))
        extra = " | ".join(x for x in [job.get("Company Stage"), job.get("Hiring Momentum")] if x)
        print(f"\n[{idx}] {(fresh + '  ') if fresh else ''}{job.get('Job Profile')} @ {job.get('Target Company')}"
              + (f"  ({extra})" if extra else ""))
        print(f"    Location: {job.get('Location')} | Match Score: {job.get('Match Score')}% ({job.get('Rating')}) "
              f"| Posted: {job.get('Date Posted', 'Not listed')} | Link: {job.get('Link Status', '?')}")
        print(f"    Rationale: {job.get('Rationale')}")
        print(f"    Link: {job.get('Job Link')}")

    print("\n" + "-"*68)
    user_input = input(
        f"Enter IDs to APPROVE (e.g., '1, 3'), 'all', 'none', or 'done' to stop: "
    ).strip().lower()

    newly_approved = []
    if user_input == "all":
        newly_approved.extend(batch)
    elif user_input not in ["none", "done", ""]:
        selected_indices = [int(i.strip()) for i in re.findall(r"\d+", user_input)]
        for i in selected_indices:
            if 1 <= i <= len(batch):
                newly_approved.append(batch[i - 1])

    if user_input not in ("done", ""):  # shown but not approved = rejected, never shown again
        save_rejected_jobs([j for j in batch if j not in newly_approved])

    total_approved = already_approved + newly_approved
    print(f"✓ Total Approved: {len(total_approved)}/{target}")
    shown_keys = (state.get("shown_job_keys") or []) + [k for j in batch for k in job_dedupe_keys(j)]  # never shown twice

    if user_input == "done" or len(total_approved) >= target:
        decision = "proceed"
    elif remaining:
        decision = "review_more"
    else:
        fetch_more = input("Reviewed all jobs in this batch. Fetch broader live search? (y/n): ").strip().lower()
        decision = "search_more" if fetch_more == "y" else "proceed"

    return {
        "approved_jobs": total_approved,
        "evaluated_jobs": remaining,
        "user_decision": decision,
        "shown_job_keys": shown_keys
    }


# Node 5: Real Contact Discovery (Exa SERP)





# ==========================================
# 2. UPDATED NODE 5 WITH GETEMAIL.IO ENRICHMENT
# ==========================================
import time

def verify_email_via_getemail(fullname: str, company: str, max_wait_seconds: int = 8) -> str:
    """Verifies live corporate email via GetEmail.io with automatic polling."""
    api_key = os.environ.get("GETEMAIL_API_KEY", "").strip()
    if not api_key or api_key == "your_actual_getemail_api_key_here":
        return None

    clean_domain = re.sub(r'[^a-zA-Z0-9]', '', company).lower()
    find_url = "https://api.getemail.io/v2/find-email"
    params = {
        "api_key": api_key,
        "fullname": fullname,
        "domain": clean_domain,
        "isEnrichToDo": "true"
    }

    try:
        resp = requests.get(find_url, params=params, timeout=12)
        if resp.status_code != 200:
            return None
            
        data = resp.json()
        if data.get("email"):
            return data.get("email")

        # Async polling for live SMTP handshake completion
        search_id = data.get("id")
        if data.get("code") == "EMAIL_FINDING_PROCESSING" and search_id:
            status_url = "https://api.getemail.io/v2/status-find-email"
            status_params = {"api_key": api_key, "id": search_id}
            
            start_time = time.time()
            while time.time() - start_time < max_wait_seconds:
                time.sleep(2.5)
                status_resp = requests.get(status_url, params=status_params, timeout=10)
                if status_resp.status_code == 200:
                    status_data = status_resp.json()
                    if status_data.get("email"):
                        return status_data.get("email")
                    if status_data.get("code") != "EMAIL_FINDING_PROCESSING":
                        break
    except Exception:
        pass

    return None


def find_contacts_node(state: PipelineState) -> Dict[str, Any]:
    print("\n🤝 Node 5: Enriching Approved Jobs with Real Recruiter Contacts & Verified Emails...")
    approved = state.get("approved_jobs", [])

    for job in approved:
        company = job.get("Target Company", "").strip()
        if not company or company in ["N/A", "Unknown Company", "Direct ATS"]:
            job["Prospect_Employer_Name"] = "Recruiting Team"
            job["Prospect_Employer_Title"] = "Talent Acquisition"
            job["Prospect_Employer_Profile_LI"] = "Not Found"
            job["Prospect_Employer_Profile_mail"] = "N/A"
            continue

        clean_comp = re.sub(r'(?i)\b(inc|ltd|pvt|technologies|corporation|llc|corp|co)\b', '', company).strip()
        clean_comp = re.sub(r'[^a-zA-Z0-9\s]', '', clean_comp).strip()
        domain_slug = re.sub(r'[^a-zA-Z0-9]', '', company).lower()

        # Step 1: Find real recruiter profile via Exa
        query = f'site:linkedin.com/in "{clean_comp or company}" ("Talent Acquisition" OR "Technical Recruiter" OR "Head of Product")'
        
        name = "Recruiting Team"
        headline = "Talent Acquisition"
        li_url = "Not Found"

        try:
            res = exa.search(
                query=query,
                type="auto",
                include_domains=["linkedin.com"],
                num_results=2
            )
            for r in res.results:
                if "/in/" in r.url:
                    li_url = r.url
                    raw_title = r.title.replace(" | LinkedIn", "").replace(" - LinkedIn", "")
                    parts = raw_title.split(" - ") if " - " in raw_title else raw_title.split(" | ")
                    name = parts[0].strip()
                    headline = parts[1].strip() if len(parts) > 1 else "Talent Acquisition"
                    break
        except Exception as e:
            print(f"  ⚠️ Recruiter search issue for {company}: {e}")

        # Step 2: Live SMTP Email Verification via GetEmail.io
        verified_email = None
        if name != "Recruiting Team":
            print(f"  📧 Verifying inbox for '{name}' at '{company}'...")
            verified_email = verify_email_via_getemail(fullname=name, company=clean_comp or company)

        # Step 3: Fallback formula if GetEmail has no match or credits are exhausted
        if not verified_email:
            first_clean = re.sub(r'[^a-zA-Z]', '', name.split()[0]).lower()
            if first_clean and len(first_clean) >= 3 and name != "Recruiting Team":
                verified_email = f"{first_clean}@{domain_slug}.com"
            else:
                verified_email = f"hiring@{domain_slug}.com"

        job["Prospect_Employer_Name"] = name
        job["Prospect_Employer_Title"] = headline
        job["Prospect_Employer_Profile_LI"] = li_url
        job["Prospect_Employer_Profile_mail"] = verified_email
        print(f"  ✓ {company}: {name} ({verified_email})")

    return {"approved_jobs": approved}

# Node 6: Deduplicating Excel Exporter
def export_excel_node(state: PipelineState) -> Dict[str, Any]:
    print("\n📊 Node 6: Appending Approved Jobs to Excel Database...")
    approved = state.get("approved_jobs", [])
    if not approved:
        print("  ⚠️ No approved jobs to export.")
        return {}

    new_rows = []
    for j in approved:
        new_rows.append({
            "Resume Applied With": state.get("candidate_name", "Default_CV"),
            "Job Profile": j.get("Job Profile", "N/A"),
            "Target Company": j.get("Target Company", "N/A"),
            "Location": j.get("Location", "Remote"),
            "Match Score": f"{j.get('Match Score', 'N/A')}%",
            "Rating": j.get("Rating", "N/A"),
            "Rationale": j.get("Rationale", "N/A"),
            "Score Breakdown": j.get("Score Breakdown", ""),
            "Scored By": j.get("Scored By", ""),
            "Job Link": j.get("Job Link", "N/A"),
            "Date Posted": j.get("Date Posted", state.get("duration_label", "Recent")),
            "Platform Name": j.get("Platform Name", "LinkedIn"),
            "Freshness": j.get("Freshness") or posting_freshness(j.get("Date Posted")),
            "Company Stage": j.get("Company Stage", ""),
            "Prospect_Employer_Name": j.get("Prospect_Employer_Name", "N/A"),
            "Prospect_Employer_Title": j.get("Prospect_Employer_Title", "N/A"),
            "Prospect_Employer_Profile_LI": j.get("Prospect_Employer_Profile_LI", "N/A"),
            "Prospect_Employer_Profile_mail": j.get("Prospect_Employer_Profile_mail", "N/A")
        })

    new_df = pd.DataFrame(new_rows)

    if os.path.exists(EXCEL_PATH):
        try:
            existing_df = pd.read_excel(EXCEL_PATH, sheet_name="Matched Jobs")
            final_df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(
                subset=["Job Link"], keep="first"
            )
        except Exception:
            final_df = new_df
    else:
        final_df = new_df

    final_df.to_excel(EXCEL_PATH, index=False, sheet_name="Matched Jobs")
    print(f"  ✓ Successfully saved {len(final_df)} total entries in {EXCEL_PATH}")
    return {}

# ==========================================
# 5b. APPLICATION TRACKER (Make → Neon → this script) + OUTREACH EMAIL AGENT
# Make copies job-related Gmail messages into Neon (table job_emails).
# This script classifies each new email's stage with local Qwen, updates the
# applications table, notifies you on Telegram/Slack, and rebuilds the
# "Applications" + "Outreach Drafts" sheets in the same Excel file.
# ==========================================
NEON_DATABASE_URL = os.getenv("NEON_DATABASE_URL", "")
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "")
SLACK_WEBHOOK_URL = os.getenv("SLACK_WEBHOOK_URL", "")

# Your past outreach emails, separated by a line containing only ---
STYLE_EXAMPLES_FILE = r"C:\Users\simar\Desktop\CV_2026\email_style_examples.txt"

APPLICATION_STAGES = ["Shortlisted", "Applied", "Application received", "HR response", "Interview stage", "Offer stage"]
TERMINAL_STAGES = ["Rejected"]

TRACKER_SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS job_emails (
    id               BIGSERIAL PRIMARY KEY,
    gmail_message_id TEXT UNIQUE,
    thread_id        TEXT,
    from_email       TEXT,
    from_name        TEXT,
    subject          TEXT,
    body_text        TEXT,
    received_at      TIMESTAMPTZ,
    processed        BOOLEAN NOT NULL DEFAULT FALSE,
    stage_detected   TEXT,
    application_id   BIGINT,
    inserted_at      TIMESTAMPTZ NOT NULL DEFAULT now()
);
CREATE TABLE IF NOT EXISTS applications (
    id                 BIGSERIAL PRIMARY KEY,
    company            TEXT NOT NULL,
    role               TEXT,
    job_link           TEXT UNIQUE,
    location           TEXT,
    platform           TEXT,
    match_score        TEXT,
    resume_used        TEXT,
    stage              TEXT NOT NULL DEFAULT 'Shortlisted',
    stage_updated_at   TIMESTAMPTZ NOT NULL DEFAULT now(),
    last_email_subject TEXT,
    recruiter_name     TEXT,
    recruiter_email    TEXT,
    created_at         TIMESTAMPTZ NOT NULL DEFAULT now()
);
CREATE TABLE IF NOT EXISTS stage_history (
    id             BIGSERIAL PRIMARY KEY,
    application_id BIGINT REFERENCES applications(id),
    from_stage     TEXT,
    to_stage       TEXT,
    source         TEXT,
    changed_at     TIMESTAMPTZ NOT NULL DEFAULT now()
);
CREATE TABLE IF NOT EXISTS outreach_drafts (
    id              BIGSERIAL PRIMARY KEY,
    job_link        TEXT UNIQUE,
    company         TEXT,
    role            TEXT,
    recipient_name  TEXT,
    recipient_email TEXT,
    subject         TEXT,
    body            TEXT,
    word_count      INT,
    created_at      TIMESTAMPTZ NOT NULL DEFAULT now()
);
ALTER TABLE outreach_drafts ADD COLUMN IF NOT EXISTS linkedin_note TEXT;
"""


def neon_connect():
    import psycopg  # pip install "psycopg[binary]"
    return psycopg.connect(NEON_DATABASE_URL, autocommit=True)


def notify(text: str):
    """Sends a message to Telegram and/or Slack, whichever is configured in .env."""
    try:
        if TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID:
            requests.post(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage",
                          json={"chat_id": TELEGRAM_CHAT_ID, "text": text}, timeout=10)
        if SLACK_WEBHOOK_URL:
            requests.post(SLACK_WEBHOOK_URL, json={"text": text}, timeout=10)
    except Exception as e:
        print(f"  ⚠️ Notification issue: {e}")


def _norm_company(name: str) -> str:
    name = re.sub(r'(?i)\b(inc|ltd|pvt|private|limited|technologies|technology|corporation|llc|corp|co|gmbh|bv|sa|labs?)\b', '', name or "")
    return re.sub(r'[^a-z0-9]', '', name.lower())


# ---------- Job-mail gate: keep promo / social / spam / informational mail out (no LLM call) ----------
SKIP_EMAIL_STAGES = ["Promotional", "Not job related"]   # stored in Neon but hidden from the Email Log
JOB_ALERT_STAGE = "Job alert"                             # kept in the Email Log, never creates an application

_JOB_SIGNAL = re.compile(r"(?i)\b(your application|application (?:for|to|received|status|update)|thank(?:s| you) for (?:applying|your (?:interest|application))"
                         r"|interview|assessment|coding (?:test|challenge)|next (?:steps|round)|shortlisted|offer letter|job offer|we regret"
                         r"|unfortunately|not (?:be )?moving forward|position has been filled|recruit(?:er|ing|ment)|talent acquisition"
                         r"|hiring (?:manager|team)|your candidacy|candidate|opportunity at|role at|position at|reached out about)\b")
_JOB_ALERT_SENDER = re.compile(r"(?i)(jobalerts?-noreply@linkedin|jobs-noreply@linkedin|alert@indeed|noreply@indeed|naukri|glassdoor|monster|"
                               r"wellfound|instahyre|foundit|iimjobs|hirist|jobs?alert)")
_JOB_ALERT_SUBJECT = re.compile(r"(?i)(job alert|jobs? (?:for|matching|you may|recommended)|new jobs?|is hiring|are hiring|jobs? near|"
                                r"recommended jobs?|apply now|\d+\+? (?:new )?(?:jobs|openings|roles))")
_PROMO_SENDER = re.compile(r"(?i)^(newsletter|news|marketing|promo(?:tions?)?|offers?|deals?|digest|updates?|events?|webinars?|community|"
                           r"info|hello|team|notifications?-noreply|invitations?|messages-noreply-digest|groups?-noreply)[@._-]|"
                           r"@(?:e\.|em\.|email\.|mail\.|news\.|info\.|marketing\.|mailer\.|mailchimp|sendgrid|hubspot|substack|medium\.com)")
_SOCIAL_SENDER = re.compile(r"(?i)@(?:facebookmail|instagram|twitter|x\.com|quora|pinterest|reddit|youtube|discord|whatsapp|snapchat|tiktok)|"
                            r"(?:invitations|notifications-noreply|updates-noreply|groups-noreply)@linkedin")
_PROMO_SUBJECT = re.compile(r"(?i)\b(\d+\s?% off|sale|discount|coupon|deal|offer ends|limited time|last chance|free trial|upgrade|premium|"
                            r"webinar|workshop|masterclass|course|certification|bootcamp|register now|save your seat|newsletter|digest|"
                            r"weekly|monthly|top stories|trending|this week in|black friday|cyber monday|cashback|reward|subscription|"
                            r"invoice|receipt|order|delivery|otp|verification code|password|security alert|sign-?in|viewed your profile|"
                            r"endorsed|congratulate|birthday|work anniversary|new follower|liked your|commented on|mentioned you|connection request)\b")
_BULK_FOOTER = re.compile(r"(?i)(unsubscribe|manage (?:your )?(?:email )?preferences|view (?:this email )?in (?:your )?browser|"
                          r"you (?:are )?receiving this (?:email|because)|email preferences|opt[- ]out)")


def prefilter_job_email(subject: str, body: str, from_email: str, from_name: str = "") -> str:
    """Cheap rules before any LLM. Returns 'llm' (looks like a real job mail), JOB_ALERT_STAGE, or 'Promotional'."""
    subj, sender, head = subject or "", (from_email or "").lower(), (body or "")[:2500]
    if _JOB_SIGNAL.search(subj):
        return "llm"                                    # application / interview / recruiter words in the subject win
    if _JOB_ALERT_SENDER.search(sender) or _JOB_ALERT_SUBJECT.search(subj):
        return JOB_ALERT_STAGE
    if _SOCIAL_SENDER.search(sender) or _PROMO_SENDER.search(sender) or _PROMO_SUBJECT.search(subj):
        return "Promotional"
    if _BULK_FOOTER.search(body or "") and not _JOB_SIGNAL.search(head):
        return "Promotional"                            # mass mail with no job wording in the first part of the body
    return "llm"


def classify_job_email(subject: str, body: str, from_email: str) -> Dict[str, Any]:
    """Local Qwen reads one email and returns {is_job_related, company, role, stage}."""
    prompt = f"""You track a candidate's job applications. Read this email and classify it.

From: {from_email}
Subject: {subject}
Body:
{(body or '')[:3000]}

Stages (pick exactly one):
- "Applied": the candidate's own application was submitted (e.g. a submission copy)
- "Application received": automated acknowledgement / "thank you for applying"
- "HR response": a recruiter or HR person personally replied, asked questions or asked to schedule a first call
- "Interview stage": interview scheduled, interview invite, assessment or next round
- "Offer stage": offer, compensation discussion, offer letter, references before offer
- "Rejected": not moving forward / position filled
- "Job alert": automated list of job openings / "jobs you may like" from a job board, no application involved
- "Promotional": newsletters, marketing, courses, webinars, events, social-network notifications, company news,
  receipts, OTPs, spam, or any other mail that is not about a specific job or the candidate's application

Only pick an application stage when the email is clearly about a job the candidate applied to or a recruiter
contacting the candidate about a specific role. When in doubt, pick "Promotional".

Return STRICTLY one JSON object:
{{"stage": "<one of the stages above>", "company": "<hiring company name or empty>", "role": "<job title or empty>"}}"""
    try:
        match = re.search(r"\{.*\}", ollama_llm.invoke(prompt).content, re.DOTALL)
        data = json.loads(match.group(0)) if match else {}
    except Exception as e:
        print(f"  ⚠️ Qwen email classification notice: {e}")
        data = {}
    stage = data.get("stage", "Promotional")
    if stage not in APPLICATION_STAGES + TERMINAL_STAGES + [JOB_ALERT_STAGE]:
        stage = "Promotional"
    return {"stage": stage, "company": (data.get("company") or "").strip(), "role": (data.get("role") or "").strip()}


def _find_application(cur, company: str, role: str, from_email: str):
    """Matches an email to an existing application by company name or sender domain."""
    cur.execute("SELECT id, company, role, stage FROM applications")
    rows = [r for r in cur.fetchall() if _norm_company(r[1])]
    target = _norm_company(company)
    domain = _norm_company((from_email or "").split("@")[-1].split(".")[0]) if "@" in (from_email or "") else ""
    if domain in ("gmail", "outlook", "yahoo", "greenhouse", "lever", "ashbyhq", "workday", "myworkday", "myworkdayjobs",
                  "smartrecruiters", "workable", "linkedin", "noreply", "mail"):
        domain = ""  # ATS / mail-provider senders say nothing about the company

    def similar(a: str, b: str) -> bool:  # exact, or one contains the other when the shorter is >= 4 chars
        return bool(a and b) and (a == b or (min(len(a), len(b)) >= 4 and (a in b or b in a)))

    candidates = [r for r in rows if _norm_company(r[1]) == target] if target else []
    candidates = candidates or [r for r in rows if similar(_norm_company(r[1]), target) or similar(_norm_company(r[1]), domain)]
    if len(candidates) > 1 and role:  # same company, several roles: prefer the matching title
        by_role = [r for r in candidates if r[2] and title_matches_roles(r[2], [role])]
        candidates = by_role or candidates
    return candidates[0] if candidates else None


def _stage_moves_forward(old: str, new: str) -> bool:
    if new in TERMINAL_STAGES:
        return old not in TERMINAL_STAGES
    if old in TERMINAL_STAGES:
        return False
    return APPLICATION_STAGES.index(new) > APPLICATION_STAGES.index(old) if old in APPLICATION_STAGES else True


def process_new_job_emails(cur) -> int:
    cur.execute("SELECT id, from_email, from_name, subject, body_text FROM job_emails WHERE processed = FALSE ORDER BY received_at NULLS LAST, id")
    emails = cur.fetchall()
    if emails:
        print(f"  📬 Checking {len(emails)} new email(s): rules drop promo/social mail, Qwen classifies the rest...")
    changes, skipped, alerts = 0, 0, 0
    for email_id, from_email, from_name, subject, body in emails:
        gate = prefilter_job_email(subject or "", body or "", from_email or "", from_name or "")
        info = classify_job_email(subject or "", body or "", from_email or "") if gate == "llm" \
            else {"stage": gate, "company": "", "role": ""}
        skipped += info["stage"] in SKIP_EMAIL_STAGES
        alerts += info["stage"] == JOB_ALERT_STAGE
        app_id = None
        if info["stage"] in APPLICATION_STAGES + TERMINAL_STAGES:
            app = _find_application(cur, info["company"], info["role"], from_email)
            if app:
                app_id, company, role, old_stage = app
            else:  # application you made outside this script: start tracking it from the email
                company = info["company"] or (from_name or from_email or "Unknown Company")
                role, old_stage = info["role"], None
                cur.execute("INSERT INTO applications (company, role, stage, platform, last_email_subject) VALUES (%s, %s, %s, 'Email', %s) RETURNING id",
                            (company, role, info["stage"], subject))
                app_id = cur.fetchone()[0]
                cur.execute("INSERT INTO stage_history (application_id, from_stage, to_stage, source) VALUES (%s, NULL, %s, %s)",
                            (app_id, info["stage"], f"email: {subject}"))
                notify(f"🆕 Tracking new application: {company} – {role or 'role n/a'} → {info['stage']}\n✉️ {subject}")
                changes += 1
            if old_stage and _stage_moves_forward(old_stage, info["stage"]):
                cur.execute("UPDATE applications SET stage = %s, stage_updated_at = now(), last_email_subject = %s WHERE id = %s",
                            (info["stage"], subject, app_id))
                cur.execute("INSERT INTO stage_history (application_id, from_stage, to_stage, source) VALUES (%s, %s, %s, %s)",
                            (app_id, old_stage, info["stage"], f"email: {subject}"))
                notify(f"📈 {company} – {role or 'role n/a'}: {old_stage} → {info['stage']}\n✉️ {subject}")
                changes += 1
        cur.execute("UPDATE job_emails SET processed = TRUE, stage_detected = %s, application_id = %s WHERE id = %s",
                    (info["stage"], app_id, email_id))
    if emails:
        print(f"  🧹 Kept out {skipped} promo/social/other email(s); {alerts} job alert(s); "
              f"{len(emails) - skipped - alerts} application email(s) checked.")
    return changes


def _write_extra_sheets(sheets: Dict[str, pd.DataFrame]):
    """Replaces only the given sheets; 'Matched Jobs' and other sheets are left untouched."""
    try:
        if os.path.exists(EXCEL_PATH):
            with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
                for name, df in sheets.items():
                    df.to_excel(writer, sheet_name=name, index=False)
        else:
            with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
                for name, df in sheets.items():
                    df.to_excel(writer, sheet_name=name, index=False)
    except PermissionError:
        print(f"  ⚠️ Close {os.path.basename(EXCEL_PATH)} in Excel and run again to refresh the tracker sheets.")


def sync_application_tracker(approved_jobs: List[Dict[str, Any]] = None, drafts: List[Dict[str, Any]] = None,
                             candidate_name: str = "") -> None:
    """Upserts approved jobs + drafts, processes new emails, rebuilds the Applications / Outreach Drafts sheets."""
    if not NEON_DATABASE_URL:
        print("  ⚠️ NEON_DATABASE_URL not set in .env; skipping application tracker.")
        return
    try:
        with neon_connect() as conn, conn.cursor() as cur:
            cur.execute(TRACKER_SCHEMA_SQL)

            for j in approved_jobs or []:
                cur.execute("""
                    INSERT INTO applications (company, role, job_link, location, platform, match_score, resume_used,
                                              recruiter_name, recruiter_email)
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (job_link) DO UPDATE SET recruiter_name = EXCLUDED.recruiter_name,
                                                         recruiter_email = EXCLUDED.recruiter_email""",
                            (j.get("Target Company", "Unknown Company"), j.get("Job Profile"), j.get("Job Link"),
                             j.get("Location"), j.get("Platform Name"), f"{j.get('Match Score', 'N/A')}%", candidate_name,
                             j.get("Prospect_Employer_Name"), j.get("Prospect_Employer_Profile_mail")))

            for d in drafts or []:
                cur.execute("""
                    INSERT INTO outreach_drafts (job_link, company, role, recipient_name, recipient_email, subject, body, word_count, linkedin_note)
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (job_link) DO UPDATE SET recipient_name = EXCLUDED.recipient_name, recipient_email = EXCLUDED.recipient_email,
                        subject = EXCLUDED.subject, body = EXCLUDED.body, word_count = EXCLUDED.word_count,
                        linkedin_note = EXCLUDED.linkedin_note, created_at = now()""",
                            (d["job_link"], d["company"], d["role"], d["recipient_name"], d["recipient_email"],
                             d["subject"], d["body"], d["word_count"], d.get("linkedin_note")))

            changes = process_new_job_emails(cur)
            print(f"  ✓ Tracker updated: {len(approved_jobs or [])} approved job(s), {len(drafts or [])} draft(s), {changes} stage change(s).")

            cur.execute("""SELECT company AS "Company", role AS "Role", stage AS "Stage",
                                  to_char(stage_updated_at, 'YYYY-MM-DD HH24:MI') AS "Stage Updated", last_email_subject AS "Last Email",
                                  location AS "Location", platform AS "Platform", match_score AS "Match Score",
                                  resume_used AS "Resume Used", recruiter_name AS "Recruiter", recruiter_email AS "Recruiter Email",
                                  job_link AS "Job Link", to_char(created_at, 'YYYY-MM-DD') AS "Added On"
                           FROM applications ORDER BY stage_updated_at DESC""")
            apps_df = pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
            cur.execute("""SELECT company AS "Company", role AS "Role", recipient_name AS "To", recipient_email AS "Email",
                                  subject AS "Subject", body AS "Draft", word_count AS "Words", linkedin_note AS "LinkedIn Note", job_link AS "Job Link",
                                  to_char(created_at, 'YYYY-MM-DD') AS "Drafted On"
                           FROM outreach_drafts ORDER BY created_at DESC""")
            drafts_df = pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
            cur.execute("""SELECT to_char(e.received_at, 'YYYY-MM-DD HH24:MI') AS "Received", e.from_name AS "From",
                                  e.from_email AS "From Email", e.subject AS "Subject", e.stage_detected AS "Stage Detected",
                                  a.company AS "Matched Company", a.role AS "Matched Role", LEFT(e.body_text, 2000) AS "Email Text"
                           FROM job_emails e LEFT JOIN applications a ON a.id = e.application_id
                           WHERE COALESCE(e.stage_detected, '') NOT IN ('Promotional', 'Not job related')
                           ORDER BY e.received_at DESC NULLS LAST, e.id DESC""")
            emails_df = pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
        _write_extra_sheets({"Applications": apps_df, "Outreach Drafts": drafts_df, "Email Log": emails_df})
        print(f"  ✓ 'Applications', 'Outreach Drafts' and 'Email Log' ({len(emails_df)} emails) sheets refreshed in {EXCEL_PATH}")
    except Exception as e:
        print(f"  ⚠️ Application tracker issue: {e}")


# ---------- Outreach email agent ----------
def load_style_examples() -> str:
    if not os.path.exists(STYLE_EXAMPLES_FILE):
        return ""
    with open(STYLE_EXAMPLES_FILE, "r", encoding="utf-8") as f:
        examples = [e.strip() for e in re.split(r"^\s*---\s*$", f.read(), flags=re.M) if e.strip()]
    return "\n\n".join(f"Example {i}:\n{e}" for i, e in enumerate(examples[:5], 1))


def fetch_job_description(url: str, max_chars: int = 4000) -> str:
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"}, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "header", "footer", "noscript"]):
            tag.decompose()
        return re.sub(r"\s+", " ", soup.get_text(" ", strip=True))[:max_chars]
    except Exception:
        return ""


def _llm_text(prompt: str) -> str:
    """Claude CLI first (same as scoring), local Qwen as fallback."""
    try:
        if os.path.exists(CLAUDE_EXE):
            res = subprocess.run([CLAUDE_EXE, "-p", prompt, "--tools", ""], capture_output=True, text=True,
                                 encoding="utf-8", timeout=120)
            if res.returncode == 0 and res.stdout.strip():
                return res.stdout.strip()
    except Exception as e:
        print(f"  ⚠️ Claude CLI drafting notice: {e}")
    try:
        return ollama_llm.invoke(prompt).content.strip()
    except Exception as e:
        print(f"  ⚠️ No model could draft this ({str(e)[:120]}); left blank in the sheet.")
        return ""


def draft_outreach_email(resume_summary: str, job: Dict[str, Any], style_examples: str) -> Dict[str, Any]:
    jd = fetch_job_description(job.get("Job Link", "")) or "(Job description not available; use the title and company.)"
    style_block = f"Match the tone, structure and phrasing of these emails I wrote before:\n{style_examples}" if style_examples \
        else "Write in a warm, confident, plain-spoken voice."
    prompt = f"""[Output ONLY the email in this exact format, nothing else:
Subject: <subject line>

<email body>]

Write a short outreach email from me to the hiring/recruiting manager for this job.
Start the body with exactly "Hi {{first_name}}," (keep the placeholder as is) and end with my first name.

Rules: under 200 words for the body; sound like a real person, not a template; be precise and specific:
connect 2-3 concrete things from my profile to what this JD asks for; one clear ask (a short call or referral to the right person);
no buzzwords, no flattery, no "I hope this email finds you well".

{style_block}

My profile:
{resume_summary}

Job: {job.get('Job Profile')} at {job.get('Target Company')} ({job.get('Location')})
Job description:
{jd}"""
    text = _llm_text(prompt)
    for _ in range(2):  # enforce the 200-word limit
        body_words = len(re.sub(r"(?is)^subject:.*?\n", "", text).split())
        if body_words < 200:
            break
        text = _llm_text(f"Shorten this email to under 180 words. Keep the same voice, format, placeholder and ask:\n\n{text}")

    m = re.match(r"(?is)\s*subject:\s*(.+?)\n(.*)", text)
    subject, body = (m.group(1).strip(), m.group(2).strip()) if m else (f"{job.get('Job Profile')} at {job.get('Target Company')}", text)
    return {"job_link": job.get("Job Link"), "company": job.get("Target Company"), "role": job.get("Job Profile"),
            "subject": subject, "body": body, "word_count": len(body.split()),
            "linkedin_note": draft_linkedin_note(resume_summary, job, style_examples, jd)}


# ---------- LinkedIn connection-request note ----------
LINKEDIN_NOTE_LIMIT = 200  # LinkedIn's cap on connection notes; set 300 only if your Premium account shows 300


def _fit_note(text: str, limit: int = LINKEDIN_NOTE_LIMIT) -> str:
    """Hard cap: cut at the last full sentence (or word) that fits the limit."""
    text = re.sub(r"\s+", " ", text or "").strip().strip('"')
    if len(text) <= limit:
        return text
    cut = text[:limit]
    end = max(cut.rfind(". "), cut.rfind("! "), cut.rfind("? "), cut.rfind("."))
    return cut[:end + 1] if end > limit * 0.6 else cut[:cut.rfind(" ")].rstrip(",;:-") + "…"


def draft_linkedin_note(resume_summary: str, job: Dict[str, Any], style_examples: str, jd: str) -> str:
    budget = LINKEDIN_NOTE_LIMIT - 12  # leaves room for the recipient's first name
    style_block = f"Match the tone of these messages I wrote before:\n{style_examples}" if style_examples else ""
    prompt = f"""[Output ONLY the note text, nothing else.]
Write a LinkedIn connection-request note from me to someone at this company (a hiring manager or recruiter).
HARD LIMIT: {budget} characters including spaces. Start with exactly "Hi {{first_name}}," (keep the placeholder).
Keep it non-transactional, a genuine connection: one specific, sincere point of common ground (the problem their team
works on, the space we both work in, something concrete about their product). Use the job description only to
understand what the team does; do NOT mention the opening, applying, a job, a referral, an interview or a call.
End with a simple wish to connect or keep in touch. Warm, curious, peer-to-peer. No links, emojis or hashtags.

{style_block}

My profile:
{resume_summary}

Job: {job.get('Job Profile')} at {job.get('Target Company')}
Job description:
{jd[:1500]}"""
    note = _llm_text(prompt)
    for _ in range(2):  # ask for a shorter version before falling back to a hard cut
        if len(note) <= budget:
            break
        note = _llm_text(f"Rewrite this LinkedIn note in under {budget} characters including spaces. "
                         f"Keep the same voice, the 'Hi {{first_name}},' start and the no-job-ask, connection-only tone. "
                         f"Output only the note:\n\n{note}")
    return _fit_note(note, budget)


# Fan-out after approval: contact search and email drafting run in parallel
def start_post_approval_node(state: PipelineState) -> Dict[str, Any]:
    print(f"\n🔀 {len(state.get('approved_jobs', []))} approved job(s): finding contacts + drafting outreach emails in parallel...")
    return {}


# Node 5b: Outreach Email Agent
def draft_outreach_node(state: PipelineState) -> Dict[str, Any]:
    approved = state.get("approved_jobs", [])
    if not approved:
        return {"outreach_drafts": []}
    print(f"\n✍️ Node 5b: Drafting {len(approved)} outreach email(s) in your style...")
    style = load_style_examples()
    if not style:
        print(f"  ⚠️ No style examples at {STYLE_EXAMPLES_FILE}; drafting in a neutral voice.")
    drafts = []
    for job in approved:
        try:
            d = draft_outreach_email(state.get("resume_summary", ""), job, style)
            drafts.append(d)
            print(f"  ✓ {job.get('Target Company')}: draft ready ({d['word_count']} words)")
        except Exception as e:
            print(f"  ⚠️ Draft issue for {job.get('Target Company')}: {e}")
    return {"outreach_drafts": drafts}


# Node 7: Application Tracker Sync (after the existing Excel export)
def sync_tracker_node(state: PipelineState) -> Dict[str, Any]:
    print("\n📌 Node 7: Syncing application tracker (Neon) + outreach drafts...")
    contacts = {j.get("Job Link"): j for j in state.get("approved_jobs", [])}
    drafts = []
    for d in state.get("outreach_drafts", []):  # address each draft to the contact Node 5 found
        c = contacts.get(d["job_link"], {})
        name = c.get("Prospect_Employer_Name") or "Recruiting Team"
        first = name.split()[0] if name not in ("Recruiting Team", "", None) else "there"
        drafts.append({**d, "recipient_name": name, "recipient_email": c.get("Prospect_Employer_Profile_mail"),
                       "body": d["body"].replace("{first_name}", first),
                       "linkedin_note": _fit_note((d.get("linkedin_note") or "").replace("{first_name}", first))})
    sync_application_tracker(state.get("approved_jobs", []), drafts, state.get("candidate_name", ""))
    return {}


# ==========================================
# 6. GRAPH COMPILATION
# ==========================================
def hitl_routing(state: PipelineState) -> str:
    decision = state.get("user_decision", "proceed")
    if decision == "review_more":
        return "human_review"
    elif decision == "search_more":
        return "search_jobs"
    return "find_contacts"

workflow = StateGraph(PipelineState)

workflow.add_node("parse_resume", parse_resume_node)
workflow.add_node("start_search", start_search_node)
workflow.add_node("search_jobs", search_jobs_node)
workflow.add_node("shortlist_companies", shortlist_companies_node)
workflow.add_node("search_career_pages", search_career_pages_node)
workflow.add_node("search_job_boards", search_job_boards_node)
workflow.add_node("discover_companies", discover_companies_node)
workflow.add_node("search_discovered_careers", search_discovered_careers_node)
workflow.add_node("merge_results", merge_search_results_node)
workflow.add_node("evaluate_matches", evaluate_matches_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("find_contacts", find_contacts_node)
workflow.add_node("export_excel", export_excel_node)
workflow.add_node("start_post_approval", start_post_approval_node)
workflow.add_node("draft_outreach", draft_outreach_node)
workflow.add_node("sync_tracker", sync_tracker_node)

workflow.add_edge(START, "parse_resume")
workflow.add_edge("parse_resume", "start_search")
# Three search agents run in parallel (LinkedIn | web | company discovery), then join before scoring
workflow.add_edge("start_search", "search_jobs")
workflow.add_edge("start_search", "shortlist_companies")
workflow.add_edge("start_search", "discover_companies")
workflow.add_edge("shortlist_companies", "search_career_pages")
workflow.add_edge("search_career_pages", "search_job_boards")
workflow.add_edge("discover_companies", "search_discovered_careers")
workflow.add_edge(["search_jobs", "search_job_boards", "search_discovered_careers"], "merge_results")
workflow.add_edge("merge_results", "evaluate_matches")
workflow.add_edge("evaluate_matches", "human_review")

workflow.add_conditional_edges(
    "human_review",
    hitl_routing,
    {
        "human_review": "human_review",
        "search_jobs": "start_search",  # broader search re-runs both branches (next company batch too)
        "find_contacts": "start_post_approval"  # approved → contacts + email drafts in parallel
    }
)

workflow.add_edge("start_post_approval", "find_contacts")
workflow.add_edge("start_post_approval", "draft_outreach")
workflow.add_edge(["find_contacts", "draft_outreach"], "export_excel")
workflow.add_edge("export_excel", "sync_tracker")
workflow.add_edge("sync_tracker", END)

app = workflow.compile()

# ==========================================
# 7. EXECUTION
# ==========================================
# 'track' = only read new job emails from Neon and refresh the tracker (also settable via GET_HIRED_MODE=track in .env)
if __name__ == "__main__" and (os.getenv("GET_HIRED_MODE") or input(
        "Press Enter to search jobs, or type 'track' to only update the application tracker: ")).strip().lower().startswith("t"):
    print("\n📌 Updating application tracker from your job emails...")
    sync_application_tracker()
elif __name__ == "__main__":
    print("\n" + "="*55)
    print("🤖 HITL JOB AGENT (Claude Pro + Exa Contacts + Local CV)")
    print("="*55 + "\n")

    available_cvs = []
    if os.path.exists(RESUMES_DIR):
        available_cvs = [os.path.splitext(f)[0] for f in os.listdir(RESUMES_DIR) if f.endswith(".pdf")]

    if available_cvs:
        print("Available Resumes in Directory:")
        for idx, cv in enumerate(available_cvs, 1):
            print(f"  [{idx}] {cv}")
        choice = input(f"\nSelect Persona (1-{len(available_cvs)}): ").strip()
        selected_persona = available_cvs[int(choice) - 1] if choice.isdigit() and 1 <= int(choice) <= len(available_cvs) else available_cvs[0]
    else:
        selected_persona = "Simar_Dhingra_CV"

    print(f"\n👉 Active Candidate Profile: {selected_persona}\n")

    dur_input = input("Enter posting recency (e.g. '24 hours', '3 days', '1 week') [Default: 1 week]: ").strip() or "1 week"
    loc_input = input("Enter locations (e.g. 'Delhi / Bangalore / European Union (Remote)') [Default: Remote]: ").strip() or "Remote"
    roles_input = input("Enter target roles (e.g. 'Product Manager / Chief of Staff'): ").strip() or "Product Manager"
    target_count = input("How many approved jobs would you like to target? [Default: 5]: ").strip() or "5"
    years_input = input("Your years of experience (jobs whose JD asks for far more, or a range far below, are skipped) [Enter = read from CV]: ").strip()

    mode_input = input("Search agents: 'linkedin' only, 'web' only (career pages, VC portfolios, ATS, job boards, recruiters, Reddit, X), "
                       "'discover' only (company discovery by CV domain) or Enter for ALL THREE in parallel [Default: all]: ").strip().lower()
    search_mode = ("linkedin" if mode_input.startswith("l") else "career" if mode_input.startswith(("c", "w", "o"))
                   else "discover" if mode_input.startswith("d") else "both")
    domain_input, company_count, funds_input, discovery_count = "", "15", "", "60"
    if search_mode in ("both", "discover"):
        discovery_count = input("Company discovery: companies to read per location (50-100) [Default: 60]: ").strip() or "60"
    if search_mode != "linkedin":
        domain_input = input("Target domain: Enter for ALL domains, 'resume' for your CV's domain, or type one (e.g. 'Fintech') [Default: all]: ").strip()
        domain_input = "" if domain_input.lower() in ("all", "any") else domain_input
        company_count = input("Companies to shortlist per location [Default: 15]: ").strip() or "15"
        funds_input = input(f"VC/PE funds to include (e.g. 'Accel / Matrix Partners / Sequoia') [Default: {', '.join(DEFAULT_VC_FUNDS)} + local funds]: ").strip()

    inputs = {
        "candidate_name": selected_persona,
        "roles": [r.strip() for r in re.split(r"[/,]", roles_input) if r.strip()],
        "target_geos": [l.strip() for l in re.split(r"[/,]", loc_input) if l.strip()],
        "job_duration_days": parse_duration_to_days(dur_input),
        "duration_label": dur_input,
        "target_approved_count": int(target_count),
        "approved_jobs": [],
        "search_iteration": 1,
        "search_mode": search_mode,
        "target_domain": domain_input,
        "career_company_count": int(company_count) if company_count.isdigit() else 15,
        "shortlisted_companies": [],
        "company_batch": [],
        "career_jobs": [],
        "board_jobs": [],
        "vc_funds": [f.strip() for f in re.split(r"[/,]", funds_input) if f.strip()],
        "fund_boards": [],
        "new_fund_boards": [],
        "outreach_drafts": [],
        "scoring_mode": "",
        "candidate_years": parse_candidate_years(years_input, selected_persona),
        "shown_job_keys": [],
        "discovery_company_count": int(discovery_count) if discovery_count.isdigit() else 60,
        "discovery_batch": [],
        "discovery_jobs": []
    }

    print("\n🚀 Starting Pipeline Execution...\n")
    final_output = app.invoke(inputs)

    print("\n" + "="*55)
    print("📊 LOW MATCH JOBS ANALYSIS (<50 words per role)")
    print("="*55)
    print(final_output.get("low_match_summary"))
    print("\n✓ Process completed! Check Matched_Jobs.xlsx for your approved jobs and verified recruiter contacts.")

📦 Indexing updated resume persona: 1792075619258-Simar_CV_Feedback_AI_Europe.pdf into ChromaDB...
  ✓ 1792075619258-Simar_CV_Feedback_AI_Europe.pdf indexed with persona metadata.
📦 Indexing updated resume persona: CV-Simar-Dhingra.pdf into ChromaDB...
  ✓ CV-Simar-Dhingra.pdf indexed with persona metadata.
📦 Indexing updated resume persona: LI_Profile.pdf into ChromaDB...
  ✓ LI_Profile.pdf indexed with persona metadata.

🤖 HITL JOB AGENT (Claude Pro + Exa Contacts + Local CV)

Available Resumes in Directory:
  [1] 1792075619258-Simar_CV_Feedback_AI_Europe
  [2] CV-Simar-Dhingra
  [3] LI_Profile
  [4] Simar_Dhingra_CV
  [5] Simar_Dhingra_CV_PM



Select Persona (1-5):  5



👉 Active Candidate Profile: Simar_Dhingra_CV_PM



Enter posting recency (e.g. '24 hours', '3 days', '1 week') [Default: 1 week]:  24 hours
Enter locations (e.g. 'Delhi / Bangalore / European Union (Remote)') [Default: Remote]:  India remote
Enter target roles (e.g. 'Product Manager / Chief of Staff'):  Product manager
How many approved jobs would you like to target? [Default: 5]:  15
Search agents: 'linkedin' only, 'web' only (career pages, VC portfolios, ATS, job boards, recruiters, Reddit, X) or Enter for BOTH in parallel [Default: both]:  
Target domain: Enter for ALL domains, 'resume' for your CV's domain, or type one (e.g. 'Fintech') [Default: all]:  all
Companies to shortlist per location [Default: 15]:  
VC/PE funds to include (e.g. 'Accel / Matrix Partners / Sequoia') [Default: Accel, Matrix Partners, Peak XV, Lightspeed, Antler, Entrepreneur First, Y Combinator + local funds]:  



🚀 Starting Pipeline Execution...


📄 Node 1: Checking Resume Profile Cache for 'Simar_Dhingra_CV_PM'...
  ⚡ [Instant] Loaded cached profile for 'Simar_Dhingra_CV_PM' (0ms)!

🔀 Starting search branches: LinkedIn agent + web agent (parallel)

🔍 Node 2: Searching live postings [Iteration 1] (past 1 days)...
  ⚡ Querying live LinkedIn index for 'Product manager' in 'India remote'...
  ✓ Found 10 fresh live postings in ~3 seconds!

🏢 Node 2a: Shortlisting companies for ['Product manager'] in 'all domains' (15 per location)...
  ↳ Matching similar titles: product manager, product owner, product leader, product strategist, product development manager
  ✓ India remote: VC/PE funds → Accel Limited (0 portfolio cos), Z47 (job board), Peak XV Partners (0 portfolio cos), Lightspeed India (job board), Antler (job board), Entrepreneurs First (2 portfolio cos), Y Combinator (0 portfolio cos), IvyCap Ventures (0 portfolio cos), Blume Ventures (0 portfolio cos), GVFL (0 portfolio cos), JSW Ventures (0

Enter IDs to APPROVE (e.g., '1, 3'), 'all', 'none', or 'done' to stop:  1,3,4,5


... (5 outputs left out)

Enter IDs to APPROVE (e.g., '1, 3'), 'all', 'none', or 'done' to stop:  1,2,5


✓ Total Approved: 14/15

🧑‍💼 HUMAN-IN-THE-LOOP SIGN-OFF (Approved so far: 14/15)

[1] Product Manager @ Weave
    Location: India remote | Match Score: 80% (Good)
    Rationale: General PM role; candidate's product ownership and data-driven discovery fit well.
    Link: https://jobs.ashbyhq.com/weave/5ec9edb5-fb5d-408c-a822-a92139dd61f8/application

[2] Sr. Technical Product Manager @ Kantiv
    Location: India remote | Match Score: 80% (Good)
    Rationale: Senior technical PM; candidate's AI/ML and quant background supports technical product ownership.
    Link: https://jobs.ashbyhq.com/kantiv/f38fb2c6-2d4d-4d90-848c-e52b2664ff62

[3] Sr Product Manager - IC @ Signoz
    Location: India remote | Match Score: 80% (Good)
    Rationale: Senior IC PM at an observability company; real-time monitoring and reporting experience is relevant.
    Link: https://jobs.ashbyhq.com/SigNoz/a41d80b2-389e-42c3-817e-feb7c3aa8fff?locationId=fd8c83bf-e884-43d7-be81-d7b794715a2c&workplaceType=Remote

[4] 

Enter IDs to APPROVE (e.g., '1, 3'), 'all', 'none', or 'done' to stop:  1,3,5


✓ Total Approved: 17/15

🤝 Node 5: Enriching Approved Jobs with Real Recruiter Contacts & Verified Emails...
  ✓ upcover: Recruiting Team (hiring@upcover.com)
  📧 Verifying inbox for 'Kavita Nair' at 'AlphaSense India'...
  ✓ AlphaSense India: Kavita Nair (kavita@alphasenseindia.com)
  📧 Verifying inbox for 'Anna Smirnova' at 'Wrike'...
  ✓ Wrike: Anna Smirnova (anna@wrike.com)
  📧 Verifying inbox for 'Pranit Shrivastava' at 'Housewise.in • Pune • Remote (Work from Home)'...
  ✓ Housewise.in • Pune • Remote (Work from Home): Pranit Shrivastava (pranit@housewiseinpuneremoteworkfromhome.com)
  📧 Verifying inbox for 'ritika sharma' at 'HackerRank'...
  ✓ HackerRank: ritika sharma (ritika.sharma@hackerrank.com)
  📧 Verifying inbox for 'pritinanda panda' at 'HighLevel'...
  ✓ HighLevel: pritinanda panda (pritinanda@highlevel.com)
  📧 Verifying inbox for 'ayush jain' at 'Firstwork'...
  ✓ Firstwork: ayush jain (ayush@firstwork.com)
  ✓ Jobgether: Recruiting Team (hiring@jobgether.com)
  📧 Ve

In [ ]:
# ==========================================
# JOB QUALITY EVAL: small labelled regression checks (paste into the cell BELOW the main cell and run it)
# Part 1 (offline, ~1s): company parsing, job-ID dedupe, posting-age parsing, closed-page detection on saved HTML.
# Part 2 (live, ~20s, set LIVE=True): the real links you reported as dead / old must be dropped by the link check.
# Every case comes from a bad result you saw on 2026-10-31. Add a line here whenever you spot a new bad result.
# ==========================================
import requests as _rq
from unittest import mock as _mock

LIVE = False   # True = also open the real links below (run on your machine; needs internet)

COMPANY_CASES = [  # (title as scraped, link, expected company, expected clean title)
    ("Momentive Software hiring AI-Native Senior Product Manager in United States",
     "https://www.linkedin.com/jobs/view/ai-native-senior-product-manager-at-momentive-software-4434376449",
     "Momentive Software", "AI-Native Senior Product Manager"),
    ("Ladders hiring Senior Product Manager, AI in United States",
     "https://www.linkedin.com/jobs/view/senior-product-manager-ai-at-ladders-4431087385", "Ladders", "Senior Product Manager, AI"),
    ("Sr. Product Manager - AI Security - REMOTE",
     "https://www.linkedin.com/jobs/view/sr-product-manager-ai-security-remote-at-lenovo-4426167993", "Lenovo", None),
    ("Senior Product Manager, AI", "https://www.linkedin.com/jobs/view/senior-product-manager-ai-at-bold-4235530373", "Bold", None),
    ("Sr. Product Manager, AI-Powered Engagement",
     "https://www.linkedin.com/jobs/view/sr-product-manager-ai-powered-engagement-at-five9-4160232928", "Five9", None),
    ("Tempus AI hiring Senior Technical Product Manager (REMOTE, USA) in Chicago, IL",
     "https://www.linkedin.com/jobs/view/senior-technical-product-manager-remote-usa-at-tempus-ai-4436264474",
     "Tempus AI", "Senior Technical Product Manager (REMOTE, USA)"),
    ("NAVEX hiring Senior Technical Product Manager, AI • Remote (Work from Home)",
     "https://himalayas.app/companies/navex/jobs/senior-technical-product-manager-ai-561672161?pos=1&src=search",
     "NAVEX", "Senior Technical Product Manager, AI"),
    ("Senior Product Manager (AI & Workflow Automation)",
     "https://jobs.workable.com/view/vAmbAr94pVtFZXCpNwQ4Mv/remote-senior-product-manager-(ai-%26-workflow-automation)-in-united-states-at-arcsite",
     "Arcsite", None),
    ("Senior Product Manager, Agentic AI", "https://jobs.ashbyhq.com/jerry.ai/ad20fade-9e1b-4096-bae1-531211b026b2", "Jerry.ai", None),
    ("Senior AI Product Manager | REMOTE", "https://jobs.lever.co/distru/9c3ca4f6-fd74-4d70-b7e3-682c914c1a22", "Distru", None),
    ("Product Manager | Insurance & AI", "https://careers.antler.co/companies/upcover-2/jobs/54596870-product-manager-insurance-ai",
     "Upcover", None),
    ("Product Manager at Scale AI", "", "Scale AI", "Product Manager"),
]
SAME_JOB_LINKS = [  # every group = one posting reached through different URLs (the batch-2 repeats)
    ["https://jobs.lever.co/distru/9c3ca4f6-fd74-4d70-b7e3-682c914c1a22",
     "https://jobs.lever.co/distru/9c3ca4f6-fd74-4d70-b7e3-682c914c1a22/apply?lever-source=LinkedIn"],
    ["https://jobs.ashbyhq.com/teamworks/a137182f-abe7-4a76-8dac-5dc13b9aaf69/",
     "https://jobs.ashbyhq.com/teamworks/a137182f-abe7-4a76-8dac-5dc13b9aaf69"],
    ["https://www.linkedin.com/jobs/view/senior-product-manager-ai-at-ladders-4431087385",
     "https://es.linkedin.com/jobs/view/senior-product-manager-ai-at-ladders-4431087385?refId=abc&trackingId=xyz",
     "https://www.linkedin.com/jobs/view/4431087385/"],
]
SAME_JOB_TITLES = [("Lenovo", "Sr. Product Manager - AI Security - REMOTE"), ("Lenovo", "Senior Product Manager, AI Security (Remote)")]
AGE_CASES = [("4 months ago", 120), ("Reposted 2 weeks ago", 14), ("30+ days ago", 30), ("2026-10-20", None), ("Past 7 days", "none")]

# Saved pages: (link, fake HTTP status, fake body, expected alive)
_LI_CLOSED = '<html><span class="posted-time-ago__text">4 months ago</span><figure class="closed-job">No longer accepting applications</figure></html>'
_LI_OPEN = ('<html><a class="topcard__org-name-link">Ladders</a><span class="posted-time-ago__text">2 days ago</span>'
            '<div class="show-more-less-html__markup">' + "Responsibilities: own the AI roadmap. " * 20 + '</div></html>')
_LD_EXPIRED = '<script type="application/ld+json">{"@type":"JobPosting","validThrough":"2026-01-01","title":"PM"}</script>'
PAGE_CASES = [
    ("https://www.linkedin.com/jobs/view/x-at-ladders-4431087385", 200, _LI_CLOSED, False),
    ("https://www.linkedin.com/jobs/view/x-at-ladders-4431087386", 200, _LI_OPEN, True),
    ("https://himalayas.app/companies/navex/jobs/pm-1", 404, "Not found", False),
    ("https://example-careers.com/jobs/123", 200, _LD_EXPIRED, False),
    ("https://example-careers.com/jobs/124", 200, "<title>Job not found</title><p>This job is no longer available.</p>", False),
    ("https://jobs.ashbyhq.com/teamworks/a137182f-abe7-4a76-8dac-5dc13b9aaf69", 200, {"jobs": [{"id": "other"}]}, False),
]
LIVE_DEAD = [  # links you reported as 'job not found' / months old / no longer accepting
    "https://jobs.workable.com/view/vAmbAr94pVtFZXCpNwQ4Mv/remote-senior-product-manager-(ai-%26-workflow-automation)-in-united-states-at-arcsite",
    "https://jobs.ashbyhq.com/jerry.ai/ad20fade-9e1b-4096-bae1-531211b026b2",
    "https://jobs.ashbyhq.com/teamworks/a137182f-abe7-4a76-8dac-5dc13b9aaf69/",
    "https://www.linkedin.com/jobs/view/ai-native-senior-product-manager-at-momentive-software-4434376449",
    "https://www.linkedin.com/jobs/view/senior-product-manager-ai-at-ladders-4431087385",
    "https://www.linkedin.com/jobs/view/sr-product-manager-ai-security-remote-at-lenovo-4426167993",
]

# --- Added 2026-11-02 from your second batch of bad results ---
NOT_A_POSTING = [  # listing / search / category / company pages: must never reach scoring
    "https://dynamitejobs.com/category/remote-product-jobs",
    "https://linkedin.com/company/best-pm-jobs",
    "https://jobs.workable.com/search/global/remote-product-manager-jobs",
    "https://himalayas.app/jobs/worldwide/product-manager",
    "https://www.linkedin.com/jobs/search/?keywords=Product%20manager&location=Remote",
    "https://weworkremotely.com/categories/remote-product-jobs",
]
REAL_POSTINGS = [  # single postings that must NOT be mistaken for listing pages
    "https://weworkremotely.com/remote-jobs/upsun-product-manager",
    "https://wellfound.com/jobs/4748746-product-manager",
    "https://himalayas.app/companies/navex/jobs/senior-technical-product-manager-ai-561672161",
    "https://jobs.lever.co/distru/9c3ca4f6-fd74-4d70-b7e3-682c914c1a22",
    "https://www.linkedin.com/jobs/view/product-manager-canvas-us-remote-at-instructure-4427869650/",
    "https://www.linkedin.com/jobs/search-results/?currentJobId=4480183341&keywords=Product%20manager%20remote%20jobs",
]
_LI_2W = '<span class="posted-time-ago__text">2 weeks ago</span><div class="show-more-less-html__markup">' + "Own the roadmap. " * 30 + "</div>"
_LI_PUBLIC_CLOSED = ('<html><title>Product Manager | Instructure | LinkedIn</title><h1>Product Manager</h1>'
                     '<span class="posted-time-ago__text">3 months ago</span>'
                     '<div class="closed-job">No longer accepting applications</div></html>')
STALE_CASES = [  # (link, fake responses {url part: (status, body)}, expect kept after the link check with a 7-day window)
    ("https://weworkremotely.com/remote-jobs/toptal-agile-product-manager",          # date only in the page text
     {"weworkremotely": (200, "<h1>Agile Product Manager</h1><p>Posted 21 days ago</p><div>" + "Own the roadmap. " * 30 + "</div>")}, False),
    ("https://weworkremotely.com/remote-jobs/upsun-product-manager",
     {"weworkremotely": (200, "<h1>Product Manager</h1><div class='listing-header'>Posted Oct 21, 2026</div>")}, False),
    ("https://wellfound.com/jobs/4748746-product-manager",                            # board blocks bots, no date anywhere
     {"wellfound": (403, "Access denied")}, False),
    ("https://www.linkedin.com/jobs/search-results/?currentJobId=4480183341&keywords=Product%20manager%20remote%20jobs",
     {"jobPosting/4480183341": (200, _LI_2W)}, False),
    ("https://www.linkedin.com/jobs/view/product-manager-canvas-us-remote-at-instructure-4427869650/",   # guest page rate-limited
     {"jobPosting/4427869650": (429, ""), "jobs/view/": (200, _LI_PUBLIC_CLOSED)}, False),
    ("https://www.linkedin.com/jobs/view/product-manager-%E2%80%94-priorauth-at-prescriberpoint-4432052578/",
     {"jobPosting/4432052578": (200, _LI_CLOSED)}, False),
    ("https://weworkremotely.com/remote-jobs/fresh-product-manager",                  # control: fresh posting stays
     {"weworkremotely": (200, "<h1>Product Manager</h1><p>Posted 2 days ago</p><div>" + "Own the roadmap. " * 30 + "</div>")}, True),
]
LIVE_DEAD += [
    "https://weworkremotely.com/remote-jobs/toptal-agile-product-manager",
    "https://wellfound.com/jobs/4748746-product-manager",
    "https://www.linkedin.com/jobs/search-results/?currentJobId=4480183341&keywords=Product%20manager%20remote%20jobs",
    "https://www.linkedin.com/jobs/view/product-manager-canvas-us-remote-at-instructure-4427869650/",
    "https://www.linkedin.com/jobs/view/product-manager-%E2%80%94-priorauth-at-prescriberpoint-4432052578/",
    "https://weworkremotely.com/remote-jobs/upsun-product-manager",
]


# --- Added 2026-11-02 (2): same job on LinkedIn and on Lever (Jobgether reposts) ---
_JG_INTRO = ("This position is posted by Jobgether on behalf of a partner company. We are currently looking for a {t} in the "
             "United States. ")
_JG_OUTRO = (" Why Jobgether? Jobgether is a talent matching platform that partners with companies worldwide to efficiently "
             "connect top talent with the right opportunities through AI-driven job matching. ")
_JD_A = ("Responsibilities: Own the product roadmap for the payments platform, run discovery with merchants, write PRDs, "
         "partner with engineering on delivery of embedded finance features, define success metrics and report on adoption. "
         "Requirements: 6+ years of product management in fintech or payments, experience with B2B APIs, strong SQL, "
         "excellent written communication, experience working with distributed teams across time zones.")
_JD_B = ("Responsibilities: Lead the AI assistant product for healthcare clinics, prioritise LLM features with clinicians, "
         "set up evaluation and guardrails for model output, manage the backlog and run weekly experiments. "
         "Requirements: 5+ years as a product manager, shipped machine learning products, HIPAA familiarity, "
         "comfortable with data analysis and with presenting to executives.")
_li = lambda title, jd: (f'<a class="topcard__org-name-link">Jobgether</a><span class="posted-time-ago__text">1 day ago</span>'
                         f'<div class="show-more-less-html__markup">{_JG_INTRO.format(t=title)}{jd}{_JG_OUTRO}</div>')
_lever = lambda title, jd: {"text": title, "createdAt": time.time() * 1000 - 86400000,
                            "descriptionPlain": _JG_INTRO.format(t=title) + jd + _JG_OUTRO, "lists": []}
CROSS_SITE_CASES = [  # (label, [(link, title, company, {url part: (status, body)})], jobs expected after dedupe)
    # url parts must be unique per job: the first part found in a URL answers it
    ("LinkedIn 4485470862 = Lever jobgether/8ad3dc68", [
        ("https://www.linkedin.com/jobs/view/4485470862/", "Senior Product Manager (Remote - US)", "Jobgether",
         {"jobPosting/4485470862": (200, _li("Senior Product Manager", _JD_A))}),
        ("https://jobs.lever.co/jobgether/8ad3dc68-a774-4270-9a21-79a04338159e", "Senior Product Manager", "",
         {"api.lever.co": (200, _lever("Senior Product Manager", _JD_A))})], 1),
    ("LinkedIn 4485463356 = Lever jobgether/5adc92e1 (real company named on LinkedIn)", [
        ("https://www.linkedin.com/jobs/view/4485463356/", "Product Manager, AI Assistant", "Acme Health",
         {"jobPosting/4485463356": (200, _li("Product Manager - AI", _JD_B))}),
        ("https://jobs.lever.co/jobgether/5adc92e1-4b78-41de-9c52-597277114cfe", "Product Manager - AI", "",
         {"api.lever.co": (200, _lever("Product Manager - AI", _JD_B))})], 1),
    ("two DIFFERENT Jobgether jobs share boilerplate: keep both", [
        ("https://jobs.lever.co/jobgether/11111111-1111-4111-8111-111111111111", "Senior Product Manager", "",
         {"postings/jobgether/11111111": (200, _lever("Senior Product Manager", _JD_A))}),
        ("https://jobs.lever.co/jobgether/22222222-2222-4222-8222-222222222222", "Senior Product Manager", "",
         {"postings/jobgether/22222222": (200, _lever("Senior Product Manager", _JD_B))})], 2),
]

LIVE_SAME_JOB = [  # EASIEST WAY TO ADD A DUPLICATE: paste the two links of one job here (checked when LIVE=True)
    ("https://www.linkedin.com/jobs/view/4485470862/", "https://jobs.lever.co/jobgether/8ad3dc68-a774-4270-9a21-79a04338159e"),
    ("https://www.linkedin.com/jobs/view/4485463356/", "https://jobs.lever.co/jobgether/5adc92e1-4b78-41de-9c52-597277114cfe"),
]


def _fake_get(routes):
    """requests.get stand-in: the first route whose key is in the URL answers."""
    def get(url, **kw):
        for part, (status, body) in routes.items():
            if part in url:
                return _mock.Mock(status_code=status, url=url, text=body if isinstance(body, str) else "",
                                  json=lambda b=body: b if isinstance(b, dict) else {})
        raise _rq.exceptions.ConnectionError("no route in test")
    return get


def run_job_quality_eval(live: bool = LIVE):
    fails, total = [], 0

    def check(ok, label):
        nonlocal total
        total += 1
        if not ok:
            fails.append(label)

    for title, link, company, clean in COMPANY_CASES:   # 1. company + clean title
        job = tidy_job_fields({"Job Profile": title, "Target Company": "Unknown Company", "Job Link": link})
        check(job["Target Company"] == company, f"company: '{title[:40]}' -> '{job['Target Company']}' (want '{company}')")
        if clean:
            check(job["Job Profile"] == clean, f"title: '{title[:40]}' -> '{job['Job Profile']}' (want '{clean}')")
    for group in SAME_JOB_LINKS:                          # 2. same posting, different URLs
        ids = {canonical_job_id(u) for u in group}
        check(len(ids) == 1, f"dedupe by ID: {ids}")
    k1, k2 = (job_dedupe_keys({"Job Link": f"https://x.com/{i}", "Target Company": c, "Job Profile": t})[1]
              for i, (c, t) in enumerate(SAME_JOB_TITLES))
    check(k1 == k2, f"dedupe by company+title: {k1} vs {k2}")
    shown = job_dedupe_keys({"Job Link": SAME_JOB_LINKS[0][0], "Target Company": "Distru", "Job Profile": "Senior AI Product Manager"})
    with _mock.patch.dict(globals(), {"load_rejected_jobs": lambda: (set(), set())}):   # ignore your real rejected_jobs.csv here
        _, drops = gate_jobs_for_scoring([{"Job Link": SAME_JOB_LINKS[0][1], "Target Company": "Distru",
                                           "Job Profile": "Senior AI Product Manager | REMOTE"}],
                                         {"roles": ["AI Product Manager"], "shown_job_keys": shown})
    check(drops["shown earlier this run"] == 1, "batch 2 still shows a job from batch 1")
    for text, days in AGE_CASES:                          # 3. posting age
        d = parse_posted_text(text)
        if days == "none":
            check(d is None, f"age: '{text}' should be unknown")
        elif days is None:
            check(d is not None, f"age: '{text}' not parsed")
        else:
            check(d is not None and abs((datetime.now() - d).days - days) <= 1, f"age: '{text}' -> {d}")
    for link, status, body, alive in PAGE_CASES:          # 4. closed / dead page detection (saved pages, no internet)
        resp = _mock.Mock(status_code=status, url=link, text=body if isinstance(body, str) else "",
                          json=lambda b=body: b if isinstance(body, dict) else {})
        _POSTING_INFO.pop(link, None)
        _ATS_BOARD_CACHE.clear()
        with _mock.patch.object(_rq, "get", return_value=resp):
            info = inspect_posting(link)
        check(info["alive"] == alive, f"page check: {link[:60]} -> {info['alive']} ({info['reason']}), want {alive}")
        _POSTING_INFO.pop(link, None)
    _flag = globals().get("is_listing_url", lambda u: False)
    for u in NOT_A_POSTING:                               # 6. listing / search / company pages
        check(_flag(u), f"listing page not caught: {u}")
    for u in REAL_POSTINGS:
        check(not _flag(u), f"real posting flagged as a listing: {u}")
    for link, routes, keep in STALE_CASES:                # 7. old / closed postings that got through on 2026-11-02
        _POSTING_INFO.clear()
        _ATS_BOARD_CACHE.clear()
        job = tidy_job_fields({"Job Profile": "Product Manager", "Target Company": "", "Job Link": link, "Date Posted": "Not listed"})
        with _mock.patch.object(_rq, "get", side_effect=_fake_get(routes)), _mock.patch("time.sleep"):
            kept, _ = verify_postings([job], days=7)
        check(bool(kept) == keep, f"age/closed: {link[:70]} kept={bool(kept)} "
              f"({(kept[0].get('Link Status'), kept[0].get('Date Posted')) if kept else 'dropped'}), want kept={keep}")
    _merge = globals().get("dedupe_across_sites", lambda jobs, shown=None: (jobs, 0))
    for label, entries, want in CROSS_SITE_CASES:         # 8. same job on two sites
        _POSTING_INFO.clear()
        _ATS_BOARD_CACHE.clear()
        jobs, routes = [], {}
        for link, title, company, r in entries:
            routes.update(r)
            jobs.append(tidy_job_fields({"Job Profile": title, "Target Company": company or "Unknown Company",
                                         "Job Link": link, "Date Posted": "Not listed"}))
        with _mock.patch.object(_rq, "get", side_effect=_fake_get(routes)), _mock.patch("time.sleep"):
            kept, _ = verify_postings(jobs, days=7)
        merged, _ = _merge(kept)
        check(len(merged) == want, f"cross-site: {label}: {len(merged)} job(s) left, want {want}")
    _POSTING_INFO.clear()
    if live:                                              # 5. the real links you reported
        jobs = [tidy_job_fields({"Job Profile": "Senior Product Manager", "Target Company": "", "Job Link": u,
                                 "Date Posted": "Not listed"}) for u in LIVE_DEAD]
        kept, _ = verify_postings(jobs, days=7)
        for j in kept:
            check(False, f"LIVE: still kept: {j['Job Link'][:80]} ({j.get('Link Status')})")
        total += len(LIVE_DEAD) - len(kept)
        for pair in LIVE_SAME_JOB:                        # real pages: the two copies must merge into one
            _POSTING_INFO.clear()
            jobs = [tidy_job_fields({"Job Profile": "Product Manager", "Target Company": "Unknown Company", "Job Link": u,
                                     "Date Posted": "Not listed"}) for u in pair]
            kept, _ = verify_postings(jobs, days=365)
            merged, _ = dedupe_across_sites(kept)
            check(len(kept) < 2 or len(merged) == 1,
                  f"LIVE same job: {pair[0][:50]} + {pair[1][:50]} -> {len(merged)} jobs (JD read: {[j.get('JD Read') for j in kept]})")

    print(f"\n🧪 Job quality eval: {total - len(fails)}/{total} checks passed" + (" (offline only, set LIVE=True for real links)" if not live else ""))
    for f in fails:
        print(f"   ✗ {f}")
    return not fails


run_job_quality_eval()


In [ ]:
# ==========================================
# COMPANY DISCOVERY EVAL (paste into a new cell BELOW the main cell and run it; offline, ~2s, no API calls)
# Checks the 3rd search agent: batch rotation (>= 70-80% new companies), stage mix, hot carry-overs,
# 2-3 roles per company, freshness badges, "two similar roles at one company are NOT merged as duplicates",
# Hacker News 'Who is hiring' parsing + companies, and a full mocked run of the three parallel search branches.
# The live part (real careers pages) can only run on your machine: run the agent with search mode 'discover'.
# ==========================================
import tempfile as _tf
import requests as _rq
from unittest import mock as _mock
from datetime import datetime as _dt, timedelta as _td
from langgraph.graph import StateGraph as _SG, START as _START, END as _END


def _fake_pool(n_per_stage=40, hot=(), old=(), dead=()):
    out, today = [], _dt.now()
    for s_i, stage in enumerate(COMPANY_STAGES):
        for i in range(n_per_stage):
            key = f"co{s_i}-{i}.com"
            c = {"name": f"Co {s_i}-{i}", "website": f"https://{key}", "key": key, "stage": stage}
            if key in hot:
                c.update(matches=2, recent_posts=5, newest_post=(today - _td(days=1)).strftime("%Y-%m-%d"))
            if key in old:
                c.update(matches=2, recent_posts=5, newest_post=(today - _td(days=40)).strftime("%Y-%m-%d"))
            if key in dead:
                c["no_source"] = 2
            out.append(c)
    return out


def _days_ago(d):
    return (_dt.now() - _td(days=d)).strftime("%Y-%m-%d")


class _R:   # Exa result stand-in
    def __init__(self, title, url):
        self.title, self.url = title, url


def run_company_discovery_eval():
    fails, total = [], 0

    def check(ok, label):
        nonlocal total
        total += 1
        if not ok:
            fails.append(label)

    # 1. Rotation: batch 2 is >= 75% new; carry-overs are only last batch's HOT companies, capped at 25%
    pool = _fake_pool()
    b1 = select_company_batch(pool, [], 60, new_share=0.75, seed=1)
    check(len(b1) == 60, f"batch 1 size {len(b1)}, want 60")
    keys1 = [c["key"] for c in b1]
    hot = set(keys1[:30])                                   # 30 of batch 1 posted recently
    pool = _fake_pool(hot=hot)
    picked = {k: _days_ago(0) for k in keys1}
    b2 = select_company_batch(pool, keys1, 60, new_share=0.75, picked=picked, seed=2)
    overlap = len(set(keys1) & {c["key"] for c in b2})
    check(len(b2) == 60, f"batch 2 size {len(b2)}, want 60")
    check(overlap <= 15, f"batch 2 repeats {overlap}/60 companies from batch 1 (max 15 = 25%)")
    check(overlap == 15, f"batch 2 should recheck 15 hot companies, got {overlap}")
    check(all(c["key"] in hot for c in b2 if c["key"] in keys1), "a non-hot company was carried over")
    for share, max_rep in ((0.7, 18), (0.8, 12)):
        b = select_company_batch(pool, keys1, 60, new_share=share, picked=picked, seed=3)
        rep = len(set(keys1) & {c["key"] for c in b})
        check(rep <= max_rep, f"new_share={share}: {rep} repeats, max {max_rep}")

    # 2. No hot companies -> zero repeats; never-picked companies come before already-picked ones
    pool = _fake_pool()
    b2 = select_company_batch(pool, keys1, 60, picked=picked, seed=4)
    check(not set(keys1) & {c["key"] for c in b2}, "repeats without any recent postings")
    keys2 = [c["key"] for c in b2]
    picked.update({k: _days_ago(0) for k in keys2})
    b3 = select_company_batch(pool, keys2, 60, picked=picked, seed=5)
    never = {c["key"] for c in pool} - set(keys1) - set(keys2)        # 40 companies never picked yet
    check(never <= {c["key"] for c in b3}, f"batch 3 skipped {len(never - {c['key'] for c in b3})} never-picked companies")

    # 3. Stage mix: every stage present roughly in its share (seed 20%, A-B 25%, growth 25%, large 30%)
    counts = {s: sum(1 for c in b1 if c["stage"] == s) for s in COMPANY_STAGES}
    for s, (share, _) in COMPANY_STAGES.items():
        check(abs(counts[s] - round(60 * share)) <= 1, f"stage {s}: {counts[s]} in batch, want ~{round(60 * share)}")

    # 4. Hot = recent; a company whose last posting is 40 days old is not 'hot'; dead careers pages are skipped
    pool = _fake_pool(old=set(keys1[:30]), dead={"co0-39.com", "co1-39.com"})
    b = select_company_batch(pool, keys1, 60, picked=picked, seed=6)
    check(not set(keys1) & {c["key"] for c in b}, "a company with only 40-day-old postings was carried over as hot")
    check(not {"co0-39.com", "co1-39.com"} & {c["key"] for c in b}, "company with no careers page (2 tries) was picked again")
    # Small pool: fewer companies rather than breaking the overlap rule
    tiny = _fake_pool(n_per_stage=5)
    b = select_company_batch(tiny, [c["key"] for c in tiny[:10]], 60)
    check(len(b) == 10 and not {c["key"] for c in tiny[:10]} & {c["key"] for c in b}, f"tiny pool: {len(b)} picked")

    # 5. 2-3 roles per company: the closest titles are kept, newest first on ties
    jobs = [{"Job Profile": t, "Date Posted": _days_ago(d), "Job Link": f"https://x/{i}"} for i, (t, d) in enumerate([
        ("Senior Product Manager, Payments", 2), ("Product Manager", 1), ("Product Manager - Lending", 20),
        ("Senior Product Manager", 5), ("Product Manager", 9), ("Product Operations Manager", 1)])]
    best = pick_best_roles(jobs, ["product manager"], 3)
    check(len(best) == 3, f"roles per company: {len(best)}, want 3")
    check([j["Job Profile"] for j in best[:2]] == ["Product Manager", "Product Manager"] and best[0]["Date Posted"] == _days_ago(1),
          f"role order: {[(j['Job Profile'], j['Date Posted']) for j in best]}")

    # 6. Freshness badges
    check(posting_freshness(_days_ago(0)).startswith("🔥"), "today should be 🔥")
    check(posting_freshness(_days_ago(5)).startswith("🆕"), "5 days should be 🆕")
    check(posting_freshness(_days_ago(20)) == "" and posting_freshness("Not listed") == "", "old/undated should have no badge")

    # 7. Two similar roles at ONE company share boilerplate JD text but are two openings (not merged).
    #    The same job on two different boards is still merged (existing cross-site rule).
    boiler = "We are a fast growing fintech building payments for millions of small businesses across India. " * 6
    jd_a = boiler + "You will own the merchant onboarding roadmap and partner with risk."
    jd_b = boiler + "You will own the lending product roadmap and partner with credit."
    a = {"Job Profile": "Senior Product Manager, Onboarding", "Target Company": "Acme", "Job Link": "https://boards.greenhouse.io/acme/jobs/111"}
    b = {"Job Profile": "Senior Product Manager, Lending", "Target Company": "Acme", "Job Link": "https://boards.greenhouse.io/acme/jobs/222"}
    c = {"Job Profile": "Senior Product Manager, Onboarding", "Target Company": "Jobgether", "Job Link": "https://jobs.lever.co/jobgether/abc-123"}
    _POSTING_INFO.clear(); _JD_FP.clear()
    _POSTING_INFO.update({a["Job Link"]: {"jd": jd_a}, b["Job Link"]: {"jd": jd_b}, c["Job Link"]: {"jd": jd_a}})
    kept, _ = dedupe_across_sites([dict(a), dict(b)])
    check(len(kept) == 2, f"two roles at one company merged into {len(kept)}")
    kept, _ = dedupe_across_sites([dict(a), dict(b), dict(c)])
    check(len(kept) == 2 and any("Also Posted At" in k for k in kept), f"cross-site copy not merged: {len(kept)} kept")
    _POSTING_INFO.clear(); _JD_FP.clear()

    # 8. Agent switches
    for mode, want in (("both", (1, 1, 1)), ("linkedin", (1, 0, 0)), ("career", (0, 1, 0)), ("discover", (0, 0, 1))):
        got = tuple(int(agent_on({"search_mode": mode}, a)) for a in ("linkedin", "web", "discover"))
        check(got == want, f"search_mode={mode}: agents {got}, want {want}")

    # 9. Pool building: Exa + LLM results merged by website; job boards / ATS hosts never become 'companies'
    llm_json = ('[{"name": "Razorpay", "website": "https://razorpay.com", "stage": "Growth (Series C+)"},'
                ' {"name": "Naukri", "website": "https://www.naukri.com", "stage": "Large / Public"},'
                ' {"name": "Tiny", "website": "https://tiny.in/about", "stage": "Seed / Pre-seed"}]')
    exa_hits = [_R("Razorpay - Payments", "https://www.razorpay.com/"), _R("Lever", "https://jobs.lever.co/foo"),
                _R("Paytm | Payments", "https://paytm.com")]
    with _tf.TemporaryDirectory() as tmp, \
            _mock.patch.dict(globals(), {"COMPANY_POOL_FILE": os.path.join(tmp, "pool.json"), "GEMINI_API_KEY": "x",
                                         "gemini_text": lambda p: llm_json}), \
            _mock.patch.object(exa, "search", create=True, side_effect=lambda **k: _mock.Mock(results=exa_hits)):
        data = load_company_pool()
        cos = ensure_company_pool(data, "Fintech", "Delhi", 10)
        keys = sorted(c["key"] for c in cos)
        check(keys == ["paytm.com", "razorpay.com", "tiny.in"], f"pool keys {keys}")
        check(next(c for c in cos if c["key"] == "razorpay.com")["stage"] == "Growth (Series C+)", "LLM stage lost on merge")

        # 10. Full mocked run: 3 branches in parallel -> merge. Discovery adds badges, stage, max 3 roles/company.
        today = _dt.now()
        def fake_ats(ats, slug, roles):
            n = 5 if slug[0] == "razorpay" else 1
            return [{"title": f"Product Manager {i}", "location": "Delhi, India", "url": f"https://boards.greenhouse.io/{slug[0]}/jobs/{i}",
                     "posted": today - _td(days=i)} for i in range(n)]
        patches = {"derive_domain_from_resume": lambda *a: "Fintech", "expand_role_variants_cached": lambda r: ["product manager"],
                   "detect_career_source": lambda site: ("Greenhouse", [_site_key(site).split(".")[0]], site),
                   "fetch_ats_jobs": fake_ats, "fetch_live_linkedin_jobs": lambda *a, **k: [
                       {"Job Profile": "Product Manager 0", "Target Company": "Razorpay", "Location": "Delhi",
                        "Job Link": "https://boards.greenhouse.io/razorpay/jobs/0", "Platform Name": "LinkedIn", "Date Posted": "Past 7 days"}],
                   "search_job_boards_node": lambda s: {"board_jobs": []}, "shortlist_companies_node": lambda s: {},
                   "search_career_pages_node": lambda s: {"career_jobs": []}}
        with _mock.patch.dict(globals(), patches):
            g = _SG(PipelineState)
            for name, fn in [("start_search", start_search_node), ("search_jobs", search_jobs_node),
                             ("shortlist_companies", globals()["shortlist_companies_node"]),
                             ("search_career_pages", globals()["search_career_pages_node"]),
                             ("search_job_boards", globals()["search_job_boards_node"]),
                             ("discover_companies", discover_companies_node),
                             ("search_discovered_careers", search_discovered_careers_node), ("merge_results", merge_search_results_node)]:
                g.add_node(name, fn)
            g.add_edge(_START, "start_search")
            for a_, b_ in [("start_search", "search_jobs"), ("start_search", "shortlist_companies"), ("start_search", "discover_companies"),
                           ("shortlist_companies", "search_career_pages"), ("search_career_pages", "search_job_boards"),
                           ("discover_companies", "search_discovered_careers")]:
                g.add_edge(a_, b_)
            g.add_edge(["search_jobs", "search_job_boards", "search_discovered_careers"], "merge_results")
            g.add_edge("merge_results", _END)
            out = g.compile().invoke({"roles": ["Product Manager"], "target_geos": ["Delhi"], "job_duration_days": 7,
                                      "search_mode": "both", "target_domain": "", "discovery_company_count": 10,
                                      "candidate_name": "t", "resume_summary": "", "search_iteration": 1})
        jobs = out["raw_jobs"]
        rz = [j for j in jobs if j["Target Company"] == "Razorpay"]
        check(len(rz) == 3, f"Razorpay roles kept: {len(rz)}, want 3 (5 open, max 3)")
        check(any(j.get("Freshness", "").startswith("🔥") for j in rz), "no 🔥 badge on a 0-day-old posting")
        li = next((j for j in jobs if j["Job Link"].endswith("/razorpay/jobs/0")), {})
        check(li.get("Platform Name") == "LinkedIn" and li.get("Company Stage") == "Growth (Series C+)",
              f"same job from LinkedIn + discovery should keep LinkedIn and gain the stage: {li.get('Platform Name')}, {li.get('Company Stage')}")
        saved = load_company_pool()["pools"][pool_key("Fintech", "Delhi")]["companies"]["razorpay.com"]
        check(saved.get("ats") == "Greenhouse" and saved.get("matches") == 3 and saved.get("recent_posts") == 5,
              f"pool not updated after the scan: {saved}")
        hist = load_company_pool()["history"][pool_key("Fintech", "Delhi")]
        check(len(hist["last_batch"]) == 3, f"last batch saved: {hist['last_batch']}")

    # 11. Hacker News 'Who is hiring': parsing real-looking posts
    now_i = int(_dt.now().timestamp())
    hn_items = [
        {"id": 901, "author": "a", "created_at_i": now_i - 2 * 86400,
         "text": 'Acme Pay (YC W21) | Senior Product Manager | Bangalore, India (Hybrid) | <a href="https:&#x2F;&#x2F;jobs.lever.co&#x2F;acmepay">https://jobs.lever.co/acmepay</a><p>We build payments for SMBs.'},
        {"id": 902, "author": "b", "created_at_i": now_i - 1 * 86400,
         "text": 'Ledgerly | Bengaluru | ONSITE<p>Roles:<p>- Backend Engineer<p>- Product Manager, Lending<p>Apply: <a href="https://ledgerly.io/jobs">https://ledgerly.io/jobs</a>'},
        {"id": 903, "author": "c", "created_at_i": now_i - 3 * 86400,
         "text": 'Faraway Inc | Product Manager | San Francisco, CA | ONSITE | <a href="https://faraway.com">faraway.com</a>'},
        {"id": 904, "author": "d", "created_at_i": now_i - 3 * 86400,
         "text": 'GPU Co | Senior Backend Engineer | Bangalore | <a href="https://gpu.co/careers">gpu.co</a>'},
        {"id": 905, "author": "e", "created_at_i": now_i - 20 * 86400,
         "text": 'OldCo | Product Manager | Bangalore | <a href="https://oldco.in/careers">oldco.in</a>'},
        {"id": 906, "author": None, "text": None},   # deleted comment
    ]
    p1 = parse_hn_post(hn_items[0])
    check(p1["company"] == "Acme Pay", f"HN company: '{p1['company']}' (want 'Acme Pay', YC tag removed)")
    check(p1["ats"] == "Lever" and p1["slug"][0] == "acmepay", f"HN ATS link: {p1['ats']} {p1['slug']}")
    check(hn_role_title(p1, ["product manager"]) == "Senior Product Manager", f"HN header role: {hn_role_title(p1, ['product manager'])}")
    p2 = parse_hn_post(hn_items[1])
    check(hn_role_title(p2, ["product manager"]) == "Product Manager, Lending", f"HN body role: '{hn_role_title(p2, ['product manager'])}'")
    check(p2["website"] == "https://ledgerly.io", f"HN website: {p2['website']}")
    check(not is_listing_url(p1["hn_link"]), "an HN post link is treated as a listing page")

    def hn_get(url, params=None, **kw):
        if "search_by_date" in url:
            return _mock.Mock(status_code=200, json=lambda: {"hits": [
                {"objectID": "900", "title": "Ask HN: Who is hiring? (November 2026)", "created_at_i": now_i - 3 * 86400},
                {"objectID": "800", "title": "Ask HN: Who wants to be hired? (November 2026)", "created_at_i": now_i}]})
        if url.endswith("/items/900"):
            return _mock.Mock(status_code=200, json=lambda: {"id": 900, "children": hn_items})
        raise _rq.exceptions.ConnectionError("no route in test")
    with _mock.patch.object(_rq, "get", side_effect=hn_get), _mock.patch.dict(globals(), {"_HN_CACHE": {}}):
        names = [p["company"] for p in hn_matching_posts(["product manager"], "Bangalore")]
        check(names == ["Ledgerly", "Acme Pay", "OldCo"], f"HN matches for Bangalore PM: {names} (SF / engineer-only / deleted out)")
        check(len(hn_hiring_threads()) == 1, "'Who wants to be hired' thread was read as a hiring thread")

        # 12. HN companies join the batch: careers board read first; HN post is the job only when the board has nothing
        with _tf.TemporaryDirectory() as tmp, _mock.patch.dict(globals(), {
                "COMPANY_POOL_FILE": os.path.join(tmp, "pool.json"), "GEMINI_API_KEY": "", "HN_ENABLED": True,
                "derive_domain_from_resume": lambda *a: "Fintech", "expand_role_variants_cached": lambda r: ["product manager"],
                "ensure_company_pool": lambda data, d, g, n: list(data["pools"].setdefault(pool_key(d, g), {"built": "", "companies": {}})["companies"].values()),
                "detect_career_source": lambda site: (None, None, None), "guess_ats_from_name": lambda n, w: (None, None),
                "fetch_ats_jobs": lambda ats, slug, roles: [{"title": "Senior Product Manager", "location": "Bangalore, India",
                                                            "url": "https://jobs.lever.co/acmepay/1111-aaaa", "posted": _dt.now() - _td(days=1)}]}):
            st = {"roles": ["Product Manager"], "target_geos": ["Bangalore"], "job_duration_days": 7, "search_mode": "discover",
                  "target_domain": "", "discovery_company_count": 10, "candidate_name": "t", "resume_summary": ""}
            b = discover_companies_node(st)["discovery_batch"]
            check(sorted(c["name"] for c in b) == ["Acme Pay", "Ledgerly", "OldCo"], f"HN batch: {[c['name'] for c in b]}")
            jobs = search_discovered_careers_node({**st, "discovery_batch": b})["discovery_jobs"]
            got = sorted((j["Target Company"], j["Platform Name"].split(" (")[0]) for j in jobs)
            check(got == [("Acme Pay", "Company Discovery"), ("Ledgerly", "Hacker News")],
                  f"HN jobs: {got} (Acme from its Lever board, Ledgerly from the HN post, OldCo too old for 7 days)")
            led = next((j for j in jobs if j["Target Company"] == "Ledgerly"), {})
            check(led.get("Job Link", "").startswith("https://news.ycombinator.com/item?id=902") and "ledgerly.io" in led.get("Also Posted At", ""),
                  f"HN job links: {led.get('Job Link')} / {led.get('Also Posted At')}")
            b2 = discover_companies_node(st)["discovery_batch"]
            check(sorted(c["name"] for c in b2) == ["Acme Pay", "Ledgerly"],   # hot ones are rechecked, OldCo (no fit) is not
                  f"next batch after HN: {[c['name'] for c in b2]} (want only the 2 hot companies)")

    print(f"\n🧪 Company discovery eval: {total - len(fails)}/{total} checks passed")
    for f in fails:
        print(f"   ✗ {f}")
    return not fails


run_company_discovery_eval()


### job application agent

agent 1- a RAG agent which has access to my profile pdf, is able to extract relevant information from it like job titles, geo, years of experience, technical capability
Role- Senior Knowdlegdge management Specialist  
Goal- Converting the knowledge into legible, readible and comprehendible structured bullet points without changing any meaning or coherency.  
Backstory- You are a meticulous researcher. You specialize in parsing local documentation
        to answer queries without introducing outside hallucinations.

Task- to process the GIVEN PDF docs without editing anything and storing in memory

(I want to use multiple documents - here resumes)




agent 2 - a job researcher which is able to search for relevant jobs for {candidate_name} specific geo/profile/experience 
domain/industry etc.
Role- A specialist career job finder
Goal- to look for relevant jobs for the profiles provided  as per the user query: {user_query}
Backstory- You have successfully managed to search jobs from all known platforms which may have a job posting  (like LinkedIn/IIm jobs/wellfound, other social media platform posts, public forum posts by individuals and all local job posting sites for each geography provided)

Task- based on user query, find jobs posted online across different job listing platforms across organisations of all sizes posted at any duration
Output- A list of 25 jobs with their job description links 


agent3- a verifier which provides a similarity score b/w my profile doc and job description and background 
and verifies if the score is above a certain threshold
Role- Job Profile Evaluator in a Talent Acquisition Team
Goal- To search for relevancy of profile match 
Backstory- You have a background of matching candidate profiles for organisations ranging from new age startups to big tech and fortune 500 companies across globe. You also check if the job posting is updated, posted by a legitimate company/org

Task- Using the the links given, evaluate which job description title matches the agent 1 document. Cite a brief reason for match in less than 50 words
Output- a score out of Poor/Average/Good/Perfect match for profiles matching my resume. Cite a reason for match in less than 50 words 



agent 4- creates a list of jobs posted in the previous x duration 
Format should be excel with -
Role- Executive Assitant & Data Entry specialist
Goal- You enter data fetched to a legible/readable and structured format like excel for humans to process 
Backstory- You have history of reading and scraping useful information into a human readable structure & format. You are an expert at Excel 

Task- Convert the job matches with Average or higher score in an excel table to be accessed from system
Output-An excel table with the following columns:
1.JOb Profile- scrape the Title of the job
2.Job Link- the web link to directly access the job description
3. Platform Name- fetch/scrape the name of the platform where it is posted
4. Date posted- fetch the date of job post
Append all matched jobs to the above mentioned columns. 

Tools- excel


agent 5 - takes the input as csv and starts finding people on linkedIn who are senior to me or might be hiring in that org, anyone related to tech or management roles near to my role
Role- Networking Specialist
Goal- Headhunting best profiles via using web search
Backstory- You have a track record of hunting down profiles from all social media platforms who have or are likely to hire for relevant roles

Task- Based on the best profile matches, find 3 relevant profiles who are in either management roles or are senior in roles {candidate_name} is looking for
Output- A plan for cold reachout for the mentioned profiles



agent 6 -returns the best profile linkedin URLs + retrieves their email id using appolo as a tool, adds it in another column of the same excel
Role- data entry Specialist  
Goal- You enter data fetched to a legible/readable and structured format like excel for humans to process 
 
Backstory- You have a background in sales navigation and finding ways via official emails/ linkedIn profile links to reach out to prospect employers

Task-To retrieve emails and LinkedIn profiles for people selected by the agent 5
Output- add a column in the given excel with name "Prospect_Employer_Profile_mail" & "Prospect_Employer_Profile_LI" 
A list of 3 LinkedIn profile URLs & their official emails- each in a single cell 
Tools- LinkedIn_Search or google search or apollo.io search (whichever is accessible)




in London, Spain, Netherlands, Singapore, India or any Remote Roles
 